# Archived analysis workflow

This notebook records the author workflow used during revision. It requires the corresponding external model-ready inputs, saved predictions and/or checkpoints. Its `/content/drive/MyDrive` paths describe the author environment and must be reviewed before execution. Do not run all cells merely to inspect the repository. See [analysis/README.md](../README.md) and the main repository README for the supported benchmark entry point.


# SpaGAT：R2 实际资源补测

这是实际测量 notebook，**不再收集日志**。默认 SEA-AD、五模型、seed=123，使用 GPU runtime（建议与原记录一致的 Tesla T4）。

已有两套数据、三 seeds 的训练时间已经核对，不重新训练那 30 次实验。这里补测：

- 四个神经模型：加载已保存 checkpoint，完整 test 推理；另外在独立进程里执行 10 个预热批次和 100 个测量批次，batch size=32，记录训练资源。
- LightGBM：完整 test 推理；另对完整 140 基因按原设置拟合 100 棵树并预测 validation，用于 CPU 内存测量。新拟合模型不保存、不替换旧模型。这一步通常最耗时，原 SEA-AD 日志中的基因拟合/验证循环约 44 分钟，实际依 CPU 而变。
- 共享空间邻域：按提供的距离/排序规则重建 50 个邻居索引并核对现有索引，单独计时一次。它是邻域重建时间，不冒充当时未记录的原始全流程预处理时间。

**原训练目录、输入、checkpoint、预测不覆盖。** 所有输出和缓存写入新的 `/content/R2_resource_profile_...` 目录；下载包不包含大矩阵或权重。

建议为默认配置预留约 1–2 小时；这只是安排时间的估计，构图、Drive 复制和当前 CPU 会影响实际耗时。神经模型的峰值仅对应本次固定批次测量，不能写成整个 10-epoch 训练的峰值。

代码已做语法、路径及纯函数测试；本地没有 CUDA，尚未在真实 GPU/数据上执行。各模型完整推理会与原预测逐项核对，差异会明确记录，不修改论文准确率结果。


## 1. 环境与 Drive

In [ ]:
import importlib.util, subprocess, sys
# 优先使用原训练环境；只补装确实缺失的轻量依赖，不重装 PyTorch。
for module, package in [('psutil', 'psutil'), ('lightgbm', 'lightgbm==4.6.0'), ('torch_geometric', 'torch-geometric')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
import torch
if not torch.cuda.is_available():
    raise RuntimeError('请将 Colab 的硬件加速器切换为 GPU，再运行。')
print('本次实际 GPU：', torch.cuda.get_device_name())
print('若不是 Tesla T4，结果将按当前硬件报告，不能当作原 T4 的计时。')
from google.colab import drive
drive.mount('/content/drive')


## 2. 测量范围

In [ ]:
from pathlib import Path
# 默认只对规模较大的 SEA-AD 做一套完整资源测量。
# 如需 Mouse，改为 DATASETS = ['SEA_AD', 'Mouse']；耗时会增加。
DATASETS = ['SEA_AD']
ROOT = Path('/content/drive/MyDrive/SpaGAT_revision/inductive_training/20260922_104353_552891_UTC_f291de00')
MODELS = ['SpaGAT', 'GAT', 'GITIII_official', 'SPICE_adapted', 'LightGBM']


## 3. 写入本次测量程序并检查输入

以下程序使用你上传的冻结模型代码，按哈希验证，并在每次测量中检查参数量、数据划分和样本顺序。它不会执行原来的完整训练入口。

In [ ]:
SOURCES = {'spagat/__init__.py': '"""\r\nSpaGAT package.\r\n"""\r\n\r\nfrom .gene_program_model import SpaGP, SpaGP_Loss\r\nfrom .dataloader import SPAGAT_dataset\r\n\r\n__all__ = [\r\n    "SpaGP",\r\n    "SpaGP_Loss",\r\n    "SPAGAT_dataset",\r\n]', 'spagat/attention.py': 'import torch\r\nimport torch.nn as nn\r\nimport torch.nn.functional as F\r\nfrom .embedding import FFN\r\n\r\ndef rho(x):\r\n    return torch.sqrt(F.relu(x)) - torch.sqrt(F.relu(-x))\r\n\r\ndef get_diagonal(x):\n    index = torch.arange(0, x.shape[1], 1)\n    return x[:, index, index, :]\n\n\ndef _normalize_pairwise_tensors(node, edge, embedding, edge_dim=None):\n    """Normalize Embedding\'s initial singleton receiver axis.\n\n    ``Embedding.forward`` supplies pairwise tensors as ``(B, 1, N, E)``: one\n    sender-wise edge representation shared by every receiver.  GRIT attention\n    operates on an explicit receiver/sender representation ``(B, N, N, E)``.\n    Expanding the singleton receiver axis before Q/K--edge arithmetic prevents\n    PyTorch from broadcasting the batch dimension into an unintended\n    ``(B, B, N, N, E)`` tensor.  Later GRIT layers already use the expanded\n    four-dimensional convention.\n    """\n    if node.ndim != 3:\n        raise ValueError(f"node must have shape (B, N, node_dim); got {tuple(node.shape)}.")\n    B, N, _ = node.shape\n\n    def normalize(name, tensor):\n        if tensor.ndim != 4:\n            raise ValueError(\n                f"{name} must have shape (B, 1, N, edge_dim) or (B, N, N, edge_dim); "\n                f"got {tuple(tensor.shape)}."\n            )\n        if tensor.shape[0] != B or tensor.shape[2] != N:\n            raise ValueError(\n                f"{name} batch and sender dimensions must be B={B} and N={N}; "\n                f"got {tuple(tensor.shape)}."\n            )\n        if tensor.shape[1] == 1:\n            tensor = tensor.expand(B, N, N, tensor.shape[-1])\n        elif tensor.shape[1] != N:\n            raise ValueError(\n                f"{name} receiver dimension must be 1 or N={N}; got {tuple(tensor.shape)}."\n            )\n        return tensor\n\n    edge = normalize("edge", edge)\n    embedding = normalize("embedding", embedding)\n    if edge.shape[-1] != embedding.shape[-1]:\n        raise ValueError(\n            "edge and embedding must have the same feature dimension; got "\n            f"{edge.shape[-1]} and {embedding.shape[-1]}."\n        )\n    if edge_dim is not None and edge.shape[-1] != edge_dim:\n        raise ValueError(\n            f"edge and embedding feature dimension must be edge_dim={edge_dim}; "\n            f"got {edge.shape[-1]}."\n        )\n    return edge, embedding\n\r\nclass GRIT_attention(nn.Module):\n    def __init__(self, node_dim, edge_dim, att_dim=8, program_aware=False, routing_mode="program"):\n        super().__init__()\n        self.program_aware = program_aware\n        if routing_mode not in {"grit", "program"}:\n            raise ValueError("routing_mode must be \'grit\' or \'program\'.")\n        self.routing_mode = routing_mode\n        self.node_dim = node_dim\n        self.base_edge_dim = edge_dim\n        self.edge_dim = edge_dim * 2\n        self.W_Q = nn.Linear(node_dim, self.edge_dim * att_dim)\r\n        self.W_K = nn.Linear(node_dim, self.edge_dim * att_dim)\r\n        self.W_V = nn.Linear(node_dim, node_dim)\r\n        self.W_Ew = nn.Linear(edge_dim, edge_dim, bias=False)\r\n        self.W_Eb = nn.Linear(edge_dim, edge_dim, bias=False)\r\n        self.W_En = nn.Linear(edge_dim, node_dim)\r\n        self.W_A = nn.Linear(edge_dim, 1, bias=False)\r\n        self.W_No = nn.Linear(node_dim, node_dim)\n        self.W_Eo = nn.Linear(edge_dim, edge_dim, bias=False)\n\n        # Program tokens live in the common edge-feature space (edge_dim).  They\n        # are supplied by a parent model so that token k has the same biological\n        # identity in every graph layer; these projections remain layer-specific.\n        if self.program_aware:\n            self.W_program_edge = nn.Linear(edge_dim, edge_dim, bias=False)\n            self.W_program_receiver = nn.Linear(node_dim, edge_dim, bias=False)\n            self.W_program_gate = nn.Linear(node_dim, edge_dim, bias=False)\n\n    def forward(self, x, program_tokens=None, program_edge_mask=None):\n        """Run edge-aware attention, optionally routed through communication programs.\n\n        Args:\n            x: ``[node, edge, embedding]`` with node ``(B, N, node_dim)`` and\n                broadcastable edge/embedding tensors ``(B, 1 or N, N, edge_dim)``.\n            program_tokens: Optional shared token bank ``(K, edge_dim)``.  A\n                token is a globally named latent communication program; the\n                same bank should be passed to every program-aware GRIT layer.\n            program_edge_mask: Optional inference-only ``(B, N, N, K)`` mask.\n                Zero entries remove sender edges from a program\'s softmax.\n\n        Returns:\n            ``[node_updated, edge_updated, program_state]``.  ``program_state``\n            is ``None`` in baseline mode, preserving the original computation.\n        """\n        node, edge, embedding = x\n        B, N, C = node.shape\n\n        # Always normalize the initial Embedding output, including the legacy\n        # path.  The routing mode must not determine whether Q/K arithmetic is\n        # performed on a correctly shaped pairwise edge tensor.\n        edge, embedding = _normalize_pairwise_tensors(\n            node, edge, embedding, edge_dim=self.base_edge_dim\n        )\n\n        if program_tokens is not None and not self.program_aware:\n            raise ValueError(\n                "program_tokens were provided, but this GRIT_attention was "\n                "constructed with program_aware=False."\n            )\n        if program_tokens is not None and program_tokens.ndim != 2:\n            raise ValueError(\n                "program_tokens must have shape (n_programs, edge_dim); got "\n                f"{tuple(program_tokens.shape)}."\n            )\n        if program_tokens is not None and program_tokens.shape[1] != self.base_edge_dim:\n            raise ValueError(\n                "program_tokens must have shape (n_programs, edge_dim), where "\n                f"edge_dim={self.base_edge_dim}; got {tuple(program_tokens.shape)}."\n            )\n        if program_edge_mask is not None and program_tokens is None:\n            raise ValueError("program_edge_mask requires program-aware routing tokens.")\n\n        Q = self.W_Q(node).reshape(B, N, -1, self.edge_dim).permute(0, 3, 1, 2)\n        K = self.W_K(node).reshape(B, N, -1, self.edge_dim).permute(0, 3, 2, 1)\r\n        QK = (Q @ K).permute(0, 2, 3, 1)\r\n        edge = F.gelu(rho((QK[:, :, :, :self.edge_dim // 2]) * self.W_Ew(edge)) + self.W_Eb(edge) + QK[:, :, :, self.edge_dim // 2:] + embedding)\r\n        V = self.W_V(node)\n\n        if program_tokens is None:\n            # This is the original GRIT message-passing path, kept numerically\n            # unchanged when no shared program bank is supplied.\n            alpha = self.W_A(edge).squeeze(dim=-1)\n            alpha = F.softmax(alpha, dim=-1)\n            node = alpha @ V + self.W_En(torch.sum(edge * alpha.unsqueeze(dim=-1), dim=-2))\n            node = self.W_No(node)\n            edge = self.W_Eo(edge)\n            return [node, edge, None]\n\n        if self.routing_mode == "grit":\n            if program_tokens.shape[0] != 1:\n                raise ValueError("routing_mode=\'grit\' is defined for the controlled K=1 ablation only.")\n            # Controlled routing ablation: retain GRIT\'s original scalar edge\n            # routing and expose it as the K=1 program-state interface.  The\n            # downstream program gate/messages/free decoder are unchanged.\n            program_attention = F.softmax(self.W_A(edge).squeeze(dim=-1), dim=-1).unsqueeze(dim=-1)\n        else:\n            # Program routing: the edge, receiver, and shared program token\n            # interact multiplicatively. This is the pre-existing SpaGP score.\n            edge_program = self.W_program_edge(edge)\n            receiver_program = self.W_program_receiver(node)\n            program_scores = torch.einsum(\n                "bije,bie,ke->bijk", edge_program, receiver_program, program_tokens\n            ) / (self.base_edge_dim ** 0.5)\n            program_attention = None\n        if program_edge_mask is not None:\n            expected_shape = (B, N, N, program_tokens.shape[0])\n            if tuple(program_edge_mask.shape) != expected_shape:\n                raise ValueError(\n                    f"program_edge_mask must have shape {expected_shape}; got "\n                    f"{tuple(program_edge_mask.shape)}."\n                )\n            allowed = program_edge_mask.to(device=program_scores.device, dtype=torch.bool)\n            if not torch.all(allowed.any(dim=2)):\n                raise ValueError("program_edge_mask removes every sender for at least one receiver/program.")\n            # This is inference-time graph intervention: masked edges receive\n            # exactly zero attention and remaining edges are renormalized by\n            # the existing neighbor softmax.\n            if self.routing_mode == "grit":\n                grit_scores = self.W_A(edge).squeeze(dim=-1).masked_fill(~allowed[..., 0], float("-inf"))\n                program_attention = F.softmax(grit_scores, dim=-1).unsqueeze(dim=-1)\n            else:\n                program_scores = program_scores.masked_fill(~allowed, float("-inf"))\n                program_attention = F.softmax(program_scores, dim=2)\n        elif program_attention is None:\n            program_attention = F.softmax(program_scores, dim=2)\n\n        # Aggregate node and edge evidence independently for each program:\n        # m[i, k] = sum_j alpha[i, j, k] * message[i, j].\n        node_messages = torch.einsum("bijk,bjd->bikd", program_attention, V)\n        edge_messages = torch.einsum("bijk,bije->bike", program_attention, edge)\n        program_messages = node_messages + self.W_En(edge_messages)\n\n        # Receiver-specific, non-negative gates fuse the K program messages back\n        # into the original node_dim, keeping the downstream tensor shape intact.\n        gate_scores = torch.einsum(\n            "bin,kn->bik", self.W_program_gate(node), program_tokens\n        ) / (self.base_edge_dim ** 0.5)\n        program_activations = F.softplus(gate_scores)\n        node = torch.sum(program_messages * program_activations.unsqueeze(dim=-1), dim=2)\n        node = self.W_No(node)\n        edge = self.W_Eo(edge)\n\n        program_state = {\n            "attention": program_attention,\n            "messages": program_messages,\n            "activations": program_activations,\n        }\n        return [node, edge, program_state]\n\r\nclass DegScaler(nn.Module):\r\n    def __init__(self, node_dim):\r\n        super().__init__()\r\n        scaler = (2 / node_dim) ** 0.5\r\n        self.theta1 = nn.Parameter(torch.randn(node_dim) * scaler)\r\n        self.theta2 = nn.Parameter(torch.randn(node_dim) * scaler)\r\n\r\n    def forward(self, node, degree):\r\n        node = self.theta1 * node + torch.log(degree + 1).unsqueeze(dim=-1) * node * self.theta2\r\n        return node\r\n\r\nclass Multi_Head_Attention(nn.Module):\r\n    def __init__(self, node_dim, edge_dim, num_heads, att_dim=8):\r\n        super().__init__()\r\n        self.attentions = nn.ModuleList([\r\n            GRIT_attention(node_dim, edge_dim, att_dim) for _ in range(num_heads)\r\n        ])\r\n        self.W_hn = nn.Parameter(torch.ones(num_heads, 1) / num_heads)\r\n        self.W_he = nn.Parameter(torch.ones(num_heads, 1) / num_heads)\r\n\r\n    def forward(self, x):\r\n        results = [attentioni(x) for attentioni in self.attentions]\r\n        node = (torch.stack([tmp[0] for tmp in results], dim=-1) @ self.W_hn).squeeze(dim=-1)\r\n        edge = torch.stack([tmp[1] for tmp in results], dim=-1) @ self.W_he\r\n        edge = edge.squeeze(dim=-1)\r\n        return [node, edge, x[2]]\r\n\r\nclass GRIT_encoder(nn.Module):\n    def __init__(self, node_dim, edge_dim, num_heads, att_dim=8, program_aware=False, routing_mode="program"):\n        super().__init__()\n        self.program_aware = program_aware\n        self.edge_dim = edge_dim\n        self.routing_mode = routing_mode\n        self.attentions = Multi_Head_Attention(node_dim, edge_dim, num_heads, att_dim)\n        if program_aware:\n            # Keep Multi_Head_Attention unchanged for baseline callers, but replace\n            # its heads with token-aware GRIT attention modules for this encoder.\n            # The shared tokens themselves are supplied to forward(), not owned by\n            # a layer, so program index k remains consistent across GNN layers.\n            self.attentions.attentions = nn.ModuleList([\n                GRIT_attention(node_dim, edge_dim, att_dim, program_aware=True, routing_mode=routing_mode)\n                for _ in range(num_heads)\n            ])\n        self.FFN = FFN(node_dim)\n        self.ln1 = nn.LayerNorm(node_dim)\n        self.ln2 = nn.LayerNorm(node_dim)\n        self.ln_edge = nn.LayerNorm(edge_dim)\n\n    def forward(self, x, program_tokens=None, program_edge_mask=None):\n        """Encode one graph layer, optionally with shared program-aware routing.\n\n        Without ``program_tokens``, this follows the original GRIT encoder path\n        and returns ``[node, edge, distance_embedding]``.  With a shared token\n        bank, each head returns program-specific routing state; the encoder\n        combines it across heads and returns it as a fourth element.\n        """\n        node, edge, distance = x\n        # Keep the residual edge tensor in the same four-dimensional convention\n        # as GRIT_attention.  This is essential for the baseline residual add.\n        edge, distance = _normalize_pairwise_tensors(\n            node, edge, distance, edge_dim=self.edge_dim\n        )\n        x = [node, edge, distance]\n        if program_tokens is None:\n            # Preserve the existing multi-head aggregation and output contract.\n            x = self.attentions(x)\n            edge = self.ln_edge(edge + x[1])\n            node = self.ln1(node + x[0])\n            node = self.ln2(node + self.FFN(node))\n            return [node, edge, distance]\n\n        if not self.program_aware:\n            raise ValueError(\n                "program_tokens were provided, but this GRIT_encoder was "\n                "constructed with program_aware=False."\n            )\n\n        # Multi_Head_Attention\'s baseline forward intentionally remains unchanged.\n        # Here, route the same shared program tokens through each token-aware head\n        # and use its existing learned head weights for node and edge aggregation.\n        results = [\n            attentioni(x, program_tokens, program_edge_mask=program_edge_mask)\n            for attentioni in self.attentions.attentions\n        ]\n        node_attention = (\n            torch.stack([result[0] for result in results], dim=-1) @ self.attentions.W_hn\n        ).squeeze(dim=-1)\n        edge_attention = torch.stack([result[1] for result in results], dim=-1) @ self.attentions.W_he\n        edge_attention = edge_attention.squeeze(dim=-1)\n\n        # Combine per-head routing metadata with normalized head weights.  This\n        # yields one program axis K shared by all heads and preserves alpha\'s\n        # interpretation as a neighbor distribution for each receiver/program.\n        head_weights = F.softmax(self.attentions.W_hn.squeeze(dim=-1), dim=0)\n        program_state = {\n            key: torch.sum(\n                torch.stack([result[2][key] for result in results], dim=-1)\n                * head_weights,\n                dim=-1,\n            )\n            for key in ("attention", "messages", "activations")\n        }\n\n        # Pairwise tensors were normalized at entry, so the residual update is\n        # always performed in the explicit ``(B, N, N, E)`` convention.\n        edge = self.ln_edge(edge + edge_attention)\n        node = self.ln1(node + node_attention)\n        node = self.ln2(node + self.FFN(node))\n        return [node, edge, distance, program_state]\n\r\nclass GRIT_encoder_last_layer(nn.Module):\r\n    def __init__(self, node_dim, in_node, edge_dim, node_dim_small=16, att_dim=8):\r\n        super().__init__()\r\n        self.node_dim_small = node_dim_small\r\n        self.in_node = in_node\r\n        self.edge_dim = edge_dim * 2\r\n        self.att_dim = att_dim\r\n        self.W_Q = nn.Linear(node_dim, self.edge_dim * att_dim)\r\n        self.W_K = nn.Linear(node_dim, self.edge_dim * att_dim)\r\n        self.W_Ew = nn.Linear(edge_dim, edge_dim, bias=False)\r\n        self.W_Eb = nn.Linear(edge_dim, edge_dim, bias=False)\r\n        self.W_En = nn.Linear(edge_dim, node_dim)\r\n        self.W_A = nn.Linear(edge_dim, in_node, bias=False)\r\n        self.node_transform = nn.Linear(node_dim, in_node)\r\n        self.edge_transform = nn.Linear(edge_dim, in_node)\r\n        self.head = FFN(in_node, in_dim=in_node * 2, out_dim=in_node)\r\n        self.head2 = nn.Sequential(nn.LayerNorm(in_node), FFN(in_node))\r\n        self.scaler = 2 / (node_dim ** 0.5)\r\n\r\n    def forward(self, x):\r\n        node, edge, embedding = x\r\n        B, N, C = node.shape\r\n        Q = self.W_Q(node[:, 0:1, :]).reshape(B, 1, self.att_dim, self.edge_dim).permute(0, 3, 1, 2)\r\n        K = self.W_K(node).reshape(B, N, self.att_dim, self.edge_dim).permute(0, 3, 2, 1)\r\n        QK = (Q @ K).permute(0, 2, 3, 1)\r\n        edge = edge[:, 0:1, :, :]\r\n        edge = F.gelu(rho((QK[:, :, :, :self.edge_dim // 2]) * self.W_Ew(edge)) + self.W_Eb(edge) + QK[:, :, :, self.edge_dim // 2:] + embedding[:, 0:1, :, :])\r\n        edge = edge[:, 0, 1:, :]\r\n        alphas = F.softmax(self.W_A(edge).permute(0, 2, 1), dim=-1).permute(0, 2, 1)\r\n        edge = self.edge_transform(edge) * alphas\r\n        node = self.node_transform(node[:, 1:, :]) * alphas\r\n        tmp = self.head(torch.concat([edge, node], dim=-1))\r\n        node = torch.sum(tmp, dim=-2) * self.scaler\r\n        return [node, [tmp.permute(0, 2, 1).unsqueeze(dim=-2), edge.permute(0, 2, 1).unsqueeze(dim=-2)]]\r\n\r\nclass ProgramLastLayer(nn.Module):\r\n    def __init__(self, node_dim, in_node, edge_dim, n_programs=8, node_dim_small=16, att_dim=8):\r\n        super().__init__()\r\n        self.node_dim_small = node_dim_small\r\n        self.in_node = in_node\r\n        self.n_programs = n_programs\r\n        self.edge_dim = edge_dim * 2\r\n        self.att_dim = att_dim\r\n        self.W_Q = nn.Linear(node_dim, self.edge_dim * att_dim)\r\n        self.W_K = nn.Linear(node_dim, self.edge_dim * att_dim)\r\n        self.W_Ew = nn.Linear(edge_dim, edge_dim, bias=False)\r\n        self.W_Eb = nn.Linear(edge_dim, edge_dim, bias=False)\r\n        self.W_A = nn.Linear(edge_dim, in_node * n_programs, bias=False)\r\n        self.node_transform = nn.Linear(node_dim, in_node)\r\n        self.edge_transform = nn.Linear(edge_dim, in_node)\r\n        self.head = FFN(in_node, in_dim=in_node * 2, out_dim=in_node)\r\n        self.scaler = 2 / (node_dim ** 0.5)\r\n\r\n    def forward(self, x):\r\n        node, edge, embedding = x\r\n        B, N, C = node.shape\r\n        Q = self.W_Q(node[:, 0:1, :]).reshape(B, 1, self.att_dim, self.edge_dim).permute(0, 3, 1, 2)\r\n        K_mat = self.W_K(node).reshape(B, N, self.att_dim, self.edge_dim).permute(0, 3, 2, 1)\r\n        QK = (Q @ K_mat).permute(0, 2, 3, 1)\r\n        edge = edge[:, 0:1, :, :]\r\n        edge = F.gelu(\r\n            rho((QK[:, :, :, :self.edge_dim // 2]) * self.W_Ew(edge))\r\n            + self.W_Eb(edge)\r\n            + QK[:, :, :, self.edge_dim // 2:]\r\n            + embedding[:, 0:1, :, :]\r\n        )\r\n        edge = edge[:, 0, 1:, :]\r\n        alphas_all = self.W_A(edge)\r\n        alphas_all = alphas_all.reshape(B, N - 1, self.in_node, self.n_programs)\r\n        alphas_all = alphas_all.permute(0, 3, 2, 1)\r\n        alphas_all = F.softmax(alphas_all, dim=-1)\r\n        alphas_all = alphas_all.permute(0, 3, 1, 2)\r\n        edge_feat = self.edge_transform(edge)\r\n        node_feat = self.node_transform(node[:, 1:, :])\r\n        program_outputs = []\r\n        program_influences = []\r\n        for k in range(self.n_programs):\r\n            alpha_k = alphas_all[:, :, k, :]\r\n            edge_k = edge_feat * alpha_k\r\n            node_k = node_feat * alpha_k\r\n            tmp_k = self.head(torch.cat([edge_k, node_k], dim=-1))\r\n            output_k = torch.sum(tmp_k, dim=1) * self.scaler\r\n            program_outputs.append(output_k)\r\n            program_influences.append(tmp_k)\r\n        y_pred = torch.stack(program_outputs, dim=0).sum(dim=0)\r\n        total_influence = torch.stack(program_influences, dim=0).sum(dim=0)\r\n        influence_for_compat = total_influence.permute(0, 2, 1).unsqueeze(dim=-2)\r\n        program_info = {\r\n            \'per_program_influence\': torch.stack(program_influences, dim=0),\r\n            \'per_program_output\': torch.stack(program_outputs, dim=0),\r\n            \'per_program_attention\': alphas_all,\r\n        }\r\n        return [y_pred, [influence_for_compat, program_info]]\n', 'spagat/dataloader.py': 'import torch\nfrom torch.utils.data import Dataset\nimport os\nimport numpy as np\nimport pandas as pd\n\nclass SPAGAT_dataset(Dataset):\n    def __init__(self, processed_dir, num_neighbors=50, col_centerx="centerx",\n                 col_centery="centery", col_cell_type="subclass"):\n        if processed_dir[-1] != "/":\n            processed_dir = processed_dir + "/"\n\n        samples = []\n        for filei in os.listdir(processed_dir):\n            if filei.find("_TypeExp.npz") >= 0:\n                samples.append(filei.split("_TypeExp.npz")[0])\n        self.samples = list(sorted(list(set(samples))))\n        print("Have samples:", len(self.samples), self.samples)\n\n        self.index_index = ["index_" + str(i) for i in range(num_neighbors)]\n        self.interactions = torch.load("/".join(processed_dir.split("/")[:-2]) + "/ligands.pth", weights_only=False)\n        self.genes = torch.load("/".join(processed_dir.split("/")[:-2]) + "/genes.pth", weights_only=False)\n\n        # Collect all cell types from all data files\n        all_cell_types = set()\n        for samplei in self.samples:\n            try:\n                dfi = pd.read_csv(processed_dir + samplei + ".csv")\n                all_cell_types.update(dfi[col_cell_type].unique())\n            except Exception as e:\n                print(f"Warning: Cannot read sample {samplei}: {e}")\n                continue\n        # Build cell type dictionary, sorted for consistency\n        self.cell_types_dict = {ct: i for i, ct in enumerate(sorted(all_cell_types))}\n        print(f"Built cell type dictionary with {len(self.cell_types_dict)} cell types: {sorted(all_cell_types)}")\n\n        self.indexes = []\n        self.flags = []\n        self.centerx = []\n        self.centery = []\n        self.exps = []\n        self.cell_types = []\n        self.type_exp = []\n        for samplei in self.samples:\n            try:\n                dfi = pd.read_csv(processed_dir + samplei + ".csv")\n                # Assign cell type numbers for each cell in the sample\n                dfi["cell_type_number"] = np.array([self.cell_types_dict.get(cell_type, -1)\n                                                     for cell_type in dfi[col_cell_type]])\n                if "flag" not in dfi.columns.tolist():\n                    dfi["flag"] = [True for _ in range(dfi.shape[0])]\n\n                type_exp_dicti = np.load(processed_dir + samplei + "_TypeExp.npz", allow_pickle=True)\n                type_exp_dicti = {k: torch.Tensor(np.array(v).astype(np.float64)) for k, v in type_exp_dicti.items()}\n                # Only keep cell types present in both the dict and the global cell type dict\n                valid_cell_types = [ct for ct in dfi[col_cell_type].unique()\n                                   if ct in type_exp_dicti and ct in self.cell_types_dict]\n                type_exp = torch.stack([type_exp_dicti[cell_type] for cell_type in dfi[col_cell_type]\n                                       if cell_type in valid_cell_types], dim=0)\n                # Skip sample if no valid cell types\n                if len(valid_cell_types) == 0:\n                    print(f"Skip sample {samplei}: no valid cell types")\n                    continue\n                # Only keep rows with valid cell types\n                valid_mask = dfi[col_cell_type].isin(valid_cell_types)\n                dfi_valid = dfi[valid_mask].copy()\n                if len(dfi_valid) == 0:\n                    print(f"Skip sample {samplei}: no valid rows after filtering")\n                    continue\n                self.indexes.append(torch.LongTensor(dfi_valid.loc[:, self.index_index].values))\n                self.flags.append(dfi_valid.loc[:, "flag"].values)\n                self.centerx.append(torch.Tensor(dfi_valid.loc[:, col_centerx].values))\n                self.centery.append(torch.Tensor(dfi_valid.loc[:, col_centery].values))\n                self.exps.append(torch.Tensor(dfi_valid.loc[:, self.genes].values))\n                self.cell_types.append(torch.LongTensor(dfi_valid.loc[:, "cell_type_number"].values))\n                # Rebuild type_exp for valid cell types\n                type_exp_valid = torch.stack([type_exp_dicti[cell_type] for cell_type in dfi_valid[col_cell_type]], dim=0)\n                self.type_exp.append(type_exp_valid)\n            except Exception as e:\n                print(f"Error: Cannot process sample {samplei}: {e}")\n                import traceback\n                traceback.print_exc()\n                continue\n\n        self.meta_counts = []\n        self.arg_meta = []\n        for i in range(len(self.samples)):\n            if i < len(self.flags):\n                self.meta_counts.append(np.sum(self.flags[i]))\n                self.arg_meta.append(torch.LongTensor(np.where(self.flags[i] != 0)[0]))\n        self.meta_counts = np.array(self.meta_counts) if self.meta_counts else np.array([])\n        self.cumsum_meta = np.cumsum(self.meta_counts) if len(self.meta_counts) > 0 else np.array([])\n        print("There are totally", self.cumsum_meta[-1] if len(self.cumsum_meta) > 0 else 0, "cells in this dataset")\n\n    def __len__(self):\n        return self.cumsum_meta[-1] if len(self.cumsum_meta) > 0 else 0\n\n    def __getitem__(self, idx):\n        if len(self.cumsum_meta) == 0:\n            raise RuntimeError("No valid data in dataset")\n        # Find which sample this idx comes from\n        sample_id = np.searchsorted(self.cumsum_meta, idx, side=\'right\')\n        idx = idx - self.cumsum_meta[sample_id]\n        idx = self.arg_meta[sample_id][idx]\n        indices = self.indexes[sample_id][idx]\n        centerx = self.centerx[sample_id][indices]\n        centery = self.centery[sample_id][indices]\n        exp = self.exps[sample_id][indices]\n        type_exp = self.type_exp[sample_id][indices]\n        y = exp[0]\n        cell_types = torch.LongTensor(self.cell_types[sample_id][indices])\n        num_real_neighbors = len(indices)\n        neighbor_mask = torch.ones(num_real_neighbors)\n        return {\n            "x": exp,\n            "type_exp": type_exp,\n            "y": y,\n            "cell_types": cell_types,\n            "position_x": centerx,\n            "position_y": centery,\n            "neighbor_mask": neighbor_mask\n        }\n\nclass SPAGAT_evaluate_dataset(Dataset):\n    def __init__(self, processed_dir, sample, num_neighbors=50, col_centerx="centerx",\n                 col_centery="centery", col_cell_type="subclass"):\n        if processed_dir[-1] != "/":\n            processed_dir = processed_dir + "/"\n        self.samples = [sample]\n        print("Have samples:", self.samples)\n        self.index_index = ["index_" + str(i) for i in range(num_neighbors)]\n        self.interactions = torch.load("/".join(processed_dir.split("/")[:-2]) + "/ligands.pth", weights_only=False)\n        self.genes = torch.load("/".join(processed_dir.split("/")[:-2]) + "/genes.pth", weights_only=False)\n        # Collect all cell types from all data files\n        all_cell_types = set()\n        for samplei in self.samples:\n            try:\n                dfi = pd.read_csv(processed_dir + samplei + ".csv")\n                all_cell_types.update(dfi[col_cell_type].unique())\n            except Exception as e:\n                print(f"Warning: Cannot read sample {samplei}: {e}")\n                continue\n        # Build cell type dictionary, sorted for consistency\n        self.cell_types_dict = {ct: i for i, ct in enumerate(sorted(all_cell_types))}\n        print(f"Built cell type dictionary with {len(self.cell_types_dict)} cell types: {sorted(all_cell_types)}")\n        self.indexes = []\n        self.flags = []\n        self.centerx = []\n        self.centery = []\n        self.exps = []\n        self.cell_types = []\n        self.type_exp = []\n        for samplei in self.samples:\n            try:\n                dfi = pd.read_csv(processed_dir + samplei + ".csv")\n                dfi["cell_type_number"] = np.array([self.cell_types_dict.get(cell_type, -1)\n                                                     for cell_type in dfi[col_cell_type]])\n                if "flag" not in dfi.columns.tolist():\n                    dfi["flag"] = [True for _ in range(dfi.shape[0])]\n                type_exp_dicti = np.load(processed_dir + samplei + "_TypeExp.npz", allow_pickle=True)\n                type_exp_dicti = {k: torch.Tensor(np.array(v).astype(np.float64)) for k, v in type_exp_dicti.items()}\n                type_exp = torch.stack([type_exp_dicti[cell_type] for cell_type in dfi[col_cell_type]], dim=0)\n                self.type_exp.append(type_exp)\n                self.indexes.append(torch.LongTensor(dfi.loc[:, self.index_index].values))\n                self.flags.append(dfi.loc[:, "flag"].values)\n                self.centerx.append(torch.Tensor(dfi.loc[:, col_centerx].values))\n                self.centery.append(torch.Tensor(dfi.loc[:, col_centery].values))\n                self.exps.append(torch.Tensor(dfi.loc[:, self.genes].values))\n                self.cell_types.append(torch.LongTensor(dfi.loc[:, "cell_type_number"].values))\n            except Exception as e:\n                print(f"Error: Cannot process sample {samplei}: {e}")\n                import traceback\n                traceback.print_exc()\n                continue\n        self.meta_counts = []\n        self.arg_meta = []\n        for i in range(len(self.samples)):\n            if i < len(self.flags):\n                self.meta_counts.append(np.sum(self.flags[i]))\n                self.arg_meta.append(torch.LongTensor(np.where(self.flags[i] != 0)[0]))\n        self.meta_counts = np.array(self.meta_counts) if self.meta_counts else np.array([])\n        self.cumsum_meta = np.cumsum(self.meta_counts) if len(self.meta_counts) > 0 else np.array([])\n        print("There are totally", self.cumsum_meta[-1] if len(self.cumsum_meta) > 0 else 0, "cells in this dataset")\n\n    def __len__(self):\n        return self.cumsum_meta[-1] if len(self.cumsum_meta) > 0 else 0\n\n    def __getitem__(self, idx):\n        if len(self.cumsum_meta) == 0:\n            raise RuntimeError("No valid data in dataset")\n        sample_id = np.searchsorted(self.cumsum_meta, idx, side=\'right\')\n        idx = idx - self.cumsum_meta[sample_id]\n        idx = self.arg_meta[sample_id][idx]\n        indices = self.indexes[sample_id][idx]\n        centerx = self.centerx[sample_id][indices]\n        centery = self.centery[sample_id][indices]\n        exp = self.exps[sample_id][indices]\n        type_exp = self.type_exp[sample_id][indices]\n        y = exp[0]\n        cell_types = torch.LongTensor(self.cell_types[sample_id][indices])\n        indices = torch.LongTensor(indices)\n        return {\n            "x": exp,\n            "type_exp": type_exp,\n            "y": y,\n            "cell_types": cell_types,\n            "position_x": centerx,\n            "position_y": centery,\n            "NN": indices\n        }\n', 'spagat/embedding.py': 'import torch\r\nimport torch.nn as nn\r\nimport torch.nn.functional as F\r\nfrom . import preprocess\r\n\r\nclass FFN(nn.Module):\r\n    def __init__(self,dim,in_dim=None,out_dim=None,bias=True):\r\n        super().__init__()\r\n        if in_dim is None:\r\n            in_dim=dim\r\n        if out_dim is None:\r\n            out_dim=dim\r\n\r\n        self.linear1=nn.Linear(in_dim,dim*4,bias=bias)\r\n        self.linear2=nn.Linear(dim*4,out_dim,bias=bias)\r\n        self.act=nn.GELU()\r\n\r\n    def forward(self,x):\r\n        return self.linear2(self.act(self.linear1(x)))\r\n\r\ndef group_strings_by_numbers(strings, numbers):\r\n    """\r\n    Group a list of strings based on unique numbers and return as two ordered lists.\r\n\r\n    :param strings: List of strings to be grouped.\r\n    :param numbers: List of numbers indicating the grouping.\r\n    :return: Two lists, one with unique numbers and the other with corresponding grouped strings.\r\n    """\r\n    numbers=numbers.copy()[:len(strings)]\r\n    unique_numbers = sorted(set(numbers))\r\n    grouped_strings = [[] for _ in unique_numbers]\r\n\r\n    for string, number in zip(strings, numbers):\r\n        index = unique_numbers.index(number)\r\n        grouped_strings[index].append(string)\r\n\r\n    return unique_numbers, grouped_strings\r\n\r\nclass Embedding(nn.Module):\r\n    def __init__(self,genes,ligands_info,node_dim,edge_dim,use_cell_type_embedding=True):\r\n        super().__init__()\r\n        self.use_cell_type_embedding=use_cell_type_embedding\r\n        self.ligands_info=ligands_info\r\n\r\n        # for scaling the data\r\n        #self.exp_scaler=nn.Parameter(torch.ones((len(genes))))\r\n\r\n        # for node\r\n        self.cell_encoder1 = FFN(node_dim, in_dim=len(genes))\r\n        self.cell_encoder2 = nn.Embedding(29,node_dim)\r\n\r\n        # for distance\r\n        self.distance_scaler=nn.Sequential(FFN(edge_dim,out_dim=len(ligands_info[0]),in_dim=5),nn.Sigmoid())\r\n        self.distance_encoder=FFN(edge_dim,out_dim=len(ligands_info[0]),in_dim=5)\r\n        self.distance_embedding = nn.Sequential(FFN(edge_dim, in_dim=5), nn.Sigmoid())\r\n\r\n        # for edge\r\n        self.ligands_encoder=nn.Linear(len(ligands_info[0]),edge_dim)\r\n        self.edge_encoder=nn.Linear(len(ligands_info[0])+edge_dim,edge_dim,bias=False)\r\n        self.scaler=edge_dim**0.5\r\n\r\n        # calculate ligand scores\r\n        self.ligands_index=[]\r\n        for i in range(len(self.ligands_info[0])):\r\n            ligandi = self.ligands_info[0][i]\r\n            ligand_groupi = self.ligands_info[1][i]\r\n            uniquei, groupi = group_strings_by_numbers(ligandi, ligand_groupi)\r\n            groupi=[torch.LongTensor([genes.index(groupi[j][k]) for k in range(len(groupi[j]))]) for j in range(len(groupi))]\r\n            self.ligands_index.append(groupi)\r\n\r\n    def forward(self, x, ligand_feature_mask=None):\r\n        \'\'\'\r\n        :param x:\r\n        x["x"]--b,n,in_node\r\n        x["distance_matrix"]--b,5,n,n\r\n        x["cell_types"]--long,b,n\r\n        x["type_exp"]--b,n,in_node\r\n        :return:\r\n        node_features:b,n,node_dim\r\n        edge1:b,n,n,edge_dim\r\n        edge2:b,n,n,edge_dim+5\r\n        rw: b,n,n,num_hops+5\r\n        \'\'\'\r\n        x=preprocess.process(x)\r\n\r\n        # node features\r\n        #scaler=torch.square(self.exp_scaler)\r\n        exp=x["type_exp"]+x["x"]\r\n        exp=torch.where(x["cell_types"].unsqueeze(dim=-1)==x["cell_types"][:,0:1].unsqueeze(dim=-1),x["type_exp"],exp)\r\n        exp=exp#*scaler\r\n        node_features1=self.cell_encoder1(exp)\r\n        node_features2 = self.cell_encoder1(x["type_exp"])#*scaler)\r\n        node_features=torch.where(x["cell_types"].unsqueeze(dim=-1)==x["cell_types"][:,0:1].unsqueeze(dim=-1),node_features2,node_features1)\r\n\r\n        if self.use_cell_type_embedding:\r\n            node_features=node_features+self.cell_encoder2(x["cell_types"])\r\n\r\n        # calculate ligand scores\r\n        B,N,C=node_features.shape\r\n        data_ligand=[]\r\n        for i in range(len(self.ligands_info[0])):\r\n            data_ligandi=torch.ones((B,N),device=node_features.device)\r\n            for j in range(len(self.ligands_index[i])):\r\n                data_ligandi = data_ligandi * torch.mean(exp[:,:,self.ligands_index[i][j]],dim=-1)\r\n            data_ligandi=torch.pow(data_ligandi,exponent=1 / len(self.ligands_index[i]))\r\n            data_ligand.append(data_ligandi)\r\n        data_ligand=torch.stack(data_ligand,dim=-1)\r\n\r\n        # distance scaler\r\n        distance_scaler = self.distance_scaler(x["distance_matrix"])\r\n\r\n        # ligand score scaled by distance\r\n        ligand_score=distance_scaler*data_ligand\r\n        if ligand_feature_mask is not None:\r\n            ligand_score = ligand_score * ligand_feature_mask\r\n        edge_ligand=self.ligands_encoder(ligand_score)\r\n\r\n        # edge features\r\n        edge_distance=self.distance_encoder(x["distance_matrix"])\r\n        edges = self.edge_encoder(torch.concat([edge_distance,edge_ligand],dim=-1))*self.scaler\r\n\r\n        # distance embedding\r\n        embedding = self.distance_embedding(x["distance_matrix"])\r\n        return [node_features,edges.unsqueeze(dim=1),embedding.unsqueeze(dim=1)]\r\n\r\n\r\n\r\n', 'spagat/gene_program_model.py': '"""\nSpatial Gene Program Model (SpaGP)\n\nCore idea:\n    y_i = μ(cell_type_i) + Σ_k g_k(z_i) · p_k + ε\n\nwhere:\n    - μ(cell_type) is cell-type baseline expression (from type_exp)\n    - p_k ∈ R^genes is the k-th gene program (global, shared, orthogonal)\n    - g_k(z_i) ∈ R+ is scalar activation of program k for cell i (non-negative)\n    - z_i ∈ R^d is spatial context embedding from neighborhood via graph encoder\n"""\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .attention import GRIT_encoder\nfrom .embedding import Embedding, FFN\n\n\nclass GeneProgramHead(nn.Module):\n    """\n    The program-based prediction head.\n    \n    Given spatial context embedding z_i (from graph encoder),\n    predicts gene expression as:\n        y_i = baseline_i + Σ_k g_k(z_i) · p_k\n    """\n    def __init__(self, node_dim, n_genes, n_programs=16, n_neighbors=50,\n                 program_aware=False, message_decoder="free"):\n        super().__init__()\n        self.n_genes = n_genes\n        self.n_programs = n_programs\n        self.n_neighbors = n_neighbors\n        self.program_aware = program_aware\n        self.message_decoder = message_decoder\n        if message_decoder not in {"free", "program_aligned"}:\n            raise ValueError("message_decoder must be \'free\' or \'program_aligned\'.")\n        \n        # ===== Gene Programs p_k ∈ R^{K × genes} =====\n        # Initialized with Xavier, will be orthogonalized during training via loss penalty\n        self.programs = nn.Parameter(torch.empty(n_programs, n_genes))\n        nn.init.xavier_uniform_(self.programs)\n        \n        # ===== Context Aggregator: aggregate neighborhood into z_i =====\n        # Attention-weighted aggregation of neighbor node features → z_i\n        self.context_query = nn.Linear(node_dim, node_dim)\n        self.context_key = nn.Linear(node_dim, node_dim)\n        self.context_proj = nn.Sequential(\n            nn.Linear(node_dim, node_dim),\n            nn.GELU(),\n            nn.Linear(node_dim, node_dim)\n        )\n        self.scale = node_dim ** 0.5\n        \n        # ===== Activation Network: z_i → g(z_i) ∈ R+^K =====\n        self.activation_net = nn.Sequential(\n            nn.Linear(node_dim, node_dim),\n            nn.GELU(),\n            nn.Linear(node_dim, n_programs),\n            nn.Softplus()  # ensures g_k ≥ 0\n        )\n        \n        # ===== Baseline: cell-type expression → gene-level baseline =====\n        self.baseline_head = nn.Sequential(\n            nn.Linear(n_genes, n_genes),\n            nn.GELU(),\n            nn.Linear(n_genes, n_genes)\n        )\n\n        # Only program-aware SpaGP uses this decoder branch.  Keeping it absent\n        # from the legacy construction preserves existing checkpoint layouts.\n        if program_aware:\n            if message_decoder == "free":\n                self.message_to_genes = nn.Linear(node_dim, n_genes)\n            else:\n                # A shared scalar decoder yields q_k from each program message\n                # m_k.  Its gene effect is constrained to that program\'s P_k.\n                self.message_to_program = nn.Linear(node_dim, 1)\n    \n    def forward(self, node_features, type_exp_center, encoder_program_state=None, program_mask=None):\n        """\n        Args:\n            node_features: (B, N, node_dim) — encoder output for center + neighbors\n            type_exp_center: (B, n_genes) — cell-type average expression for center cell\n            encoder_program_state: Optional state returned by a program-aware\n                GRIT encoder.  When supplied, its center-cell activations replace\n                this head\'s legacy context/activation inference.\n            program_mask: Optional inference-only ``(K,)`` 0/1 mask. It is\n                applied only to program-specific decoder contributions; the\n                shared cell-type baseline remains unchanged.\n        \n        Returns:\n            y_pred: (B, n_genes)\n            program_info: dict with activations and programs for downstream analysis\n        """\n        B, N, C = node_features.shape\n        \n        if encoder_program_state is None:\n            # Legacy decoder path: retain the original head computation and\n            # parameter layout so existing SpaGP checkpoints remain usable.\n            center = node_features[:, 0:1, :]\n            neighbors = node_features[:, 1:, :]\n            Q = self.context_query(center)\n            K = self.context_key(neighbors)\n            attn = F.softmax((Q @ K.transpose(-1, -2)) / self.scale, dim=-1)\n            z_i = self.context_proj(attn @ neighbors).squeeze(1)\n            g = self.activation_net(z_i)\n            program_attention = None\n            program_messages = None\n        else:\n            # Program-aware decoder path: graph message passing has already\n            # produced non-negative activations for every node/program pair.\n            # The center-cell row is the activation vector used to decode genes.\n            g = encoder_program_state[\'activations\'][:, 0, :]\n            z_i = node_features[:, 0, :]\n            program_attention = encoder_program_state[\'attention\'][:, 0, 1:, :]\n            program_messages = encoder_program_state[\'messages\'][:, 0, :, :]\n\n            # Keep the historical (B, N-1) attn_weights field as an activation-\n            # weighted summary of the full program-specific attention tensor.\n            gate_weights = g / (torch.sum(g, dim=-1, keepdim=True) + 1e-8)\n            attn = torch.sum(program_attention * gate_weights.unsqueeze(dim=1), dim=-1).unsqueeze(dim=1)\n        \n        # --- Step 3: Normalize programs (unit norm per program) ---\n        programs_normalized = F.normalize(self.programs, dim=-1)  # (K, genes)\n        if program_mask is not None:\n            program_mask = torch.as_tensor(\n                program_mask, device=programs_normalized.device, dtype=programs_normalized.dtype\n            )\n            if program_mask.ndim != 1 or program_mask.shape[0] != self.n_programs:\n                raise ValueError(\n                    f"program_mask must have shape ({self.n_programs},); got {tuple(program_mask.shape)}."\n                )\n        mask_is_identity = program_mask is None or bool(torch.all(program_mask == 1))\n        \n        # --- Step 4: Program contribution ---\n        # Σ_k g_k(z_i) · p_k = g @ P\n        # Keep the default expression byte-for-byte on the existing path.  For\n        # an intervention, mask each program\'s gene-program contribution before\n        # summing across k; no re-normalization occurs.\n        masked_g = g if mask_is_identity else g * program_mask.unsqueeze(0)\n        program_output = masked_g @ programs_normalized   # (B, genes)\n\n        if encoder_program_state is None:\n            message_output = None\n            message_coefficients = None\n        else:\n            if not self.program_aware:\n                raise RuntimeError(\n                    "GeneProgramHead received encoder program state but was not "\n                    "constructed with program_aware=True."\n                )\n            if self.message_decoder == "free":\n                # Legacy program-aware ablation: a flexible shared gene decoder\n                # consumes the gate-weighted sum of program messages.\n                if mask_is_identity:\n                    fused_program_message = torch.sum(\n                        program_messages * gate_weights.unsqueeze(dim=-1), dim=1\n                    )\n                    message_output = self.message_to_genes(fused_program_message)\n                else:\n                    # Distribute the free decoder\'s shared bias by the original\n                    # program weights: sum_k w_k (W m_k + b) equals the legacy\n                    # W sum_k(w_k m_k) + b when every mask is one, while an\n                    # all-zero mask removes the complete program message path.\n                    message_weights = gate_weights * program_mask.unsqueeze(0)\n                    message_components = F.linear(\n                        program_messages, self.message_to_genes.weight, self.message_to_genes.bias\n                    )\n                    message_output = torch.sum(message_components * message_weights.unsqueeze(dim=-1), dim=1)\n                message_coefficients = None\n            else:\n                # Program-aligned decoder: q_k is derived from m_k, then its\n                # contribution is restricted to the corresponding gene program\n                # P_k rather than passing through an unrestricted gene decoder.\n                message_coefficients = self.message_to_program(program_messages).squeeze(dim=-1)\n                masked_coefficients = (\n                    message_coefficients if mask_is_identity\n                    else message_coefficients * program_mask.unsqueeze(0)\n                )\n                message_output = masked_coefficients @ programs_normalized\n        \n        # --- Step 5: Baseline from cell-type expression ---\n        baseline = self.baseline_head(type_exp_center)  # (B, genes)\n        \n        # --- Step 6: Final prediction ---\n        y_pred = baseline + program_output\n        if message_output is not None:\n            y_pred = y_pred + message_output\n        \n        program_info = {\n            \'activations\': g,                    # (B, K) — program activation per cell\n            \'programs\': programs_normalized,      # (K, genes) — gene program vectors\n            \'context\': z_i,                       # (B, C) — spatial context embedding\n            \'attn_weights\': attn.squeeze(1),      # (B, N-1) — neighbor attention weights\n            \'baseline\': baseline,                 # (B, genes) — cell-type baseline\n            \'program_output\': program_output      # (B, genes) — program contribution\n        }\n        if program_attention is not None:\n            program_info[\'program_attention\'] = program_attention\n            program_info[\'program_messages\'] = program_messages\n            program_info[\'program_message_output\'] = message_output\n            program_info[\'program_message_coefficients\'] = message_coefficients\n        if program_mask is not None:\n            program_info[\'program_mask\'] = program_mask\n        \n        return y_pred, program_info\n\n\nclass SpaGP(nn.Module):\n    """\n    Spatial Gene Program Model.\n    """\n    def __init__(self, genes, ligands_info, node_dim=256, edge_dim=48, num_heads=2,\n                 n_layers=1, att_dim=8, use_cell_type_embedding=True,\n                 n_programs=16, program_aware=False, program_token_dim=32,\n                 message_decoder="free", routing_mode="program"):\n        super().__init__()\n        self.genes = genes\n        self.n_programs = n_programs\n        self.program_aware = program_aware\n        self.program_token_dim = program_token_dim\n        self.message_decoder = message_decoder\n        self.routing_mode = routing_mode\n\n        if program_aware and n_layers < 1:\n            raise ValueError("SpaGP program-aware mode requires at least one GRIT encoder layer.")\n        \n        self.embeddings = Embedding(genes, ligands_info, node_dim, edge_dim,\n                                    use_cell_type_embedding=use_cell_type_embedding)\n        self.encoders = nn.ModuleList([\n            GRIT_encoder(node_dim, edge_dim, num_heads, att_dim, program_aware=program_aware, routing_mode=routing_mode)\n            for i in range(n_layers)  \n        ])\n\n        if program_aware:\n            # The learned bank is intentionally in a separate latent space P.\n            # Its shared projection supplies (K, edge_dim) tokens required by\n            # every GRIT layer, keeping program identity fixed across depth.\n            self.program_tokens = nn.Parameter(torch.empty(n_programs, program_token_dim))\n            nn.init.xavier_uniform_(self.program_tokens)\n            self.program_token_projection = nn.Linear(program_token_dim, edge_dim, bias=False)\n\n        self.program_head = GeneProgramHead(\n            node_dim=node_dim,\n            n_genes=len(genes),\n            n_programs=n_programs,\n            program_aware=program_aware,\n            message_decoder=message_decoder,\n        )\n    \n    def forward(self, x, program_mask=None, program_edge_mask=None,ligand_feature_mask=None):\n        """\n        Args:\n            x: dict from dataloader with keys:\n                "x": (B, N, genes), "type_exp": (B, N, genes),\n                "cell_types": (B, N), "position_x": (B, N), "position_y": (B, N),\n                "y": (B, genes)\n        \n        Returns:\n            y_pred: (B, genes)\n            program_info: dict with activations, programs, context, etc.\n        """\n        # Center cell\'s type expression (for baseline)\n        type_exp_center = x["type_exp"][:, 0, :]  # (B, genes)\n        encoded = self.embeddings(x,ligand_feature_mask=ligand_feature_mask)   # [node, edge, distance]\n        if self.program_aware:\n            program_tokens = self.program_token_projection(self.program_tokens)\n            final_program_state = None\n            for encoder in self.encoders:\n                # Program state is analysis metadata; only graph tensors are\n                # forwarded to the next GRIT layer.\n                encoded = encoder(\n                    encoded[:3], program_tokens=program_tokens, program_edge_mask=program_edge_mask\n                )\n                final_program_state = encoded[3]\n        else:\n            for encoder in self.encoders:\n                encoded = encoder(encoded)\n            final_program_state = None\n        node_features = encoded[0]     # (B, N, node_dim)\n        \n        # Program-based prediction\n        y_pred, program_info = self.program_head(\n            node_features, type_exp_center, encoder_program_state=final_program_state,\n            program_mask=program_mask,\n        )\n        \n        return y_pred, program_info\n\n\nclass SpaGP_Loss(nn.Module):\n    """\n    Loss function for SpaGP model:\n        L = L_mse + λ_orth * L_orthogonality + λ_sparse * L_sparsity\n    """\n    def __init__(self, gene_list, interaction_gene_list1, \n                 lambda_orth=0.1, lambda_sparse=0.01):\n        super().__init__()\n        interaction_gene_list = list(set(elem for sublist in interaction_gene_list1[0] for elem in sublist))\n        \n        self.interaction_gene_index = []\n        self.not_interaction_gene_index = []\n        for i in range(len(gene_list)):\n            if gene_list[i] in interaction_gene_list:\n                self.interaction_gene_index.append(i)\n            else:\n                self.not_interaction_gene_index.append(i)\n        self.interaction_gene_index = torch.LongTensor(self.interaction_gene_index)\n        self.not_interaction_gene_index = torch.LongTensor(self.not_interaction_gene_index)\n        \n        self.mse = nn.MSELoss()\n        self.lambda_orth = lambda_orth\n        self.lambda_sparse = lambda_sparse\n    \n    def orthogonality_loss(self, programs):\n        """\n        Soft orthogonality penalty: Σ_{i≠j} (p_i · p_j)²\n        \n        Args:\n            programs: (K, genes) — normalized program vectors\n        """\n        # Gram matrix: (K, K)\n        gram = programs @ programs.T  # p_i · p_j for all pairs\n        # Zero out diagonal (self-similarity = 1, we don\'t penalize that)\n        K = programs.shape[0]\n        if K < 2:\n            # A single program has no program pair, so orthogonality/diversity\n            # is defined exactly as zero rather than evaluating 0 / (K * (K-1)).\n            return programs.new_zeros(())\n        mask = 1.0 - torch.eye(K, device=programs.device)\n        off_diag = gram * mask\n        # Penalty: sum of squared off-diagonal entries\n        return torch.sum(off_diag ** 2) / (K * (K - 1))\n    \n    def sparsity_loss(self, activations):\n        """\n        L1 sparsity on program activations.\n        Encourages each cell to activate only a few programs.\n        \n        Args:\n            activations: (B, K) — non-negative program activations\n        """\n        return torch.mean(activations)  # L1 of non-negative = mean\n    \n    def forward(self, y_pred_tuple, y):\n        """\n        Args:\n            y_pred_tuple: (y_pred, program_info) from model\n            y: (B, genes) ground truth\n        \n        Returns:\n            total_loss: scalar (for backward)\n            mse_not_interact: scalar (for monitoring, detached)\n            orth_loss: scalar (for monitoring, detached)\n            sparse_loss: scalar (for monitoring, detached)\n        """\n        y_pred, program_info = y_pred_tuple\n        \n        # Main prediction loss\n        mse_loss = self.mse(y_pred, y)\n        \n        # Orthogonality penalty\n        orth_loss = self.orthogonality_loss(program_info[\'programs\'])\n        \n        # Sparsity penalty\n        sparse_loss = self.sparsity_loss(program_info[\'activations\'])\n        \n        # Total\n        total_loss = mse_loss + self.lambda_orth * orth_loss + self.lambda_sparse * sparse_loss\n        \n        # Monitoring: MSE on non-interaction genes\n        mse_not_interact = self.mse(\n            y_pred[:, self.not_interaction_gene_index],\n            y[:, self.not_interaction_gene_index]\n        ).detach().cpu()\n        \n        return (total_loss, mse_not_interact, \n                orth_loss.detach().cpu(), sparse_loss.detach().cpu())\nclass SpaGP_NoSpatial(nn.Module):\n    """\n    Ablation model: g_k predicted from center cell expression only.\n    No neighborhood information → if VE ≈ full model, spatial story fails.\n    """\n    def __init__(self, genes, ligands_info, node_dim=256, edge_dim=48, num_heads=2,\n                 n_layers=1, att_dim=8, use_cell_type_embedding=True,\n                 n_programs=16):\n        super().__init__()\n        self.genes = genes\n        self.n_programs = n_programs\n        n_genes = len(genes)\n        \n        # Programs (same as SpaGP)\n        self.programs = nn.Parameter(torch.empty(n_programs, n_genes))\n        nn.init.xavier_uniform_(self.programs)\n        \n        # Activation from center cell expression ONLY (no spatial context)\n        self.activation_net = nn.Sequential(\n            nn.Linear(n_genes, 128),\n            nn.GELU(),\n            nn.Linear(128, n_programs),\n            nn.Softplus()\n        )\n    \n    def forward(self, x):\n        # Only use center cell\'s raw expression — NO neighbors\n        center_exp = x["type_exp"][:, 0, :] + x["x"][:, 0, :]  # (B, genes)\n        \n        g = self.activation_net(center_exp)  # (B, K)\n        programs_norm = F.normalize(self.programs, dim=-1)\n        y_pred = g @ programs_norm\n        \n        program_info = {\n            \'activations\': g,\n            \'programs\': programs_norm,\n            \'context\': center_exp,\n        }\n        return y_pred, program_info\n', 'spagat/preprocess.py': 'import torch\r\n\r\ndef process(x,judge_distance=28):\r\n    \'\'\'\r\n    :param x:\r\n        "x": exp,\r\n        "y": y,\r\n        "ligand":ligand,\r\n        "receptor":receptor,\r\n        "cell_types": cell_types,\r\n        "morphology": volumes,\r\n        "position_x": centerx,\r\n        "position_y": centery\r\n    :return:\r\n        "x": b,n,c1\r\n        "interactions": b,n,n,c2\r\n        "mask": b,n,n\r\n        "cell_types": b,n\r\n        "distance_matrix": b,n,n,c3\r\n        "y": b,c\r\n        "morphology": b,n,c4\r\n    \'\'\'\r\n    dx = x["position_x"][:, 0:1] - x["position_x"]\r\n    dy = x["position_y"][:, 0:1] - x["position_y"]\r\n    distances=torch.sqrt(torch.square(dx)+torch.square(dy))\r\n\r\n    \'\'\'mask=torch.where(distances<judge_distance,\r\n                     torch.ones_like(distances,device=distances.device),\r\n                     torch.zeros_like(distances,device=distances.device))\'\'\'\r\n    distance_matrix = torch.stack([1 / (distances + 1), 1 / (torch.sqrt(distances) + 1),\r\n                                   1 / (1 + torch.square(distances)), torch.exp(-distances),\r\n                                   torch.exp(-torch.square(distances))], dim=-1)\r\n    return {\r\n        "x":x["x"],\r\n        "type_exp": x["type_exp"],\r\n        "y":x["y"],\r\n        "cell_types": x["cell_types"],\r\n        "distance_matrix":distance_matrix\r\n    }', 'runners/GAT.py': '# SEA-AD adaptation of the archived Mouse revision runner.\n# Only dataset metadata, split counts, dimensions checked in main, and log labels change.\n# Network classes, features, losses, optimizers, selection and scoring are retained.\n# Historical class identifiers (MouseNoLog) do NOT perform normalization.\nSEED = 123\n"""Mouse GAT revision: fixed splits, validation PCC selection, one final test."""\nimport argparse\nimport hashlib\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport random\nimport sys\nimport time\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as tnn\nfrom torch.utils.data import DataLoader, Subset\nDEFAULT_REVISION = \'/content/drive/MyDrive/SpaGAT_revision/inductive_training/20260922_104353_552891_UTC_f291de00\'\n\ndef write_json(path, content):\n    with Path(path).open(\'x\', encoding=\'utf-8\') as handle:\n        json.dump(content, handle, indent=2, ensure_ascii=False, allow_nan=False)\n\ndef load_split(path, dataset, data_dir):\n    split = json.loads(Path(path).read_text())\n    keys = (\'train_indices\', \'val_indices\', \'test_indices\')\n    for key in keys:\n        if not isinstance(split.get(key), list) or not split[key] or any((type(i) is not int for i in split[key])):\n            raise ValueError(f\'{key}: expected a nonempty integer list\')\n    indices = [i for key in keys for i in split[key]]\n    if len(indices) != len(dataset) or set(indices) != set(range(len(dataset))):\n        raise ValueError(\'Split is not disjoint, in range and complete\')\n    if Path(split[\'processed_dir\']).resolve() != Path(data_dir).resolve():\n        raise ValueError(\'Split processed_dir mismatch\')\n    for key, count_key in zip(keys, (\'n_train\', \'n_validation\', \'n_test\')):\n        if count_key in split and split[count_key] != len(split[key]):\n            raise ValueError(f\'{count_key} mismatch\')\n    audit = pd.read_csv(Path(path).with_name(Path(path).stem + \'_samples.csv\'))\n    if audit[\'sample\'].tolist() != list(dataset.samples):\n        raise ValueError(\'Sample order differs from verified audit\')\n    counts = np.asarray(dataset.meta_counts, dtype=np.int64)\n    ends = np.cumsum(counts)\n    starts = ends - counts\n    for key, expected in [(\'eligible_cells\', counts), (\'global_start\', starts), (\'global_end_exclusive\', ends)]:\n        if not np.array_equal(audit[key].to_numpy(), expected):\n            raise ValueError(f\'Sample audit {key} mismatch\')\n    membership = np.empty(len(dataset), dtype=np.int8)\n    for label, key in enumerate(keys):\n        membership[split[key]] = label\n    labels = (\'train\', \'validation\', \'test\')\n    for row, start, end in zip(audit.itertuples(index=False), starts, ends):\n        if row.split not in labels or not np.all(membership[start:end] == labels.index(row.split)):\n            raise ValueError(f\'{row.sample}: sample crosses partitions or audit mismatch\')\n    return split\n\ndef evaluate_arrays(prediction, target):\n    """Match the established SpaGAT PCC/MSE conventions; use paper EV_zero."""\n    if prediction.shape != target.shape or not np.isfinite(prediction).all() or (not np.isfinite(target).all()):\n        raise ValueError(\'Invalid prediction/target arrays\')\n    prediction = torch.as_tensor(prediction).float()\n    target = torch.as_tensor(target).float()\n    pc = prediction - prediction.mean(dim=0, keepdim=True)\n    tc = target - target.mean(dim=0, keepdim=True)\n    denominator = torch.sqrt((pc.square().sum(dim=0) * tc.square().sum(dim=0)).clamp_min(1e-12))\n    pcc = ((pc * tc).sum(dim=0) / denominator).clamp(-1, 1)\n    mse = torch.mean((prediction - target).square()).item()\n    mse0 = torch.mean(target.square()).item()\n    metrics = dict(median_gene_pcc=torch.median(pcc).item(), mean_gene_pcc=torch.mean(pcc).item(), mse=mse, mse_zero=mse0, ev_zero_percent=None if mse0 == 0 else 100 * (1 - mse / mse0), n_cells=int(target.shape[0]), n_genes=int(target.shape[1]))\n    return (metrics, pcc.numpy())\n\nclass LocalGATMouseNoLog(tnn.Module):\n\n    def __init__(self, gene_dim=254, num_celltypes=24, hidden_dim=128, type_dim=16, heads=2):\n        super().__init__()\n        from torch_geometric.nn import GATConv\n        self.expr_proj = tnn.Sequential(tnn.Linear(gene_dim, hidden_dim), tnn.GELU())\n        self.type_emb = tnn.Embedding(num_celltypes, type_dim)\n        node_dim = hidden_dim + type_dim\n        self.gat = GATConv(in_channels=node_dim, out_channels=hidden_dim, heads=heads, concat=False, add_self_loops=False, dropout=0.1)\n        self.decoder = tnn.Sequential(tnn.Linear(hidden_dim, hidden_dim), tnn.GELU(), tnn.Linear(hidden_dim, gene_dim))\n\n    def forward(self, x, celltypes):\n        B, K, G = x.shape\n        h_expr = self.expr_proj(x)\n        h_type = self.type_emb(celltypes)\n        h = torch.cat([h_expr, h_type], dim=-1)\n        h = h.reshape(B * K, -1)\n        src = []\n        dst = []\n        for b in range(B):\n            offset = b * K\n            src.append(offset + torch.arange(1, K, device=x.device, dtype=torch.long))\n            dst.append(torch.full((K - 1,), offset, device=x.device, dtype=torch.long))\n        edge_index = torch.stack([torch.cat(src), torch.cat(dst)], dim=0)\n        h = self.gat(h, edge_index)\n        center_idx = torch.arange(B, device=x.device) * K\n        h_center = h[center_idx]\n        return self.decoder(h_center)\n\ndef prepare_batch(batch, device):\n    x, ct, y = (batch[\'x\'], batch[\'cell_types\'], batch[\'y\'])\n    if x.ndim != 3 or x.shape[1] != 50 or ct.shape != x.shape[:2]:\n        raise ValueError(\'Expected aligned 50-cell neighborhoods\')\n    if \'neighbor_mask\' in batch and (not bool(torch.all(batch[\'neighbor_mask\'] == 1))):\n        raise ValueError(\'Expected all 50 nodes valid\')\n    if not torch.equal(y, x[:, 0, :]):\n        raise ValueError(\'Center residual differs from target\')\n    if not torch.isfinite(x).all() or not torch.isfinite(y).all():\n        raise ValueError(\'Nonfinite data\')\n    x = x.masked_fill((ct == ct[:, :1]).unsqueeze(-1), 0)\n    return (x.to(device), ct.to(device), y.to(device))\n\ndef predict(model, dataset, indices, device, batch_size, label):\n    model.eval()\n    predictions, targets = ([], [])\n    loader = DataLoader(Subset(dataset, indices), batch_size=batch_size, shuffle=False, num_workers=0)\n    with torch.no_grad():\n        for step, batch in enumerate(loader, 1):\n            x, ct, y = prepare_batch(batch, device)\n            predictions.append(model(x, ct).cpu().numpy())\n            targets.append(y.cpu().numpy())\n            if step == 1 or step % 200 == 0 or step == len(loader):\n                print(f\'{label}: batch {step}/{len(loader)}\', flush=True)\n    return (np.concatenate(predictions), np.concatenate(targets))\n\ndef run_experiment(dataset, split, genes, out, model, device, max_epochs=10, patience=5, batch_size=32):\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=False)\n    import shutil\n    shutil.copyfile(__file__, out / \'runner_source.py\')\n    write_json(out / \'shared_split.json\', split)\n    config = dict(dataset=\'SEA_AD\', model=\'GAT\', seed=SEED, max_epochs=max_epochs, patience=patience, batch_size=batch_size, learning_rate=0.0001, optimizer=\'AdamW\', weight_decay=0.01, betas=[0.9, 0.999], hidden_dim=128, type_dim=16, heads=2, dropout=0.1, add_self_loops=False, edges=\'49 neighbors to center\', input=\'residual expression plus learned type embedding; center and homotypic residuals zeroed; no baseline or distance features added\', selection=\'validation median_gene_pcc; strict improvement; first epoch wins ties\', epoch_numbering=\'zero-based\', monitored_metric=\'median_gene_pcc\', target_space=\'residual relative to training-derived cell-type baseline\', pcc=\'float32 centered; denominator product clamped at 1e-12; torch.median; constant columns score zero\', ev=\'100*(1-MSE/MSE0)\', parameter_count=sum((p.numel() for p in model.parameters())), device=str(device), gpu=torch.cuda.get_device_name() if device.type == \'cuda\' else None, versions=dict(python=sys.version, torch=torch.__version__, numpy=np.__version__, pandas=pd.__version__))\n    if \'torch_geometric\' in sys.modules:\n        config[\'versions\'][\'torch_geometric\'] = sys.modules[\'torch_geometric\'].__version__\n    write_json(out / \'run_config.json\', config)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001)\n    train_loader = DataLoader(Subset(dataset, split[\'train_indices\']), batch_size=batch_size, shuffle=True, num_workers=0, generator=torch.Generator().manual_seed(SEED))\n    best_score, best_epoch, stale = (-float(\'inf\'), None, 0)\n    records = []\n    checkpoint = out / f\'gat_seed{SEED}_best.pth\'\n    started = time.perf_counter()\n    for epoch in range(max_epochs):\n        model.train()\n        total, count = (0.0, 0)\n        for step, batch in enumerate(train_loader, 1):\n            x, ct, y = prepare_batch(batch, device)\n            optimizer.zero_grad(set_to_none=True)\n            prediction = model(x, ct)\n            loss = tnn.functional.mse_loss(prediction, y)\n            if not torch.isfinite(loss):\n                raise ValueError(\'Nonfinite training loss; test will not be evaluated\')\n            loss.backward()\n            optimizer.step()\n            total += loss.item() * len(y)\n            count += len(y)\n            if step == 1 or step % 200 == 0 or step == len(train_loader):\n                print(f\'Epoch {epoch}: train batch {step}/{len(train_loader)}, MSE {total / count:.6f}\', flush=True)\n        pred, target = predict(model, dataset, split[\'val_indices\'], device, batch_size, \'validation\')\n        metrics, _ = evaluate_arrays(pred, target)\n        score = metrics[\'median_gene_pcc\']\n        if not np.isfinite(score):\n            raise ValueError(\'Nonfinite validation PCC; test will not be evaluated\')\n        improved = score > best_score\n        if improved:\n            best_score, best_epoch, stale = (score, epoch, 0)\n            torch.save(dict(model_state_dict=model.state_dict(), best_epoch=epoch, best_validation_pcc=score), checkpoint)\n        else:\n            stale += 1\n        records.append(dict(epoch=epoch, train_mse=total / count, validation_median_pcc=score, validation_mse=metrics[\'mse\'], best_epoch=best_epoch, elapsed_seconds=time.perf_counter() - started))\n        pd.DataFrame(records).to_csv(out / \'training_history.csv\', index=False)\n        print(f\'Epoch {epoch} completed: validation median PCC={score:.6f}; best epoch={best_epoch}\', flush=True)\n        del pred, target\n        if stale >= patience:\n            print(\'Validation early stopping.\', flush=True)\n            break\n    if best_epoch is None:\n        raise RuntimeError(\'No valid checkpoint; test will not be evaluated\')\n    saved = torch.load(checkpoint, map_location=device, weights_only=True)\n    model.load_state_dict(saved[\'model_state_dict\'])\n    print(f"Loaded best checkpoint (epoch {saved[\'best_epoch\']}). Starting final test once.", flush=True)\n    pred, target = predict(model, dataset, split[\'test_indices\'], device, batch_size, \'test\')\n    metrics, pcc = evaluate_arrays(pred, target)\n    metrics.update(dataset=\'SEA_AD\', model=\'GAT\', seed=SEED, best_epoch=saved[\'best_epoch\'], best_validation_pcc=saved[\'best_validation_pcc\'], checkpoint=checkpoint.name, runtime_seconds=time.perf_counter() - started, target_space=config[\'target_space\'])\n    np.savez_compressed(out / \'test_predictions.npz\', prediction=pred, target=target, test_indices=np.asarray(split[\'test_indices\'], dtype=np.int64), genes=np.asarray(genes, dtype=str))\n    pd.DataFrame({\'gene\': genes, \'PCC\': pcc}).to_csv(out / \'test_per_gene_PCC.csv\', index=False)\n    write_json(out / \'test_metrics.json\', metrics)\n    pd.DataFrame([metrics]).to_csv(out / \'test_metrics.csv\', index=False)\n    pd.DataFrame([dict(seed=SEED, best_epoch=best_epoch, best_validation_pcc=best_score, monitored_metric=\'median_gene_pcc\', parameter_count=config[\'parameter_count\'], max_epochs=max_epochs, patience=patience)]).to_csv(out / \'best_runs.csv\', index=False)\n    print(json.dumps(metrics, indent=2), flush=True)\n    print(\'Completed; saved to:\', out, flush=True)\n    return metrics\n\ndef main():\n    global SEED\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--revision-dir\', default=DEFAULT_REVISION)\n    parser.add_argument(\'--output-dir\', default=None)\n    parser.add_argument(\'--seed\', type=int, default=123)\n    args = parser.parse_args()\n    SEED = args.seed\n    revision = Path(args.revision_dir)\n    out = Path(args.output_dir) if args.output_dir else revision / \'baseline_results\' / f\'SEA_AD_GAT_seed{SEED}\'\n    if out.exists():\n        raise FileExistsError(f\'Refusing to overwrite: {out}\')\n    random.seed(SEED)\n    np.random.seed(SEED)\n    torch.manual_seed(SEED)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(SEED)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    print(\'Device:\', device, flush=True)\n    from torch_geometric.nn import GATConv\n    split_path = revision / \'splits\' / \'SEA_AD_inductive_split_indices.json\'\n    metadata = json.loads(split_path.read_text())\n    if tuple((len(metadata[k]) for k in (\'train_indices\', \'val_indices\', \'test_indices\'))) != (257152, 54026, 54762):\n        raise ValueError(\'Unexpected SEA-AD split counts\')\n    loader_path = revision / \'spagat\' / \'dataloader.py\'\n    manifest = json.loads((revision / \'revision_manifest.json\').read_text())\n    digest = hashlib.sha256(loader_path.read_bytes()).hexdigest()\n    if digest != manifest[\'files_sha256\'][\'spagat/dataloader.py\']:\n        raise ValueError(\'Frozen dataloader hash mismatch\')\n    spec = importlib.util.spec_from_file_location(\'frozen_spagat_data\', loader_path)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    print(\'Loading verified processed data...\', flush=True)\n    dataset = module.SPAGAT_dataset(processed_dir=metadata[\'processed_dir\'], num_neighbors=50)\n    split = load_split(split_path, dataset, metadata[\'processed_dir\'])\n    genes = list(dataset.genes)\n    if len(genes) != 140 or len(set(genes)) != 140:\n        raise ValueError(\'Expected 140 unique genes\')\n    model = LocalGATMouseNoLog(gene_dim=len(genes), num_celltypes=len(dataset.cell_types_dict)).to(device)\n    print(f\'Split audit passed. Starting SEA-AD GAT, seed {SEED}, max 10 epochs, patience 5.\', flush=True)\n    run_experiment(dataset, split, genes, out, model, device)\n    write_json(out / \'source_provenance.json\', dict(revision=str(revision), split_sha256=hashlib.sha256(split_path.read_bytes()).hexdigest(), dataloader_sha256=digest, runner_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()))\nif __name__ == \'__main__\':\n    main()\n', 'runners/GITIII_official.py': '# SEA-AD adaptation of the archived Mouse revision runner.\n# Only dataset metadata, split counts, dimensions checked in main, and log labels change.\n# Network classes, features, losses, optimizers, selection and scoring are retained.\n# Historical class identifiers (MouseNoLog) do NOT perform normalization.\nSEED = 123\n"""Mouse official GITIII revision: fixed splits, validation PCC selection, one final test."""\nimport argparse\nimport hashlib\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport random\nimport sys\nimport time\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as tnn\nfrom torch.utils.data import DataLoader, Subset\nDEFAULT_REVISION = \'/content/drive/MyDrive/SpaGAT_revision/inductive_training/20260922_104353_552891_UTC_f291de00\'\nOFFICIAL_REPOSITORY = \'https://github.com/lugia-xiao/GITIII\'\nOFFICIAL_COMMIT = \'94024679f722c7342a1b3dbcc0cdd99f415c3695\'\nOFFICIAL_HASHES = {\'preprocess.py\': \'9bb77b4e200f8b1456bf2eb8901982816c0706be5878cea7f5e54e19676b7490\', \'embedding.py\': \'6aff516f9905da7741168e42c0c8a3a40a705fbc6321cf56bffafe7220e6a091\', \'attention.py\': \'adc62546618eaea4155a2c0b3e1b4cb03f1c60096488f3c65bba35bd3b56bdd6\', \'model.py\': \'ccb91bd1e2c3f3b2cdee18c9e3981f7ce7085403b5923684c856e1bb8b9f2cb6\'}\nOFFICIAL_SOURCES = {\'preprocess.py\': \'import torch\\r\\n\\r\\ndef process(x,judge_distance=28):\\r\\n    \\\'\\\'\\\'\\r\\n    :param x:\\r\\n        "x": exp,\\r\\n        "y": y,\\r\\n        "ligand":ligand,\\r\\n        "receptor":receptor,\\r\\n        "cell_types": cell_types,\\r\\n        "morphology": volumes,\\r\\n        "position_x": centerx,\\r\\n        "position_y": centery\\r\\n    :return:\\r\\n        "x": b,n,c1\\r\\n        "interactions": b,n,n,c2\\r\\n        "mask": b,n,n\\r\\n        "cell_types": b,n\\r\\n        "distance_matrix": b,n,n,c3\\r\\n        "y": b,c\\r\\n        "morphology": b,n,c4\\r\\n    \\\'\\\'\\\'\\r\\n    dx = x["position_x"][:, 0:1] - x["position_x"]\\r\\n    dy = x["position_y"][:, 0:1] - x["position_y"]\\r\\n    distances=torch.sqrt(torch.square(dx)+torch.square(dy))\\r\\n\\r\\n    \\\'\\\'\\\'mask=torch.where(distances<judge_distance,\\r\\n                     torch.ones_like(distances,device=distances.device),\\r\\n                     torch.zeros_like(distances,device=distances.device))\\\'\\\'\\\'\\r\\n    distance_matrix = torch.stack([1 / (distances + 1), 1 / (torch.sqrt(distances) + 1),\\r\\n                                   1 / (1 + torch.square(distances)), torch.exp(-distances),\\r\\n                                   torch.exp(-torch.square(distances))], dim=-1)\\r\\n    return {\\r\\n        "x":x["x"],\\r\\n        "type_exp": x["type_exp"],\\r\\n        "y":x["y"],\\r\\n        "cell_types": x["cell_types"],\\r\\n        "distance_matrix":distance_matrix\\r\\n    }\', \'embedding.py\': \'import torch\\r\\nimport torch.nn as nn\\r\\nimport torch.nn.functional as F\\r\\nimport gitiii.preprocess as preprocess\\r\\n\\r\\nclass FFN(nn.Module):\\r\\n    def __init__(self,dim,in_dim=None,out_dim=None,bias=True):\\r\\n        super().__init__()\\r\\n        if in_dim is None:\\r\\n            in_dim=dim\\r\\n        if out_dim is None:\\r\\n            out_dim=dim\\r\\n\\r\\n        self.linear1=nn.Linear(in_dim,dim*4,bias=bias)\\r\\n        self.linear2=nn.Linear(dim*4,out_dim,bias=bias)\\r\\n        self.act=nn.GELU()\\r\\n\\r\\n    def forward(self,x):\\r\\n        return self.linear2(self.act(self.linear1(x)))\\r\\n\\r\\ndef group_strings_by_numbers(strings, numbers):\\r\\n    """\\r\\n    Group a list of strings based on unique numbers and return as two ordered lists.\\r\\n\\r\\n    :param strings: List of strings to be grouped.\\r\\n    :param numbers: List of numbers indicating the grouping.\\r\\n    :return: Two lists, one with unique numbers and the other with corresponding grouped strings.\\r\\n    """\\r\\n    numbers=numbers.copy()[:len(strings)]\\r\\n    unique_numbers = sorted(set(numbers))\\r\\n    grouped_strings = [[] for _ in unique_numbers]\\r\\n\\r\\n    for string, number in zip(strings, numbers):\\r\\n        index = unique_numbers.index(number)\\r\\n        grouped_strings[index].append(string)\\r\\n\\r\\n    return unique_numbers, grouped_strings\\r\\n\\r\\nclass Embedding(nn.Module):\\r\\n    def __init__(self,genes,ligands_info,node_dim,edge_dim,use_cell_type_embedding=True):\\r\\n        super().__init__()\\r\\n        self.use_cell_type_embedding=use_cell_type_embedding\\r\\n        self.ligands_info=ligands_info\\r\\n\\r\\n        # for scaling the data\\r\\n        #self.exp_scaler=nn.Parameter(torch.ones((len(genes))))\\r\\n\\r\\n        # for node\\r\\n        self.cell_encoder1 = FFN(node_dim, in_dim=len(genes))\\r\\n        self.cell_encoder2 = nn.Embedding(29,node_dim)\\r\\n\\r\\n        # for distance\\r\\n        self.distance_scaler=nn.Sequential(FFN(edge_dim,out_dim=len(ligands_info[0]),in_dim=5),nn.Sigmoid())\\r\\n        self.distance_encoder=FFN(edge_dim,out_dim=len(ligands_info[0]),in_dim=5)\\r\\n        self.distance_embedding = nn.Sequential(FFN(edge_dim, in_dim=5), nn.Sigmoid())\\r\\n\\r\\n        # for edge\\r\\n        self.ligands_encoder=nn.Linear(len(ligands_info[0]),edge_dim)\\r\\n        self.edge_encoder=nn.Linear(len(ligands_info[0])+edge_dim,edge_dim,bias=False)\\r\\n        self.scaler=edge_dim**0.5\\r\\n\\r\\n        # calculate ligand scores\\r\\n        self.ligands_index=[]\\r\\n        for i in range(len(self.ligands_info[0])):\\r\\n            ligandi = self.ligands_info[0][i]\\r\\n            ligand_groupi = self.ligands_info[1][i]\\r\\n            uniquei, groupi = group_strings_by_numbers(ligandi, ligand_groupi)\\r\\n            groupi=[torch.LongTensor([genes.index(groupi[j][k]) for k in range(len(groupi[j]))]) for j in range(len(groupi))]\\r\\n            self.ligands_index.append(groupi)\\r\\n\\r\\n    def forward(self,x):\\r\\n        \\\'\\\'\\\'\\r\\n        :param x:\\r\\n        x["x"]--b,n,in_node\\r\\n        x["distance_matrix"]--b,5,n,n\\r\\n        x["cell_types"]--long,b,n\\r\\n        x["type_exp"]--b,n,in_node\\r\\n        :return:\\r\\n        node_features:b,n,node_dim\\r\\n        edge1:b,n,n,edge_dim\\r\\n        edge2:b,n,n,edge_dim+5\\r\\n        rw: b,n,n,num_hops+5\\r\\n        \\\'\\\'\\\'\\r\\n        x=preprocess.process(x)\\r\\n\\r\\n        # node features\\r\\n        #scaler=torch.square(self.exp_scaler)\\r\\n        exp=x["type_exp"]+x["x"]\\r\\n        exp=torch.where(x["cell_types"].unsqueeze(dim=-1)==x["cell_types"][:,0:1].unsqueeze(dim=-1),x["type_exp"],exp)\\r\\n        exp=exp#*scaler\\r\\n        node_features1=self.cell_encoder1(exp)\\r\\n        node_features2 = self.cell_encoder1(x["type_exp"])#*scaler)\\r\\n        node_features=torch.where(x["cell_types"].unsqueeze(dim=-1)==x["cell_types"][:,0:1].unsqueeze(dim=-1),node_features2,node_features1)\\r\\n\\r\\n        if self.use_cell_type_embedding:\\r\\n            node_features=node_features+self.cell_encoder2(x["cell_types"])\\r\\n\\r\\n        # calculate ligand scores\\r\\n        B,N,C=node_features.shape\\r\\n        data_ligand=[]\\r\\n        for i in range(len(self.ligands_info[0])):\\r\\n            data_ligandi=torch.ones((B,N),device=node_features.device)\\r\\n            for j in range(len(self.ligands_index[i])):\\r\\n                data_ligandi = data_ligandi * torch.mean(exp[:,:,self.ligands_index[i][j]],dim=-1)\\r\\n            data_ligandi=torch.pow(data_ligandi,exponent=1 / len(self.ligands_index[i]))\\r\\n            data_ligand.append(data_ligandi)\\r\\n        data_ligand=torch.stack(data_ligand,dim=-1)\\r\\n\\r\\n        # distance scaler\\r\\n        distance_scaler = self.distance_scaler(x["distance_matrix"])\\r\\n\\r\\n        # ligand score scaled by distance\\r\\n        ligand_score=distance_scaler*data_ligand\\r\\n        edge_ligand=self.ligands_encoder(ligand_score)\\r\\n\\r\\n        # edge features\\r\\n        edge_distance=self.distance_encoder(x["distance_matrix"])\\r\\n        edges = self.edge_encoder(torch.concat([edge_distance,edge_ligand],dim=-1))*self.scaler\\r\\n\\r\\n        # distance embedding\\r\\n        embedding = self.distance_embedding(x["distance_matrix"])\\r\\n        return [node_features,edges.unsqueeze(dim=1),embedding.unsqueeze(dim=1)]\\r\\n\\r\\n\\r\\n\\r\\n\', \'attention.py\': \'import torch\\r\\nimport torch.nn as nn\\r\\nimport torch.nn.functional as F\\r\\n\\r\\nfrom gitiii.embedding import FFN\\r\\n\\r\\ndef rho(x):\\r\\n    return torch.sqrt(F.relu(x)) - torch.sqrt(F.relu(-x))\\r\\n\\r\\ndef get_diagonal(x):\\r\\n    index=torch.arange(0,x.shape[1],1)\\r\\n    return x[:,index,index,:]\\r\\n\\r\\nclass GRIT_attention(nn.Module):\\r\\n    def __init__(self, node_dim, edge_dim,att_dim=8):\\r\\n        super().__init__()\\r\\n        self.edge_dim=edge_dim*2\\r\\n\\r\\n        self.W_Q = nn.Linear(node_dim, self.edge_dim*att_dim)\\r\\n        self.W_K = nn.Linear(node_dim, self.edge_dim*att_dim)\\r\\n        self.W_V = nn.Linear(node_dim, node_dim)\\r\\n        self.W_Ew = nn.Linear(edge_dim, edge_dim, bias=False)\\r\\n        self.W_Eb = nn.Linear(edge_dim, edge_dim, bias=False)\\r\\n        self.W_En = nn.Linear(edge_dim, node_dim)\\r\\n        self.W_A = nn.Linear(edge_dim, 1, bias=False)\\r\\n\\r\\n        self.W_No = nn.Linear(node_dim, node_dim)\\r\\n        self.W_Eo = nn.Linear(edge_dim, edge_dim, bias=False)\\r\\n\\r\\n    def forward(self, x):\\r\\n        node, edge, embedding = x\\r\\n        B, N, C = node.shape\\r\\n\\r\\n        Q = self.W_Q(node).reshape(B,N,-1,self.edge_dim).permute(0,3,1,2)#.unsqueeze(dim=1).repeat(1,N,1,1)#b,n,h*c->b,n,c,h->b,h,n,c\\r\\n        K = self.W_K(node).reshape(B,N,-1,self.edge_dim).permute(0,3,2,1)#.unsqueeze(dim=2).repeat(1,1,N,1)#b,n,h*c->b,n,c,h->b,h,c,n\\r\\n        QK=(Q@K).permute(0,2,3,1)\\r\\n        #print("abs", torch.mean(torch.abs(rho((QK[:,:,:,:self.edge_dim//2])*self.W_Ew(edge)))), torch.mean(torch.abs(self.W_Eb(edge))),torch.mean(torch.abs(QK[:,:,:,self.edge_dim//2:]*embedding)))\\r\\n        edge = F.gelu(rho((QK[:,:,:,:self.edge_dim//2])*self.W_Ew(edge))+self.W_Eb(edge)+QK[:,:,:,self.edge_dim//2:]+embedding)  # b,n,n,c_e\\r\\n\\r\\n        alpha=self.W_A(edge).squeeze(dim=-1)\\r\\n        alpha = F.softmax(alpha,dim=-1)\\r\\n\\r\\n        V = self.W_V(node)\\r\\n        node = alpha @ V + self.W_En(torch.sum(edge*alpha.unsqueeze(dim=-1),dim=-2))\\r\\n\\r\\n        node = self.W_No(node)\\r\\n        edge = self.W_Eo(edge)\\r\\n        return [node, edge]\\r\\n\\r\\nclass DegScaler(nn.Module):\\r\\n    def __init__(self, node_dim):\\r\\n        super().__init__()\\r\\n        scaler = (2 / node_dim) ** 0.5  # He Initialization\\r\\n        self.theta1 = nn.Parameter(torch.randn(node_dim) * scaler)\\r\\n        self.theta2 = nn.Parameter(torch.randn(node_dim) * scaler)\\r\\n\\r\\n    def forward(self, node, degree):\\r\\n        node = self.theta1 * node + torch.log(degree + 1).unsqueeze(dim=-1) * node * self.theta2\\r\\n        return node\\r\\n\\r\\nclass Multi_Head_Attention(nn.Module):\\r\\n    def __init__(self, node_dim, edge_dim, num_heads, att_dim=8):\\r\\n        super().__init__()\\r\\n        self.attentions = nn.ModuleList(\\r\\n            [GRIT_attention(node_dim, edge_dim, att_dim) for i in range(num_heads)]\\r\\n        )\\r\\n\\r\\n        self.W_hn = nn.Parameter(torch.ones(num_heads, 1)/num_heads)\\r\\n        self.W_he = nn.Parameter(torch.ones(num_heads, 1) / num_heads)\\r\\n\\r\\n    def forward(self, x):\\r\\n        results = []\\r\\n        for attentioni in self.attentions:\\r\\n            results.append(attentioni(x))\\r\\n\\r\\n        node = (torch.stack([tmp[0] for tmp in results], dim=-1)@self.W_hn).squeeze(dim=-1)\\r\\n\\r\\n        edge = [tmp[1] for tmp in results]\\r\\n        edge = torch.stack(edge, dim=-1)@self.W_he\\r\\n        edge = edge.squeeze(dim=-1)\\r\\n\\r\\n        return [node, edge, x[2]]\\r\\n\\r\\nclass GRIT_encoder(nn.Module):\\r\\n    def __init__(self, node_dim, edge_dim, num_heads, att_dim=8):\\r\\n        super().__init__()\\r\\n        self.attentions = Multi_Head_Attention(node_dim, edge_dim, num_heads, att_dim)\\r\\n        self.FFN = FFN(node_dim)\\r\\n\\r\\n        self.ln1 = nn.LayerNorm(node_dim)\\r\\n        self.ln2 = nn.LayerNorm(node_dim)\\r\\n        self.ln_edge=nn.LayerNorm(edge_dim)\\r\\n\\r\\n    def forward(self, x):\\r\\n        node, edge, distance = x\\r\\n\\r\\n        x = self.attentions(x)\\r\\n\\r\\n        edge = self.ln_edge(edge + x[1])\\r\\n\\r\\n        node = self.ln1(node+ x[0])\\r\\n        node = self.ln2(node + self.FFN(node))\\r\\n        return [node, edge, distance]\\r\\n\\r\\nclass GRIT_encoder_last_layer(nn.Module):\\r\\n    def __init__(self, node_dim, in_node, edge_dim, node_dim_small=16, att_dim=8):\\r\\n        super().__init__()\\r\\n        self.node_dim_small = node_dim_small\\r\\n        self.in_node = in_node\\r\\n        self.edge_dim=edge_dim*2\\r\\n        self.att_dim=att_dim\\r\\n\\r\\n        self.W_Q = nn.Linear(node_dim, self.edge_dim * att_dim)\\r\\n        self.W_K = nn.Linear(node_dim, self.edge_dim * att_dim)\\r\\n        self.W_Ew = nn.Linear(edge_dim, edge_dim, bias=False)\\r\\n        self.W_Eb = nn.Linear(edge_dim, edge_dim, bias=False)\\r\\n        self.W_En = nn.Linear(edge_dim, node_dim)\\r\\n\\r\\n        self.W_A=nn.Linear(edge_dim,in_node,bias=False)\\r\\n\\r\\n        self.node_transform = nn.Linear(node_dim, in_node)\\r\\n        self.edge_transform = nn.Linear(edge_dim, in_node)\\r\\n        self.head = FFN(in_node,in_dim=in_node*2, out_dim=in_node)\\r\\n        self.head2=nn.Sequential(nn.LayerNorm(in_node),FFN(in_node))\\r\\n\\r\\n        self.scaler=2/(node_dim**0.5)\\r\\n\\r\\n    def forward(self, x):\\r\\n        node, edge, embedding = x\\r\\n        B, N, C = node.shape\\r\\n\\r\\n        Q = self.W_Q(node[:,0:1,:]).reshape(B, 1, self.att_dim, self.edge_dim).permute(0,3,1,2)#b,n,h*c->b,n,c,h->b,h,n,c\\r\\n        K = self.W_K(node).reshape(B, N, self.att_dim, self.edge_dim).permute(0,3,2,1)#b,n,h*c->b,n,c,h->b,h,c,n\\r\\n        QK = (Q @ K).permute(0, 2, 3, 1)\\r\\n\\r\\n        edge=edge[:,0:1,:,:]\\r\\n        #print("abs",torch.mean(torch.abs(QK)),torch.mean(torch.abs(edge)))\\r\\n        edge = F.gelu(rho((QK[:, :, :, :self.edge_dim // 2]) * self.W_Ew(edge)) + self.W_Eb(edge) + QK[:, :, :, self.edge_dim // 2:] + embedding[:,0:1,:,:])  # b,n,n,c_e\\r\\n        edge = edge[:, 0, 1:, :]\\r\\n\\r\\n        alphas = F.softmax(self.W_A(edge).permute(0, 2, 1), dim=-1).permute(0, 2, 1)\\r\\n\\r\\n        edge = self.edge_transform(edge)*alphas\\r\\n        node = self.node_transform(node[:, 1:, :])*alphas\\r\\n        tmp = self.head(torch.concat([edge, node], dim=-1))\\r\\n\\r\\n        node = torch.sum(tmp,dim=-2)*self.scaler  # b,in_node,99->1\\r\\n        return [node, [tmp.permute(0,2,1).unsqueeze(dim=-2), edge.permute(0,2,1).unsqueeze(dim=-2)]]\\r\\n\', \'model.py\': \'import torch\\r\\nimport torch.nn as nn\\r\\nimport numpy as np\\r\\nimport random\\r\\nimport os\\r\\n\\r\\nfrom gitiii.attention import GRIT_encoder,GRIT_encoder_last_layer\\r\\nfrom gitiii.embedding import Embedding\\r\\n\\r\\nclass GITIII(nn.Module):\\r\\n    def __init__(self,genes,ligands_info,node_dim,edge_dim,num_heads,n_layers,\\r\\n                 node_dim_small=16,att_dim=8,use_cell_type_embedding=True):\\r\\n        super().__init__()\\r\\n        self.embeddings=Embedding(genes,ligands_info,node_dim,edge_dim,\\r\\n                                  use_cell_type_embedding=use_cell_type_embedding)\\r\\n        self.encoders=nn.ModuleList([\\r\\n            GRIT_encoder(node_dim,edge_dim,num_heads,att_dim) for i in range(n_layers-1)\\r\\n        ])\\r\\n        self.last_layer=GRIT_encoder_last_layer(node_dim, len(genes), edge_dim, node_dim_small, att_dim)\\r\\n\\r\\n    def forward(self,x):\\r\\n        x=self.embeddings(x)\\r\\n        for encoderi in self.encoders:\\r\\n            x=encoderi(x)\\r\\n        x=self.last_layer(x)\\r\\n        return x\\r\\n\\r\\nclass Loss_function(nn.Module):\\r\\n    def __init__(self,gene_list,interaction_gene_list1):\\r\\n        super().__init__()\\r\\n        interaction_gene_list=list(set(elem for sublist in interaction_gene_list1[0] for elem in sublist))\\r\\n\\r\\n        self.interaction_gene_index = []\\r\\n        self.not_interaction_gene_index = []\\r\\n        for i in range(len(gene_list)):\\r\\n            if gene_list[i] in interaction_gene_list:\\r\\n                self.interaction_gene_index.append(i)\\r\\n            else:\\r\\n                self.not_interaction_gene_index.append(i)\\r\\n        self.interaction_gene_index = torch.LongTensor(self.interaction_gene_index).cuda()\\r\\n        self.not_interaction_gene_index = torch.LongTensor(self.not_interaction_gene_index).cuda()\\r\\n\\r\\n        self.mse=nn.MSELoss()\\r\\n\\r\\n    def forward(self,y_pred,y):\\r\\n        output,edges=y_pred\\r\\n        mse_loss=self.mse(output,y)\\r\\n        return (mse_loss,\\r\\n                self.mse(output[:,self.not_interaction_gene_index],y[:,self.not_interaction_gene_index]).detach().cpu())\\r\\n\\r\\ndef set_seed(seed_value=42):\\r\\n    """Set seed for reproducibility."""\\r\\n    # Python random module\\r\\n    random.seed(seed_value)\\r\\n    # Numpy\\r\\n    np.random.seed(seed_value)\\r\\n    # PyTorch\\r\\n    torch.manual_seed(seed_value)\\r\\n    # CUDA\\r\\n    torch.cuda.manual_seed(seed_value)\\r\\n    torch.cuda.manual_seed_all(seed_value)  # if using multi-GPU\\r\\n    # Additional configurations to enforce deterministic behavior\\r\\n    torch.backends.cudnn.deterministic = True\\r\\n    torch.backends.cudnn.benchmark = False\\r\\n    # Setting environment variables to further ensure reproducibility\\r\\n    os.environ[\\\'PYTHONHASHSEED\\\'] = str(seed_value)\\r\\n\\r\\nset_seed(123)\\r\\n\\r\\nif __name__=="__main__":\\r\\n    import os\\r\\n\\r\\n    torch.cuda.empty_cache()\\r\\n    batch_size = 256\\r\\n    lr = 1e-4\\r\\n    folds = 5\\r\\n    epochs = 200\\r\\n\\r\\n    data_dir = "D:/project/tmp/st/"\\r\\n    from torch.utils.data import DataLoader, random_split\\r\\n\\r\\n    ligands_info = torch.load("/".join(data_dir.split("/")[:-2]) + "/ligands.pth",weights_only=False)\\r\\n    genes = torch.load("/".join(data_dir.split("/")[:-2]) + "/genes.pth",weights_only=False)\\r\\n\\r\\n    my_model = GITIII(genes, ligands_info, node_dim=256, edge_dim=48, num_heads=2, node_dim_small=16, att_dim=8,n_layers=1)\\r\\n    my_model = my_model.cuda()\\r\\n    optimizer = torch.optim.AdamW(my_model.parameters(), lr=lr, betas=(0.99, 0.999))\\r\\n    loss_func = Loss_function(genes, ligands_info).cuda()\\r\\n    import time\\r\\n\\r\\n    for i in range(20):\\r\\n        print("start training")\\r\\n        start = time.time()\\r\\n        for i in range(10):\\r\\n            batch = {\\r\\n                "x": torch.randn((batch_size, 50, len(genes))),\\r\\n                "type_exp": torch.randn((batch_size, 50, len(genes))),\\r\\n                "y": torch.randn((batch_size, len(genes))),\\r\\n                "cell_types": torch.LongTensor([1]).unsqueeze(dim=0).repeat((batch_size, 50)),\\r\\n                "position_x": torch.arange(1, 201, 4).unsqueeze(dim=0).repeat((batch_size, 1)),\\r\\n                "position_y": torch.arange(1, 201, 4).unsqueeze(dim=0).repeat((batch_size, 1))\\r\\n            }\\r\\n            batch = {k: v.cuda() for k, v in batch.items()}\\r\\n            \\\'\\\'\\\'for k, v in batch.items():\\r\\n                print(k,v.shape,torch.mean(torch.abs(v.float())))\\\'\\\'\\\'\\r\\n            output = my_model(batch)\\r\\n            print("output", output[0].shape, torch.mean(torch.abs(output[0])), torch.mean(output[0]),\\r\\n                  torch.mean(torch.abs(batch["y"])), torch.mean(batch["y"]))\\r\\n            print(loss_func(output, batch["y"]), output[1][0].shape)\\r\\n        print(time.time() - start)\\r\\n\'}\nOFFICIAL_LICENSE = \'                    GNU GENERAL PUBLIC LICENSE\\n                       Version 3, 29 June 2007\\n\\n Copyright (C) 2007 Free Software Foundation, Inc. <https://fsf.org/>\\n Everyone is permitted to copy and distribute verbatim copies\\n of this license document, but changing it is not allowed.\\n\\n                            Preamble\\n\\n  The GNU General Public License is a free, copyleft license for\\nsoftware and other kinds of works.\\n\\n  The licenses for most software and other practical works are designed\\nto take away your freedom to share and change the works.  By contrast,\\nthe GNU General Public License is intended to guarantee your freedom to\\nshare and change all versions of a program--to make sure it remains free\\nsoftware for all its users.  We, the Free Software Foundation, use the\\nGNU General Public License for most of our software; it applies also to\\nany other work released this way by its authors.  You can apply it to\\nyour programs, too.\\n\\n  When we speak of free software, we are referring to freedom, not\\nprice.  Our General Public Licenses are designed to make sure that you\\nhave the freedom to distribute copies of free software (and charge for\\nthem if you wish), that you receive source code or can get it if you\\nwant it, that you can change the software or use pieces of it in new\\nfree programs, and that you know you can do these things.\\n\\n  To protect your rights, we need to prevent others from denying you\\nthese rights or asking you to surrender the rights.  Therefore, you have\\ncertain responsibilities if you distribute copies of the software, or if\\nyou modify it: responsibilities to respect the freedom of others.\\n\\n  For example, if you distribute copies of such a program, whether\\ngratis or for a fee, you must pass on to the recipients the same\\nfreedoms that you received.  You must make sure that they, too, receive\\nor can get the source code.  And you must show them these terms so they\\nknow their rights.\\n\\n  Developers that use the GNU GPL protect your rights with two steps:\\n(1) assert copyright on the software, and (2) offer you this License\\ngiving you legal permission to copy, distribute and/or modify it.\\n\\n  For the developers\\\' and authors\\\' protection, the GPL clearly explains\\nthat there is no warranty for this free software.  For both users\\\' and\\nauthors\\\' sake, the GPL requires that modified versions be marked as\\nchanged, so that their problems will not be attributed erroneously to\\nauthors of previous versions.\\n\\n  Some devices are designed to deny users access to install or run\\nmodified versions of the software inside them, although the manufacturer\\ncan do so.  This is fundamentally incompatible with the aim of\\nprotecting users\\\' freedom to change the software.  The systematic\\npattern of such abuse occurs in the area of products for individuals to\\nuse, which is precisely where it is most unacceptable.  Therefore, we\\nhave designed this version of the GPL to prohibit the practice for those\\nproducts.  If such problems arise substantially in other domains, we\\nstand ready to extend this provision to those domains in future versions\\nof the GPL, as needed to protect the freedom of users.\\n\\n  Finally, every program is threatened constantly by software patents.\\nStates should not allow patents to restrict development and use of\\nsoftware on general-purpose computers, but in those that do, we wish to\\navoid the special danger that patents applied to a free program could\\nmake it effectively proprietary.  To prevent this, the GPL assures that\\npatents cannot be used to render the program non-free.\\n\\n  The precise terms and conditions for copying, distribution and\\nmodification follow.\\n\\n                       TERMS AND CONDITIONS\\n\\n  0. Definitions.\\n\\n  "This License" refers to version 3 of the GNU General Public License.\\n\\n  "Copyright" also means copyright-like laws that apply to other kinds of\\nworks, such as semiconductor masks.\\n\\n  "The Program" refers to any copyrightable work licensed under this\\nLicense.  Each licensee is addressed as "you".  "Licensees" and\\n"recipients" may be individuals or organizations.\\n\\n  To "modify" a work means to copy from or adapt all or part of the work\\nin a fashion requiring copyright permission, other than the making of an\\nexact copy.  The resulting work is called a "modified version" of the\\nearlier work or a work "based on" the earlier work.\\n\\n  A "covered work" means either the unmodified Program or a work based\\non the Program.\\n\\n  To "propagate" a work means to do anything with it that, without\\npermission, would make you directly or secondarily liable for\\ninfringement under applicable copyright law, except executing it on a\\ncomputer or modifying a private copy.  Propagation includes copying,\\ndistribution (with or without modification), making available to the\\npublic, and in some countries other activities as well.\\n\\n  To "convey" a work means any kind of propagation that enables other\\nparties to make or receive copies.  Mere interaction with a user through\\na computer network, with no transfer of a copy, is not conveying.\\n\\n  An interactive user interface displays "Appropriate Legal Notices"\\nto the extent that it includes a convenient and prominently visible\\nfeature that (1) displays an appropriate copyright notice, and (2)\\ntells the user that there is no warranty for the work (except to the\\nextent that warranties are provided), that licensees may convey the\\nwork under this License, and how to view a copy of this License.  If\\nthe interface presents a list of user commands or options, such as a\\nmenu, a prominent item in the list meets this criterion.\\n\\n  1. Source Code.\\n\\n  The "source code" for a work means the preferred form of the work\\nfor making modifications to it.  "Object code" means any non-source\\nform of a work.\\n\\n  A "Standard Interface" means an interface that either is an official\\nstandard defined by a recognized standards body, or, in the case of\\ninterfaces specified for a particular programming language, one that\\nis widely used among developers working in that language.\\n\\n  The "System Libraries" of an executable work include anything, other\\nthan the work as a whole, that (a) is included in the normal form of\\npackaging a Major Component, but which is not part of that Major\\nComponent, and (b) serves only to enable use of the work with that\\nMajor Component, or to implement a Standard Interface for which an\\nimplementation is available to the public in source code form.  A\\n"Major Component", in this context, means a major essential component\\n(kernel, window system, and so on) of the specific operating system\\n(if any) on which the executable work runs, or a compiler used to\\nproduce the work, or an object code interpreter used to run it.\\n\\n  The "Corresponding Source" for a work in object code form means all\\nthe source code needed to generate, install, and (for an executable\\nwork) run the object code and to modify the work, including scripts to\\ncontrol those activities.  However, it does not include the work\\\'s\\nSystem Libraries, or general-purpose tools or generally available free\\nprograms which are used unmodified in performing those activities but\\nwhich are not part of the work.  For example, Corresponding Source\\nincludes interface definition files associated with source files for\\nthe work, and the source code for shared libraries and dynamically\\nlinked subprograms that the work is specifically designed to require,\\nsuch as by intimate data communication or control flow between those\\nsubprograms and other parts of the work.\\n\\n  The Corresponding Source need not include anything that users\\ncan regenerate automatically from other parts of the Corresponding\\nSource.\\n\\n  The Corresponding Source for a work in source code form is that\\nsame work.\\n\\n  2. Basic Permissions.\\n\\n  All rights granted under this License are granted for the term of\\ncopyright on the Program, and are irrevocable provided the stated\\nconditions are met.  This License explicitly affirms your unlimited\\npermission to run the unmodified Program.  The output from running a\\ncovered work is covered by this License only if the output, given its\\ncontent, constitutes a covered work.  This License acknowledges your\\nrights of fair use or other equivalent, as provided by copyright law.\\n\\n  You may make, run and propagate covered works that you do not\\nconvey, without conditions so long as your license otherwise remains\\nin force.  You may convey covered works to others for the sole purpose\\nof having them make modifications exclusively for you, or provide you\\nwith facilities for running those works, provided that you comply with\\nthe terms of this License in conveying all material for which you do\\nnot control copyright.  Those thus making or running the covered works\\nfor you must do so exclusively on your behalf, under your direction\\nand control, on terms that prohibit them from making any copies of\\nyour copyrighted material outside their relationship with you.\\n\\n  Conveying under any other circumstances is permitted solely under\\nthe conditions stated below.  Sublicensing is not allowed; section 10\\nmakes it unnecessary.\\n\\n  3. Protecting Users\\\' Legal Rights From Anti-Circumvention Law.\\n\\n  No covered work shall be deemed part of an effective technological\\nmeasure under any applicable law fulfilling obligations under article\\n11 of the WIPO copyright treaty adopted on 20 December 1996, or\\nsimilar laws prohibiting or restricting circumvention of such\\nmeasures.\\n\\n  When you convey a covered work, you waive any legal power to forbid\\ncircumvention of technological measures to the extent such circumvention\\nis effected by exercising rights under this License with respect to\\nthe covered work, and you disclaim any intention to limit operation or\\nmodification of the work as a means of enforcing, against the work\\\'s\\nusers, your or third parties\\\' legal rights to forbid circumvention of\\ntechnological measures.\\n\\n  4. Conveying Verbatim Copies.\\n\\n  You may convey verbatim copies of the Program\\\'s source code as you\\nreceive it, in any medium, provided that you conspicuously and\\nappropriately publish on each copy an appropriate copyright notice;\\nkeep intact all notices stating that this License and any\\nnon-permissive terms added in accord with section 7 apply to the code;\\nkeep intact all notices of the absence of any warranty; and give all\\nrecipients a copy of this License along with the Program.\\n\\n  You may charge any price or no price for each copy that you convey,\\nand you may offer support or warranty protection for a fee.\\n\\n  5. Conveying Modified Source Versions.\\n\\n  You may convey a work based on the Program, or the modifications to\\nproduce it from the Program, in the form of source code under the\\nterms of section 4, provided that you also meet all of these conditions:\\n\\n    a) The work must carry prominent notices stating that you modified\\n    it, and giving a relevant date.\\n\\n    b) The work must carry prominent notices stating that it is\\n    released under this License and any conditions added under section\\n    7.  This requirement modifies the requirement in section 4 to\\n    "keep intact all notices".\\n\\n    c) You must license the entire work, as a whole, under this\\n    License to anyone who comes into possession of a copy.  This\\n    License will therefore apply, along with any applicable section 7\\n    additional terms, to the whole of the work, and all its parts,\\n    regardless of how they are packaged.  This License gives no\\n    permission to license the work in any other way, but it does not\\n    invalidate such permission if you have separately received it.\\n\\n    d) If the work has interactive user interfaces, each must display\\n    Appropriate Legal Notices; however, if the Program has interactive\\n    interfaces that do not display Appropriate Legal Notices, your\\n    work need not make them do so.\\n\\n  A compilation of a covered work with other separate and independent\\nworks, which are not by their nature extensions of the covered work,\\nand which are not combined with it such as to form a larger program,\\nin or on a volume of a storage or distribution medium, is called an\\n"aggregate" if the compilation and its resulting copyright are not\\nused to limit the access or legal rights of the compilation\\\'s users\\nbeyond what the individual works permit.  Inclusion of a covered work\\nin an aggregate does not cause this License to apply to the other\\nparts of the aggregate.\\n\\n  6. Conveying Non-Source Forms.\\n\\n  You may convey a covered work in object code form under the terms\\nof sections 4 and 5, provided that you also convey the\\nmachine-readable Corresponding Source under the terms of this License,\\nin one of these ways:\\n\\n    a) Convey the object code in, or embodied in, a physical product\\n    (including a physical distribution medium), accompanied by the\\n    Corresponding Source fixed on a durable physical medium\\n    customarily used for software interchange.\\n\\n    b) Convey the object code in, or embodied in, a physical product\\n    (including a physical distribution medium), accompanied by a\\n    written offer, valid for at least three years and valid for as\\n    long as you offer spare parts or customer support for that product\\n    model, to give anyone who possesses the object code either (1) a\\n    copy of the Corresponding Source for all the software in the\\n    product that is covered by this License, on a durable physical\\n    medium customarily used for software interchange, for a price no\\n    more than your reasonable cost of physically performing this\\n    conveying of source, or (2) access to copy the\\n    Corresponding Source from a network server at no charge.\\n\\n    c) Convey individual copies of the object code with a copy of the\\n    written offer to provide the Corresponding Source.  This\\n    alternative is allowed only occasionally and noncommercially, and\\n    only if you received the object code with such an offer, in accord\\n    with subsection 6b.\\n\\n    d) Convey the object code by offering access from a designated\\n    place (gratis or for a charge), and offer equivalent access to the\\n    Corresponding Source in the same way through the same place at no\\n    further charge.  You need not require recipients to copy the\\n    Corresponding Source along with the object code.  If the place to\\n    copy the object code is a network server, the Corresponding Source\\n    may be on a different server (operated by you or a third party)\\n    that supports equivalent copying facilities, provided you maintain\\n    clear directions next to the object code saying where to find the\\n    Corresponding Source.  Regardless of what server hosts the\\n    Corresponding Source, you remain obligated to ensure that it is\\n    available for as long as needed to satisfy these requirements.\\n\\n    e) Convey the object code using peer-to-peer transmission, provided\\n    you inform other peers where the object code and Corresponding\\n    Source of the work are being offered to the general public at no\\n    charge under subsection 6d.\\n\\n  A separable portion of the object code, whose source code is excluded\\nfrom the Corresponding Source as a System Library, need not be\\nincluded in conveying the object code work.\\n\\n  A "User Product" is either (1) a "consumer product", which means any\\ntangible personal property which is normally used for personal, family,\\nor household purposes, or (2) anything designed or sold for incorporation\\ninto a dwelling.  In determining whether a product is a consumer product,\\ndoubtful cases shall be resolved in favor of coverage.  For a particular\\nproduct received by a particular user, "normally used" refers to a\\ntypical or common use of that class of product, regardless of the status\\nof the particular user or of the way in which the particular user\\nactually uses, or expects or is expected to use, the product.  A product\\nis a consumer product regardless of whether the product has substantial\\ncommercial, industrial or non-consumer uses, unless such uses represent\\nthe only significant mode of use of the product.\\n\\n  "Installation Information" for a User Product means any methods,\\nprocedures, authorization keys, or other information required to install\\nand execute modified versions of a covered work in that User Product from\\na modified version of its Corresponding Source.  The information must\\nsuffice to ensure that the continued functioning of the modified object\\ncode is in no case prevented or interfered with solely because\\nmodification has been made.\\n\\n  If you convey an object code work under this section in, or with, or\\nspecifically for use in, a User Product, and the conveying occurs as\\npart of a transaction in which the right of possession and use of the\\nUser Product is transferred to the recipient in perpetuity or for a\\nfixed term (regardless of how the transaction is characterized), the\\nCorresponding Source conveyed under this section must be accompanied\\nby the Installation Information.  But this requirement does not apply\\nif neither you nor any third party retains the ability to install\\nmodified object code on the User Product (for example, the work has\\nbeen installed in ROM).\\n\\n  The requirement to provide Installation Information does not include a\\nrequirement to continue to provide support service, warranty, or updates\\nfor a work that has been modified or installed by the recipient, or for\\nthe User Product in which it has been modified or installed.  Access to a\\nnetwork may be denied when the modification itself materially and\\nadversely affects the operation of the network or violates the rules and\\nprotocols for communication across the network.\\n\\n  Corresponding Source conveyed, and Installation Information provided,\\nin accord with this section must be in a format that is publicly\\ndocumented (and with an implementation available to the public in\\nsource code form), and must require no special password or key for\\nunpacking, reading or copying.\\n\\n  7. Additional Terms.\\n\\n  "Additional permissions" are terms that supplement the terms of this\\nLicense by making exceptions from one or more of its conditions.\\nAdditional permissions that are applicable to the entire Program shall\\nbe treated as though they were included in this License, to the extent\\nthat they are valid under applicable law.  If additional permissions\\napply only to part of the Program, that part may be used separately\\nunder those permissions, but the entire Program remains governed by\\nthis License without regard to the additional permissions.\\n\\n  When you convey a copy of a covered work, you may at your option\\nremove any additional permissions from that copy, or from any part of\\nit.  (Additional permissions may be written to require their own\\nremoval in certain cases when you modify the work.)  You may place\\nadditional permissions on material, added by you to a covered work,\\nfor which you have or can give appropriate copyright permission.\\n\\n  Notwithstanding any other provision of this License, for material you\\nadd to a covered work, you may (if authorized by the copyright holders of\\nthat material) supplement the terms of this License with terms:\\n\\n    a) Disclaiming warranty or limiting liability differently from the\\n    terms of sections 15 and 16 of this License; or\\n\\n    b) Requiring preservation of specified reasonable legal notices or\\n    author attributions in that material or in the Appropriate Legal\\n    Notices displayed by works containing it; or\\n\\n    c) Prohibiting misrepresentation of the origin of that material, or\\n    requiring that modified versions of such material be marked in\\n    reasonable ways as different from the original version; or\\n\\n    d) Limiting the use for publicity purposes of names of licensors or\\n    authors of the material; or\\n\\n    e) Declining to grant rights under trademark law for use of some\\n    trade names, trademarks, or service marks; or\\n\\n    f) Requiring indemnification of licensors and authors of that\\n    material by anyone who conveys the material (or modified versions of\\n    it) with contractual assumptions of liability to the recipient, for\\n    any liability that these contractual assumptions directly impose on\\n    those licensors and authors.\\n\\n  All other non-permissive additional terms are considered "further\\nrestrictions" within the meaning of section 10.  If the Program as you\\nreceived it, or any part of it, contains a notice stating that it is\\ngoverned by this License along with a term that is a further\\nrestriction, you may remove that term.  If a license document contains\\na further restriction but permits relicensing or conveying under this\\nLicense, you may add to a covered work material governed by the terms\\nof that license document, provided that the further restriction does\\nnot survive such relicensing or conveying.\\n\\n  If you add terms to a covered work in accord with this section, you\\nmust place, in the relevant source files, a statement of the\\nadditional terms that apply to those files, or a notice indicating\\nwhere to find the applicable terms.\\n\\n  Additional terms, permissive or non-permissive, may be stated in the\\nform of a separately written license, or stated as exceptions;\\nthe above requirements apply either way.\\n\\n  8. Termination.\\n\\n  You may not propagate or modify a covered work except as expressly\\nprovided under this License.  Any attempt otherwise to propagate or\\nmodify it is void, and will automatically terminate your rights under\\nthis License (including any patent licenses granted under the third\\nparagraph of section 11).\\n\\n  However, if you cease all violation of this License, then your\\nlicense from a particular copyright holder is reinstated (a)\\nprovisionally, unless and until the copyright holder explicitly and\\nfinally terminates your license, and (b) permanently, if the copyright\\nholder fails to notify you of the violation by some reasonable means\\nprior to 60 days after the cessation.\\n\\n  Moreover, your license from a particular copyright holder is\\nreinstated permanently if the copyright holder notifies you of the\\nviolation by some reasonable means, this is the first time you have\\nreceived notice of violation of this License (for any work) from that\\ncopyright holder, and you cure the violation prior to 30 days after\\nyour receipt of the notice.\\n\\n  Termination of your rights under this section does not terminate the\\nlicenses of parties who have received copies or rights from you under\\nthis License.  If your rights have been terminated and not permanently\\nreinstated, you do not qualify to receive new licenses for the same\\nmaterial under section 10.\\n\\n  9. Acceptance Not Required for Having Copies.\\n\\n  You are not required to accept this License in order to receive or\\nrun a copy of the Program.  Ancillary propagation of a covered work\\noccurring solely as a consequence of using peer-to-peer transmission\\nto receive a copy likewise does not require acceptance.  However,\\nnothing other than this License grants you permission to propagate or\\nmodify any covered work.  These actions infringe copyright if you do\\nnot accept this License.  Therefore, by modifying or propagating a\\ncovered work, you indicate your acceptance of this License to do so.\\n\\n  10. Automatic Licensing of Downstream Recipients.\\n\\n  Each time you convey a covered work, the recipient automatically\\nreceives a license from the original licensors, to run, modify and\\npropagate that work, subject to this License.  You are not responsible\\nfor enforcing compliance by third parties with this License.\\n\\n  An "entity transaction" is a transaction transferring control of an\\norganization, or substantially all assets of one, or subdividing an\\norganization, or merging organizations.  If propagation of a covered\\nwork results from an entity transaction, each party to that\\ntransaction who receives a copy of the work also receives whatever\\nlicenses to the work the party\\\'s predecessor in interest had or could\\ngive under the previous paragraph, plus a right to possession of the\\nCorresponding Source of the work from the predecessor in interest, if\\nthe predecessor has it or can get it with reasonable efforts.\\n\\n  You may not impose any further restrictions on the exercise of the\\nrights granted or affirmed under this License.  For example, you may\\nnot impose a license fee, royalty, or other charge for exercise of\\nrights granted under this License, and you may not initiate litigation\\n(including a cross-claim or counterclaim in a lawsuit) alleging that\\nany patent claim is infringed by making, using, selling, offering for\\nsale, or importing the Program or any portion of it.\\n\\n  11. Patents.\\n\\n  A "contributor" is a copyright holder who authorizes use under this\\nLicense of the Program or a work on which the Program is based.  The\\nwork thus licensed is called the contributor\\\'s "contributor version".\\n\\n  A contributor\\\'s "essential patent claims" are all patent claims\\nowned or controlled by the contributor, whether already acquired or\\nhereafter acquired, that would be infringed by some manner, permitted\\nby this License, of making, using, or selling its contributor version,\\nbut do not include claims that would be infringed only as a\\nconsequence of further modification of the contributor version.  For\\npurposes of this definition, "control" includes the right to grant\\npatent sublicenses in a manner consistent with the requirements of\\nthis License.\\n\\n  Each contributor grants you a non-exclusive, worldwide, royalty-free\\npatent license under the contributor\\\'s essential patent claims, to\\nmake, use, sell, offer for sale, import and otherwise run, modify and\\npropagate the contents of its contributor version.\\n\\n  In the following three paragraphs, a "patent license" is any express\\nagreement or commitment, however denominated, not to enforce a patent\\n(such as an express permission to practice a patent or covenant not to\\nsue for patent infringement).  To "grant" such a patent license to a\\nparty means to make such an agreement or commitment not to enforce a\\npatent against the party.\\n\\n  If you convey a covered work, knowingly relying on a patent license,\\nand the Corresponding Source of the work is not available for anyone\\nto copy, free of charge and under the terms of this License, through a\\npublicly available network server or other readily accessible means,\\nthen you must either (1) cause the Corresponding Source to be so\\navailable, or (2) arrange to deprive yourself of the benefit of the\\npatent license for this particular work, or (3) arrange, in a manner\\nconsistent with the requirements of this License, to extend the patent\\nlicense to downstream recipients.  "Knowingly relying" means you have\\nactual knowledge that, but for the patent license, your conveying the\\ncovered work in a country, or your recipient\\\'s use of the covered work\\nin a country, would infringe one or more identifiable patents in that\\ncountry that you have reason to believe are valid.\\n\\n  If, pursuant to or in connection with a single transaction or\\narrangement, you convey, or propagate by procuring conveyance of, a\\ncovered work, and grant a patent license to some of the parties\\nreceiving the covered work authorizing them to use, propagate, modify\\nor convey a specific copy of the covered work, then the patent license\\nyou grant is automatically extended to all recipients of the covered\\nwork and works based on it.\\n\\n  A patent license is "discriminatory" if it does not include within\\nthe scope of its coverage, prohibits the exercise of, or is\\nconditioned on the non-exercise of one or more of the rights that are\\nspecifically granted under this License.  You may not convey a covered\\nwork if you are a party to an arrangement with a third party that is\\nin the business of distributing software, under which you make payment\\nto the third party based on the extent of your activity of conveying\\nthe work, and under which the third party grants, to any of the\\nparties who would receive the covered work from you, a discriminatory\\npatent license (a) in connection with copies of the covered work\\nconveyed by you (or copies made from those copies), or (b) primarily\\nfor and in connection with specific products or compilations that\\ncontain the covered work, unless you entered into that arrangement,\\nor that patent license was granted, prior to 28 March 2007.\\n\\n  Nothing in this License shall be construed as excluding or limiting\\nany implied license or other defenses to infringement that may\\notherwise be available to you under applicable patent law.\\n\\n  12. No Surrender of Others\\\' Freedom.\\n\\n  If conditions are imposed on you (whether by court order, agreement or\\notherwise) that contradict the conditions of this License, they do not\\nexcuse you from the conditions of this License.  If you cannot convey a\\ncovered work so as to satisfy simultaneously your obligations under this\\nLicense and any other pertinent obligations, then as a consequence you may\\nnot convey it at all.  For example, if you agree to terms that obligate you\\nto collect a royalty for further conveying from those to whom you convey\\nthe Program, the only way you could satisfy both those terms and this\\nLicense would be to refrain entirely from conveying the Program.\\n\\n  13. Use with the GNU Affero General Public License.\\n\\n  Notwithstanding any other provision of this License, you have\\npermission to link or combine any covered work with a work licensed\\nunder version 3 of the GNU Affero General Public License into a single\\ncombined work, and to convey the resulting work.  The terms of this\\nLicense will continue to apply to the part which is the covered work,\\nbut the special requirements of the GNU Affero General Public License,\\nsection 13, concerning interaction through a network will apply to the\\ncombination as such.\\n\\n  14. Revised Versions of this License.\\n\\n  The Free Software Foundation may publish revised and/or new versions of\\nthe GNU General Public License from time to time.  Such new versions will\\nbe similar in spirit to the present version, but may differ in detail to\\naddress new problems or concerns.\\n\\n  Each version is given a distinguishing version number.  If the\\nProgram specifies that a certain numbered version of the GNU General\\nPublic License "or any later version" applies to it, you have the\\noption of following the terms and conditions either of that numbered\\nversion or of any later version published by the Free Software\\nFoundation.  If the Program does not specify a version number of the\\nGNU General Public License, you may choose any version ever published\\nby the Free Software Foundation.\\n\\n  If the Program specifies that a proxy can decide which future\\nversions of the GNU General Public License can be used, that proxy\\\'s\\npublic statement of acceptance of a version permanently authorizes you\\nto choose that version for the Program.\\n\\n  Later license versions may give you additional or different\\npermissions.  However, no additional obligations are imposed on any\\nauthor or copyright holder as a result of your choosing to follow a\\nlater version.\\n\\n  15. Disclaimer of Warranty.\\n\\n  THERE IS NO WARRANTY FOR THE PROGRAM, TO THE EXTENT PERMITTED BY\\nAPPLICABLE LAW.  EXCEPT WHEN OTHERWISE STATED IN WRITING THE COPYRIGHT\\nHOLDERS AND/OR OTHER PARTIES PROVIDE THE PROGRAM "AS IS" WITHOUT WARRANTY\\nOF ANY KIND, EITHER EXPRESSED OR IMPLIED, INCLUDING, BUT NOT LIMITED TO,\\nTHE IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR\\nPURPOSE.  THE ENTIRE RISK AS TO THE QUALITY AND PERFORMANCE OF THE PROGRAM\\nIS WITH YOU.  SHOULD THE PROGRAM PROVE DEFECTIVE, YOU ASSUME THE COST OF\\nALL NECESSARY SERVICING, REPAIR OR CORRECTION.\\n\\n  16. Limitation of Liability.\\n\\n  IN NO EVENT UNLESS REQUIRED BY APPLICABLE LAW OR AGREED TO IN WRITING\\nWILL ANY COPYRIGHT HOLDER, OR ANY OTHER PARTY WHO MODIFIES AND/OR CONVEYS\\nTHE PROGRAM AS PERMITTED ABOVE, BE LIABLE TO YOU FOR DAMAGES, INCLUDING ANY\\nGENERAL, SPECIAL, INCIDENTAL OR CONSEQUENTIAL DAMAGES ARISING OUT OF THE\\nUSE OR INABILITY TO USE THE PROGRAM (INCLUDING BUT NOT LIMITED TO LOSS OF\\nDATA OR DATA BEING RENDERED INACCURATE OR LOSSES SUSTAINED BY YOU OR THIRD\\nPARTIES OR A FAILURE OF THE PROGRAM TO OPERATE WITH ANY OTHER PROGRAMS),\\nEVEN IF SUCH HOLDER OR OTHER PARTY HAS BEEN ADVISED OF THE POSSIBILITY OF\\nSUCH DAMAGES.\\n\\n  17. Interpretation of Sections 15 and 16.\\n\\n  If the disclaimer of warranty and limitation of liability provided\\nabove cannot be given local legal effect according to their terms,\\nreviewing courts shall apply local law that most closely approximates\\nan absolute waiver of all civil liability in connection with the\\nProgram, unless a warranty or assumption of liability accompanies a\\ncopy of the Program in return for a fee.\\n\\n                     END OF TERMS AND CONDITIONS\\n\\n            How to Apply These Terms to Your New Programs\\n\\n  If you develop a new program, and you want it to be of the greatest\\npossible use to the public, the best way to achieve this is to make it\\nfree software which everyone can redistribute and change under these terms.\\n\\n  To do so, attach the following notices to the program.  It is safest\\nto attach them to the start of each source file to most effectively\\nstate the exclusion of warranty; and each file should have at least\\nthe "copyright" line and a pointer to where the full notice is found.\\n\\n    <one line to give the program\\\'s name and a brief idea of what it does.>\\n    Copyright (C) <year>  <name of author>\\n\\n    This program is free software: you can redistribute it and/or modify\\n    it under the terms of the GNU General Public License as published by\\n    the Free Software Foundation, either version 3 of the License, or\\n    (at your option) any later version.\\n\\n    This program is distributed in the hope that it will be useful,\\n    but WITHOUT ANY WARRANTY; without even the implied warranty of\\n    MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the\\n    GNU General Public License for more details.\\n\\n    You should have received a copy of the GNU General Public License\\n    along with this program.  If not, see <https://www.gnu.org/licenses/>.\\n\\nAlso add information on how to contact you by electronic and paper mail.\\n\\n  If the program does terminal interaction, make it output a short\\nnotice like this when it starts in an interactive mode:\\n\\n    <program>  Copyright (C) <year>  <name of author>\\n    This program comes with ABSOLUTELY NO WARRANTY; for details type `show w\\\'.\\n    This is free software, and you are welcome to redistribute it\\n    under certain conditions; type `show c\\\' for details.\\n\\nThe hypothetical commands `show w\\\' and `show c\\\' should show the appropriate\\nparts of the General Public License.  Of course, your program\\\'s commands\\nmight be different; for a GUI interface, you would use an "about box".\\n\\n  You should also get your employer (if you work as a programmer) or school,\\nif any, to sign a "copyright disclaimer" for the program, if necessary.\\nFor more information on this, and how to apply and follow the GNU GPL, see\\n<https://www.gnu.org/licenses/>.\\n\\n  The GNU General Public License does not permit incorporating your program\\ninto proprietary programs.  If your program is a subroutine library, you\\nmay consider it more useful to permit linking proprietary applications with\\nthe library.  If this is what you want to do, use the GNU Lesser General\\nPublic License instead of this License.  But first, please read\\n<https://www.gnu.org/licenses/why-not-lgpl.html>.\'\n\ndef write_json(path, content):\n    with Path(path).open(\'x\', encoding=\'utf-8\') as handle:\n        json.dump(content, handle, indent=2, ensure_ascii=False, allow_nan=False)\n\ndef load_split(path, dataset, data_dir):\n    split = json.loads(Path(path).read_text())\n    keys = (\'train_indices\', \'val_indices\', \'test_indices\')\n    for key in keys:\n        if not isinstance(split.get(key), list) or not split[key] or any((type(i) is not int for i in split[key])):\n            raise ValueError(f\'{key}: expected a nonempty integer list\')\n    indices = [i for key in keys for i in split[key]]\n    if len(indices) != len(dataset) or set(indices) != set(range(len(dataset))):\n        raise ValueError(\'Split is not disjoint, in range and complete\')\n    if Path(split[\'processed_dir\']).resolve() != Path(data_dir).resolve():\n        raise ValueError(\'Split processed_dir mismatch\')\n    for key, count_key in zip(keys, (\'n_train\', \'n_validation\', \'n_test\')):\n        if count_key in split and split[count_key] != len(split[key]):\n            raise ValueError(f\'{count_key} mismatch\')\n    audit = pd.read_csv(Path(path).with_name(Path(path).stem + \'_samples.csv\'))\n    if audit[\'sample\'].tolist() != list(dataset.samples):\n        raise ValueError(\'Sample order differs from verified audit\')\n    counts = np.asarray(dataset.meta_counts, dtype=np.int64)\n    ends = np.cumsum(counts)\n    starts = ends - counts\n    for key, expected in [(\'eligible_cells\', counts), (\'global_start\', starts), (\'global_end_exclusive\', ends)]:\n        if not np.array_equal(audit[key].to_numpy(), expected):\n            raise ValueError(f\'Sample audit {key} mismatch\')\n    membership = np.empty(len(dataset), dtype=np.int8)\n    for label, key in enumerate(keys):\n        membership[split[key]] = label\n    labels = (\'train\', \'validation\', \'test\')\n    for row, start, end in zip(audit.itertuples(index=False), starts, ends):\n        if row.split not in labels or not np.all(membership[start:end] == labels.index(row.split)):\n            raise ValueError(f\'{row.sample}: sample crosses partitions or audit mismatch\')\n    return split\n\ndef evaluate_arrays(prediction, target):\n    """Match the established SpaGAT PCC/MSE conventions; use paper EV_zero."""\n    if prediction.shape != target.shape or not np.isfinite(prediction).all() or (not np.isfinite(target).all()):\n        raise ValueError(\'Invalid prediction/target arrays\')\n    prediction = torch.as_tensor(prediction).float()\n    target = torch.as_tensor(target).float()\n    pc = prediction - prediction.mean(dim=0, keepdim=True)\n    tc = target - target.mean(dim=0, keepdim=True)\n    denominator = torch.sqrt((pc.square().sum(dim=0) * tc.square().sum(dim=0)).clamp_min(1e-12))\n    pcc = ((pc * tc).sum(dim=0) / denominator).clamp(-1, 1)\n    mse = torch.mean((prediction - target).square()).item()\n    mse0 = torch.mean(target.square()).item()\n    metrics = dict(median_gene_pcc=torch.median(pcc).item(), mean_gene_pcc=torch.mean(pcc).item(), mse=mse, mse_zero=mse0, ev_zero_percent=None if mse0 == 0 else 100 * (1 - mse / mse0), n_cells=int(target.shape[0]), n_genes=int(target.shape[1]))\n    return (metrics, pcc.numpy())\n\ndef load_official_model_class():\n    """Execute pinned source with only import paths renamed for isolation."""\n    import types\n    namespace = \'_spagat_revision_official_gitiii\'\n    if namespace + \'.model\' in sys.modules:\n        return sys.modules[namespace + \'.model\'].GITIII\n    package = types.ModuleType(namespace)\n    package.__path__ = []\n    sys.modules[namespace] = package\n    for filename in (\'preprocess.py\', \'embedding.py\', \'attention.py\', \'model.py\'):\n        source = OFFICIAL_SOURCES[filename]\n        if hashlib.sha256(source.encode()).hexdigest() != OFFICIAL_HASHES[filename]:\n            raise ValueError(\'Embedded official source hash mismatch: \' + filename)\n        adapted = source.replace(\'from gitiii.\', \'from \' + namespace + \'.\').replace(\'import gitiii.\', \'import \' + namespace + \'.\')\n        name = filename[:-3]\n        module = types.ModuleType(namespace + \'.\' + name)\n        module.__package__ = namespace\n        sys.modules[module.__name__] = module\n        setattr(package, name, module)\n        exec(compile(adapted, \'pinned_gitiii/\' + filename, \'exec\'), module.__dict__)\n    return sys.modules[namespace + \'.model\'].GITIII\n\ndef save_official_sources(out):\n    directory = Path(out) / \'official_source\'\n    directory.mkdir()\n    for filename, source in OFFICIAL_SOURCES.items():\n        (directory / filename).write_bytes(source.encode())\n    (directory / \'LICENSE\').write_text(OFFICIAL_LICENSE)\n    write_json(directory / \'provenance.json\', dict(repository=OFFICIAL_REPOSITORY, commit=OFFICIAL_COMMIT, files_sha256=OFFICIAL_HASHES, import_adaptation=\'Only gitiii import paths renamed to isolated module namespace at load time\'))\n\ndef prepare_batch(batch, device):\n    x, ct, y = (batch[\'x\'], batch[\'cell_types\'], batch[\'y\'])\n    if x.ndim != 3 or x.shape[1] != 50 or ct.shape != x.shape[:2]:\n        raise ValueError(\'Expected aligned 50-cell neighborhoods\')\n    if not torch.equal(y, x[:, 0, :]):\n        raise ValueError(\'Center residual differs from target\')\n    if batch[\'type_exp\'].shape != x.shape:\n        raise ValueError(\'Type baseline shape mismatch\')\n    if ct.min() < 0 or ct.max() >= 29:\n        raise ValueError(\'Cell type index exceeds the official fixed 29-entry embedding\')\n    if \'neighbor_mask\' in batch and (not bool(torch.all(batch[\'neighbor_mask\'] == 1))):\n        raise ValueError(\'Expected all 50 nodes valid\')\n    values = {k: batch[k] for k in (\'x\', \'type_exp\', \'cell_types\', \'position_x\', \'position_y\')}\n    if any((not torch.isfinite(v).all() for v in values.values())) or not torch.isfinite(y).all():\n        raise ValueError(\'Nonfinite input/target\')\n    values[\'x\'] = x.masked_fill((ct == ct[:, :1]).unsqueeze(-1), 0)\n    values = {k: v.to(device) for k, v in values.items()}\n    values[\'y\'] = torch.zeros_like(y, device=device)\n    return (values, y.to(device))\n\ndef predict(model, dataset, indices, device, batch_size, label):\n    model.eval()\n    predictions, targets = ([], [])\n    loader = DataLoader(Subset(dataset, indices), batch_size=batch_size, shuffle=False, num_workers=0)\n    with torch.no_grad():\n        for step, batch in enumerate(loader, 1):\n            inputs, y = prepare_batch(batch, device)\n            predictions.append(model(inputs)[0].cpu().numpy())\n            targets.append(y.cpu().numpy())\n            if step == 1 or step % 200 == 0 or step == len(loader):\n                print(f\'{label}: batch {step}/{len(loader)}\', flush=True)\n    return (np.concatenate(predictions), np.concatenate(targets))\n\ndef run_experiment(dataset, split, genes, out, model, device, max_epochs=10, patience=5, batch_size=32):\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=False)\n    import shutil\n    shutil.copyfile(__file__, out / \'runner_source.py\')\n    save_official_sources(out)\n    write_json(out / \'shared_split.json\', split)\n    config = dict(dataset=\'SEA_AD\', model=\'GITIII-official\', seed=SEED, max_epochs=max_epochs, patience=patience, batch_size=batch_size, learning_rate=0.0001, optimizer=\'AdamW\', weight_decay=0.01, betas=[0.99, 0.999], node_dim=256, edge_dim=48, num_heads=2, n_layers=1, att_dim=8, node_dim_small=16, official_commit=OFFICIAL_COMMIT, last_layer=\'GRIT_encoder_last_layer\', input=\'Official baseline+heterotypic residual, 29-entry type embedding, ligand and distance features; center/homotypic residual zeroed; raw saved coordinates retained\', selection=\'validation median_gene_pcc; strict improvement; first epoch wins ties\', epoch_numbering=\'zero-based\', monitored_metric=\'median_gene_pcc\', target_space=\'residual relative to training-derived cell-type baseline\', pcc=\'float32 centered; denominator product clamped at 1e-12; torch.median; constant columns score zero\', ev=\'100*(1-MSE/MSE0)\', parameter_count=sum((p.numel() for p in model.parameters())), device=str(device), gpu=torch.cuda.get_device_name() if device.type == \'cuda\' else None, versions=dict(python=sys.version, torch=torch.__version__, numpy=np.__version__, pandas=pd.__version__))\n    if \'torch_geometric\' in sys.modules:\n        config[\'versions\'][\'torch_geometric\'] = sys.modules[\'torch_geometric\'].__version__\n    write_json(out / \'run_config.json\', config)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001, betas=(0.99, 0.999))\n    train_loader = DataLoader(Subset(dataset, split[\'train_indices\']), batch_size=batch_size, shuffle=True, num_workers=0, generator=torch.Generator().manual_seed(SEED))\n    best_score, best_epoch, stale = (-float(\'inf\'), None, 0)\n    records = []\n    checkpoint = out / f\'gitiii_official_seed{SEED}_best.pth\'\n    started = time.perf_counter()\n    for epoch in range(max_epochs):\n        model.train()\n        total, count = (0.0, 0)\n        for step, batch in enumerate(train_loader, 1):\n            inputs, y = prepare_batch(batch, device)\n            optimizer.zero_grad(set_to_none=True)\n            prediction = model(inputs)[0]\n            loss = tnn.functional.mse_loss(prediction, y)\n            if not torch.isfinite(loss):\n                raise ValueError(\'Nonfinite training loss; test will not be evaluated\')\n            loss.backward()\n            optimizer.step()\n            total += loss.item() * len(y)\n            count += len(y)\n            if step == 1 or step % 200 == 0 or step == len(train_loader):\n                print(f\'Epoch {epoch}: train batch {step}/{len(train_loader)}, MSE {total / count:.6f}\', flush=True)\n        pred, target = predict(model, dataset, split[\'val_indices\'], device, batch_size, \'validation\')\n        metrics, _ = evaluate_arrays(pred, target)\n        score = metrics[\'median_gene_pcc\']\n        if not np.isfinite(score):\n            raise ValueError(\'Nonfinite validation PCC; test will not be evaluated\')\n        improved = score > best_score\n        if improved:\n            best_score, best_epoch, stale = (score, epoch, 0)\n            torch.save(dict(model_state_dict=model.state_dict(), best_epoch=epoch, best_validation_pcc=score), checkpoint)\n        else:\n            stale += 1\n        records.append(dict(epoch=epoch, train_mse=total / count, validation_median_pcc=score, validation_mse=metrics[\'mse\'], best_epoch=best_epoch, elapsed_seconds=time.perf_counter() - started))\n        pd.DataFrame(records).to_csv(out / \'training_history.csv\', index=False)\n        print(f\'Epoch {epoch} completed: validation median PCC={score:.6f}; best epoch={best_epoch}\', flush=True)\n        del pred, target\n        if stale >= patience:\n            print(\'Validation early stopping.\', flush=True)\n            break\n    if best_epoch is None:\n        raise RuntimeError(\'No valid checkpoint; test will not be evaluated\')\n    saved = torch.load(checkpoint, map_location=device, weights_only=True)\n    model.load_state_dict(saved[\'model_state_dict\'])\n    print(f"Loaded best checkpoint (epoch {saved[\'best_epoch\']}). Starting final test once.", flush=True)\n    pred, target = predict(model, dataset, split[\'test_indices\'], device, batch_size, \'test\')\n    metrics, pcc = evaluate_arrays(pred, target)\n    metrics.update(dataset=\'SEA_AD\', model=\'GITIII-official\', seed=SEED, best_epoch=saved[\'best_epoch\'], best_validation_pcc=saved[\'best_validation_pcc\'], checkpoint=checkpoint.name, runtime_seconds=time.perf_counter() - started, target_space=config[\'target_space\'])\n    np.savez_compressed(out / \'test_predictions.npz\', prediction=pred, target=target, test_indices=np.asarray(split[\'test_indices\'], dtype=np.int64), genes=np.asarray(genes, dtype=str))\n    pd.DataFrame({\'gene\': genes, \'PCC\': pcc}).to_csv(out / \'test_per_gene_PCC.csv\', index=False)\n    write_json(out / \'test_metrics.json\', metrics)\n    pd.DataFrame([metrics]).to_csv(out / \'test_metrics.csv\', index=False)\n    pd.DataFrame([dict(seed=SEED, best_epoch=best_epoch, best_validation_pcc=best_score, monitored_metric=\'median_gene_pcc\', parameter_count=config[\'parameter_count\'], max_epochs=max_epochs, patience=patience)]).to_csv(out / \'best_runs.csv\', index=False)\n    print(json.dumps(metrics, indent=2), flush=True)\n    print(\'Completed; saved to:\', out, flush=True)\n    return metrics\n\ndef main():\n    global SEED\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--revision-dir\', default=DEFAULT_REVISION)\n    parser.add_argument(\'--output-dir\', default=None)\n    parser.add_argument(\'--seed\', type=int, default=123)\n    args = parser.parse_args()\n    SEED = args.seed\n    revision = Path(args.revision_dir)\n    out = Path(args.output_dir) if args.output_dir else revision / \'baseline_results\' / f\'SEA_AD_GITIII_official_seed{SEED}\'\n    if out.exists():\n        raise FileExistsError(f\'Refusing to overwrite: {out}\')\n    random.seed(SEED)\n    np.random.seed(SEED)\n    torch.manual_seed(SEED)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(SEED)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    print(\'Device:\', device, flush=True)\n    official_class = load_official_model_class()\n    # The unmodified official module seeds at import: restore requested run seed.\n    random.seed(SEED)\n    np.random.seed(SEED)\n    torch.manual_seed(SEED)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(SEED)\n    print(\'Pinned GITIII commit:\', OFFICIAL_COMMIT, flush=True)\n    split_path = revision / \'splits\' / \'SEA_AD_inductive_split_indices.json\'\n    metadata = json.loads(split_path.read_text())\n    if tuple((len(metadata[k]) for k in (\'train_indices\', \'val_indices\', \'test_indices\'))) != (257152, 54026, 54762):\n        raise ValueError(\'Unexpected SEA-AD split counts\')\n    loader_path = revision / \'spagat\' / \'dataloader.py\'\n    manifest = json.loads((revision / \'revision_manifest.json\').read_text())\n    digest = hashlib.sha256(loader_path.read_bytes()).hexdigest()\n    if digest != manifest[\'files_sha256\'][\'spagat/dataloader.py\']:\n        raise ValueError(\'Frozen dataloader hash mismatch\')\n    spec = importlib.util.spec_from_file_location(\'frozen_spagat_data\', loader_path)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    print(\'Loading verified processed data...\', flush=True)\n    dataset = module.SPAGAT_dataset(processed_dir=metadata[\'processed_dir\'], num_neighbors=50)\n    split = load_split(split_path, dataset, metadata[\'processed_dir\'])\n    genes = list(dataset.genes)\n    if len(genes) != 140 or len(set(genes)) != 140:\n        raise ValueError(\'Expected 140 unique genes\')\n    if len(dataset.cell_types_dict) > 29:\n        raise ValueError(\'Too many cell types for unchanged official architecture\')\n    model = official_class(genes, dataset.interactions, node_dim=256, edge_dim=48, num_heads=2, n_layers=1, node_dim_small=16, att_dim=8, use_cell_type_embedding=True).to(device)\n    if type(model.last_layer).__name__ != \'GRIT_encoder_last_layer\' or tuple(model.last_layer.W_A.weight.shape) != (140, 48):\n        raise ValueError(\'Unexpected last-layer architecture\')\n    batch = next(iter(DataLoader(Subset(dataset, split[\'train_indices\'][:2]), batch_size=2)))\n    inputs, target = prepare_batch(batch, device)\n    model.eval()\n    with torch.no_grad():\n        prediction = model(inputs)[0]\n        if prediction.shape != target.shape or not torch.isfinite(prediction).all():\n            raise ValueError(\'Training-sample forward check failed\')\n    del inputs, target, prediction, batch\n    print(\'Official last-layer W_A shape:\', tuple(model.last_layer.W_A.weight.shape), flush=True)\n    print(f\'Split audit passed. Starting SEA-AD official GITIII, seed {SEED}, max 10 epochs, patience 5.\', flush=True)\n    run_experiment(dataset, split, genes, out, model, device)\n    write_json(out / \'source_provenance.json\', dict(revision=str(revision), split_sha256=hashlib.sha256(split_path.read_bytes()).hexdigest(), dataloader_sha256=digest, runner_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()))\nif __name__ == \'__main__\':\n    main()\n', 'runners/SPICE_adapted.py': '# SEA-AD adaptation of the archived Mouse revision runner.\n# Only dataset metadata, split counts, dimensions checked in main, and log labels change.\n# Network classes, features, losses, optimizers, selection and scoring are retained.\n# Historical class identifiers (MouseNoLog) do NOT perform normalization.\nSEED = 123\n"""Mouse SPICE-adapted revision: fixed splits, validation PCC selection, one final test."""\nimport argparse\nimport hashlib\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport random\nimport sys\nimport time\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as tnn\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader, Subset\nDEFAULT_REVISION = \'/content/drive/MyDrive/SpaGAT_revision/inductive_training/20260922_104353_552891_UTC_f291de00\'\n\ndef write_json(path, content):\n    with Path(path).open(\'x\', encoding=\'utf-8\') as handle:\n        json.dump(content, handle, indent=2, ensure_ascii=False, allow_nan=False)\n\ndef load_split(path, dataset, data_dir):\n    split = json.loads(Path(path).read_text())\n    keys = (\'train_indices\', \'val_indices\', \'test_indices\')\n    for key in keys:\n        if not isinstance(split.get(key), list) or not split[key] or any((type(i) is not int for i in split[key])):\n            raise ValueError(f\'{key}: expected a nonempty integer list\')\n    indices = [i for key in keys for i in split[key]]\n    if len(indices) != len(dataset) or set(indices) != set(range(len(dataset))):\n        raise ValueError(\'Split is not disjoint, in range and complete\')\n    if Path(split[\'processed_dir\']).resolve() != Path(data_dir).resolve():\n        raise ValueError(\'Split processed_dir mismatch\')\n    for key, count_key in zip(keys, (\'n_train\', \'n_validation\', \'n_test\')):\n        if count_key in split and split[count_key] != len(split[key]):\n            raise ValueError(f\'{count_key} mismatch\')\n    audit = pd.read_csv(Path(path).with_name(Path(path).stem + \'_samples.csv\'))\n    if audit[\'sample\'].tolist() != list(dataset.samples):\n        raise ValueError(\'Sample order differs from verified audit\')\n    counts = np.asarray(dataset.meta_counts, dtype=np.int64)\n    ends = np.cumsum(counts)\n    starts = ends - counts\n    for key, expected in [(\'eligible_cells\', counts), (\'global_start\', starts), (\'global_end_exclusive\', ends)]:\n        if not np.array_equal(audit[key].to_numpy(), expected):\n            raise ValueError(f\'Sample audit {key} mismatch\')\n    membership = np.empty(len(dataset), dtype=np.int8)\n    for label, key in enumerate(keys):\n        membership[split[key]] = label\n    labels = (\'train\', \'validation\', \'test\')\n    for row, start, end in zip(audit.itertuples(index=False), starts, ends):\n        if row.split not in labels or not np.all(membership[start:end] == labels.index(row.split)):\n            raise ValueError(f\'{row.sample}: sample crosses partitions or audit mismatch\')\n    return split\n\ndef evaluate_arrays(prediction, target):\n    """Match the established SpaGAT PCC/MSE conventions; use paper EV_zero."""\n    if prediction.shape != target.shape or not np.isfinite(prediction).all() or (not np.isfinite(target).all()):\n        raise ValueError(\'Invalid prediction/target arrays\')\n    prediction = torch.as_tensor(prediction).float()\n    target = torch.as_tensor(target).float()\n    pc = prediction - prediction.mean(dim=0, keepdim=True)\n    tc = target - target.mean(dim=0, keepdim=True)\n    denominator = torch.sqrt((pc.square().sum(dim=0) * tc.square().sum(dim=0)).clamp_min(1e-12))\n    pcc = ((pc * tc).sum(dim=0) / denominator).clamp(-1, 1)\n    mse = torch.mean((prediction - target).square()).item()\n    mse0 = torch.mean(target.square()).item()\n    metrics = dict(median_gene_pcc=torch.median(pcc).item(), mean_gene_pcc=torch.mean(pcc).item(), mse=mse, mse_zero=mse0, ev_zero_percent=None if mse0 == 0 else 100 * (1 - mse / mse0), n_cells=int(target.shape[0]), n_genes=int(target.shape[1]))\n    return (metrics, pcc.numpy())\n\nclass SpiceAdaptedMouseNoLog(nn.Module):\n\n    def __init__(self, gene_dim=254, num_celltypes=24, hidden_dim=128, type_dim=16):\n        super().__init__()\n        from torch_geometric.nn import GCNConv\n        self.expr_proj = nn.Sequential(nn.Linear(gene_dim, hidden_dim), nn.GELU())\n        self.type_emb = nn.Embedding(num_celltypes, type_dim)\n        node_dim = hidden_dim + type_dim\n        self.gcn = GCNConv(node_dim, hidden_dim, add_self_loops=True, normalize=True)\n        self.decoder = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, gene_dim))\n\n    def forward(self, x, celltypes):\n        B, K, G = x.shape\n        h_expr = self.expr_proj(x)\n        h_type = self.type_emb(celltypes)\n        h = torch.cat([h_expr, h_type], dim=-1)\n        h = h.reshape(B * K, -1)\n        src_all = []\n        dst_all = []\n        for b in range(B):\n            offset = b * K\n            center = offset\n            neighbors = offset + torch.arange(1, K, device=x.device, dtype=torch.long)\n            centers = torch.full((K - 1,), center, device=x.device, dtype=torch.long)\n            src_all.append(neighbors)\n            dst_all.append(centers)\n            src_all.append(centers)\n            dst_all.append(neighbors)\n        edge_index = torch.stack([torch.cat(src_all), torch.cat(dst_all)], dim=0)\n        h = self.gcn(h, edge_index)\n        h = torch.relu(h)\n        center_idx = torch.arange(B, device=x.device) * K\n        h_center = h[center_idx]\n        return self.decoder(h_center)\n\ndef prepare_batch(batch, device):\n    x, ct, y = (batch[\'x\'], batch[\'cell_types\'], batch[\'y\'])\n    if x.ndim != 3 or x.shape[1] != 50 or ct.shape != x.shape[:2]:\n        raise ValueError(\'Expected aligned 50-cell neighborhoods\')\n    if \'neighbor_mask\' in batch and (not bool(torch.all(batch[\'neighbor_mask\'] == 1))):\n        raise ValueError(\'Expected all 50 nodes valid\')\n    if not torch.equal(y, x[:, 0, :]):\n        raise ValueError(\'Center residual differs from target\')\n    if not torch.isfinite(x).all() or not torch.isfinite(y).all():\n        raise ValueError(\'Nonfinite data\')\n    x = x.masked_fill((ct == ct[:, :1]).unsqueeze(-1), 0)\n    return (x.to(device), ct.to(device), y.to(device))\n\ndef predict(model, dataset, indices, device, batch_size, label):\n    model.eval()\n    predictions, targets = ([], [])\n    loader = DataLoader(Subset(dataset, indices), batch_size=batch_size, shuffle=False, num_workers=0)\n    with torch.no_grad():\n        for step, batch in enumerate(loader, 1):\n            x, ct, y = prepare_batch(batch, device)\n            predictions.append(model(x, ct).cpu().numpy())\n            targets.append(y.cpu().numpy())\n            if step == 1 or step % 200 == 0 or step == len(loader):\n                print(f\'{label}: batch {step}/{len(loader)}\', flush=True)\n    return (np.concatenate(predictions), np.concatenate(targets))\n\ndef run_experiment(dataset, split, genes, out, model, device, max_epochs=10, patience=5, batch_size=32):\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=False)\n    import shutil\n    shutil.copyfile(__file__, out / \'runner_source.py\')\n    write_json(out / \'shared_split.json\', split)\n    config = dict(dataset=\'SEA_AD\', model=\'SPICE-adapted\', seed=SEED, max_epochs=max_epochs, patience=patience, batch_size=batch_size, learning_rate=0.0001, optimizer=\'AdamW\', weight_decay=0.01, betas=[0.9, 0.999], hidden_dim=128, type_dim=16, add_self_loops=True, normalize=True, edges=\'bidirectional center-neighbor star; GCN adds self-loops\', input=\'residual expression plus learned type embedding; center and homotypic residuals zeroed; no baseline or distance features added\', selection=\'validation median_gene_pcc; strict improvement; first epoch wins ties\', epoch_numbering=\'zero-based\', monitored_metric=\'median_gene_pcc\', target_space=\'residual relative to training-derived cell-type baseline\', pcc=\'float32 centered; denominator product clamped at 1e-12; torch.median; constant columns score zero\', ev=\'100*(1-MSE/MSE0)\', parameter_count=sum((p.numel() for p in model.parameters())), device=str(device), gpu=torch.cuda.get_device_name() if device.type == \'cuda\' else None, versions=dict(python=sys.version, torch=torch.__version__, numpy=np.__version__, pandas=pd.__version__))\n    if \'torch_geometric\' in sys.modules:\n        config[\'versions\'][\'torch_geometric\'] = sys.modules[\'torch_geometric\'].__version__\n    write_json(out / \'run_config.json\', config)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001)\n    train_loader = DataLoader(Subset(dataset, split[\'train_indices\']), batch_size=batch_size, shuffle=True, num_workers=0, generator=torch.Generator().manual_seed(SEED))\n    best_score, best_epoch, stale = (-float(\'inf\'), None, 0)\n    records = []\n    checkpoint = out / f\'spice_adapted_seed{SEED}_best.pth\'\n    started = time.perf_counter()\n    for epoch in range(max_epochs):\n        model.train()\n        total, count = (0.0, 0)\n        for step, batch in enumerate(train_loader, 1):\n            x, ct, y = prepare_batch(batch, device)\n            optimizer.zero_grad(set_to_none=True)\n            prediction = model(x, ct)\n            loss = tnn.functional.mse_loss(prediction, y)\n            if not torch.isfinite(loss):\n                raise ValueError(\'Nonfinite training loss; test will not be evaluated\')\n            loss.backward()\n            optimizer.step()\n            total += loss.item() * len(y)\n            count += len(y)\n            if step == 1 or step % 200 == 0 or step == len(train_loader):\n                print(f\'Epoch {epoch}: train batch {step}/{len(train_loader)}, MSE {total / count:.6f}\', flush=True)\n        pred, target = predict(model, dataset, split[\'val_indices\'], device, batch_size, \'validation\')\n        metrics, _ = evaluate_arrays(pred, target)\n        score = metrics[\'median_gene_pcc\']\n        if not np.isfinite(score):\n            raise ValueError(\'Nonfinite validation PCC; test will not be evaluated\')\n        improved = score > best_score\n        if improved:\n            best_score, best_epoch, stale = (score, epoch, 0)\n            torch.save(dict(model_state_dict=model.state_dict(), best_epoch=epoch, best_validation_pcc=score), checkpoint)\n        else:\n            stale += 1\n        records.append(dict(epoch=epoch, train_mse=total / count, validation_median_pcc=score, validation_mse=metrics[\'mse\'], best_epoch=best_epoch, elapsed_seconds=time.perf_counter() - started))\n        pd.DataFrame(records).to_csv(out / \'training_history.csv\', index=False)\n        print(f\'Epoch {epoch} completed: validation median PCC={score:.6f}; best epoch={best_epoch}\', flush=True)\n        del pred, target\n        if stale >= patience:\n            print(\'Validation early stopping.\', flush=True)\n            break\n    if best_epoch is None:\n        raise RuntimeError(\'No valid checkpoint; test will not be evaluated\')\n    saved = torch.load(checkpoint, map_location=device, weights_only=True)\n    model.load_state_dict(saved[\'model_state_dict\'])\n    print(f"Loaded best checkpoint (epoch {saved[\'best_epoch\']}). Starting final test once.", flush=True)\n    pred, target = predict(model, dataset, split[\'test_indices\'], device, batch_size, \'test\')\n    metrics, pcc = evaluate_arrays(pred, target)\n    metrics.update(dataset=\'SEA_AD\', model=\'SPICE-adapted\', seed=SEED, best_epoch=saved[\'best_epoch\'], best_validation_pcc=saved[\'best_validation_pcc\'], checkpoint=checkpoint.name, runtime_seconds=time.perf_counter() - started, target_space=config[\'target_space\'])\n    np.savez_compressed(out / \'test_predictions.npz\', prediction=pred, target=target, test_indices=np.asarray(split[\'test_indices\'], dtype=np.int64), genes=np.asarray(genes, dtype=str))\n    pd.DataFrame({\'gene\': genes, \'PCC\': pcc}).to_csv(out / \'test_per_gene_PCC.csv\', index=False)\n    write_json(out / \'test_metrics.json\', metrics)\n    pd.DataFrame([metrics]).to_csv(out / \'test_metrics.csv\', index=False)\n    pd.DataFrame([dict(seed=SEED, best_epoch=best_epoch, best_validation_pcc=best_score, monitored_metric=\'median_gene_pcc\', parameter_count=config[\'parameter_count\'], max_epochs=max_epochs, patience=patience)]).to_csv(out / \'best_runs.csv\', index=False)\n    print(json.dumps(metrics, indent=2), flush=True)\n    print(\'Completed; saved to:\', out, flush=True)\n    return metrics\n\ndef main():\n    global SEED\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--revision-dir\', default=DEFAULT_REVISION)\n    parser.add_argument(\'--output-dir\', default=None)\n    parser.add_argument(\'--seed\', type=int, default=123)\n    args = parser.parse_args()\n    SEED = args.seed\n    revision = Path(args.revision_dir)\n    out = Path(args.output_dir) if args.output_dir else revision / \'baseline_results\' / f\'SEA_AD_SPICE_adapted_seed{SEED}\'\n    if out.exists():\n        raise FileExistsError(f\'Refusing to overwrite: {out}\')\n    random.seed(SEED)\n    np.random.seed(SEED)\n    torch.manual_seed(SEED)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(SEED)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    print(\'Device:\', device, flush=True)\n    from torch_geometric.nn import GCNConv\n    split_path = revision / \'splits\' / \'SEA_AD_inductive_split_indices.json\'\n    metadata = json.loads(split_path.read_text())\n    if tuple((len(metadata[k]) for k in (\'train_indices\', \'val_indices\', \'test_indices\'))) != (257152, 54026, 54762):\n        raise ValueError(\'Unexpected SEA-AD split counts\')\n    loader_path = revision / \'spagat\' / \'dataloader.py\'\n    manifest = json.loads((revision / \'revision_manifest.json\').read_text())\n    digest = hashlib.sha256(loader_path.read_bytes()).hexdigest()\n    if digest != manifest[\'files_sha256\'][\'spagat/dataloader.py\']:\n        raise ValueError(\'Frozen dataloader hash mismatch\')\n    spec = importlib.util.spec_from_file_location(\'frozen_spagat_data\', loader_path)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    print(\'Loading verified processed data...\', flush=True)\n    dataset = module.SPAGAT_dataset(processed_dir=metadata[\'processed_dir\'], num_neighbors=50)\n    split = load_split(split_path, dataset, metadata[\'processed_dir\'])\n    genes = list(dataset.genes)\n    if len(genes) != 140 or len(set(genes)) != 140:\n        raise ValueError(\'Expected 140 unique genes\')\n    model = SpiceAdaptedMouseNoLog(gene_dim=len(genes), num_celltypes=len(dataset.cell_types_dict)).to(device)\n    print(f\'Split audit passed. Starting SEA-AD SPICE-adapted, seed {SEED}, max 10 epochs, patience 5.\', flush=True)\n    run_experiment(dataset, split, genes, out, model, device)\n    write_json(out / \'source_provenance.json\', dict(revision=str(revision), split_sha256=hashlib.sha256(split_path.read_bytes()).hexdigest(), dataloader_sha256=digest, runner_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()))\nif __name__ == \'__main__\':\n    main()\n', 'runners/LightGBM.py': '# SEA-AD adaptation of the archived Mouse revision runner.\n# Only dataset metadata, split counts, dimensions checked in main, and log labels change.\n# Network classes, features, losses, optimizers, selection and scoring are retained.\n# Historical class identifiers (MouseNoLog) do NOT perform normalization.\nSEED = 123\n"""Mouse LightGBM: fixed inductive split, existing 100-tree settings.\n\nNo hyperparameter search. Only training-derived residuals are used. The original\n49-neighbor mean feature is retained, with homotypic residuals masked to zero\nbefore averaging to obey the same residual-information restriction as SpaGAT.\nNo new baseline/type/distance features are introduced.\n"""\nimport argparse\nimport hashlib\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport sys\nimport time\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader, Subset\nDEFAULT_REVISION = \'/content/drive/MyDrive/SpaGAT_revision/inductive_training/20260922_104353_552891_UTC_f291de00\'\nPARAMETERS = dict(objective=\'regression\', n_estimators=100, learning_rate=0.05, num_leaves=31, max_depth=-1, subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbosity=-1)\n\ndef write_json(path, content):\n    with Path(path).open(\'x\', encoding=\'utf-8\') as handle:\n        json.dump(content, handle, indent=2, ensure_ascii=False, allow_nan=False)\n\ndef load_split(path, dataset, data_dir):\n    split = json.loads(Path(path).read_text())\n    keys = (\'train_indices\', \'val_indices\', \'test_indices\')\n    for key in keys:\n        if not isinstance(split.get(key), list) or not split[key] or any((type(i) is not int for i in split[key])):\n            raise ValueError(f\'{key}: expected a nonempty integer list\')\n    indices = [i for key in keys for i in split[key]]\n    if len(indices) != len(dataset) or set(indices) != set(range(len(dataset))):\n        raise ValueError(\'Split is not disjoint, in range and complete\')\n    if Path(split[\'processed_dir\']).resolve() != Path(data_dir).resolve():\n        raise ValueError(\'Split processed_dir mismatch\')\n    for key, count_key in zip(keys, (\'n_train\', \'n_validation\', \'n_test\')):\n        if count_key in split and split[count_key] != len(split[key]):\n            raise ValueError(f\'{count_key} mismatch\')\n    audit = pd.read_csv(Path(path).with_name(Path(path).stem + \'_samples.csv\'))\n    if audit[\'sample\'].tolist() != list(dataset.samples):\n        raise ValueError(\'Sample order differs from verified audit\')\n    counts = np.asarray(dataset.meta_counts, dtype=np.int64)\n    ends = np.cumsum(counts)\n    starts = ends - counts\n    for key, expected in [(\'eligible_cells\', counts), (\'global_start\', starts), (\'global_end_exclusive\', ends)]:\n        if not np.array_equal(audit[key].to_numpy(), expected):\n            raise ValueError(f\'Sample audit {key} mismatch\')\n    membership = np.empty(len(dataset), dtype=np.int8)\n    for label, key in enumerate(keys):\n        membership[split[key]] = label\n    labels = (\'train\', \'validation\', \'test\')\n    for row, start, end in zip(audit.itertuples(index=False), starts, ends):\n        if row.split not in labels or not np.all(membership[start:end] == labels.index(row.split)):\n            raise ValueError(f\'{row.sample}: sample crosses partitions or audit mismatch\')\n    return split\n\ndef batch_features(batch):\n    x = batch[\'x\'].numpy()\n    ct = batch[\'cell_types\'].numpy()\n    if x.ndim != 3 or x.shape[1] != 50 or ct.shape != x.shape[:2]:\n        raise ValueError(\'Expected 50-cell neighborhoods with aligned cell types\')\n    if \'neighbor_mask\' in batch and (not bool(torch.all(batch[\'neighbor_mask\'] == 1))):\n        raise ValueError(\'Variable neighborhoods require explicit handling; expected all 50 valid nodes\')\n    heterotypic = ct[:, 1:] != ct[:, :1]\n    features = np.where(heterotypic[..., None], x[:, 1:, :], 0.0).mean(axis=1, dtype=np.float32)\n    target = batch[\'y\'].numpy()\n    if not np.array_equal(target, x[:, 0, :]):\n        raise ValueError(\'Dataset center target and residual disagree\')\n    if not np.isfinite(features).all() or not np.isfinite(target).all():\n        raise ValueError(\'Nonfinite features/targets\')\n    return (features, target)\n\ndef build_features(dataset, indices, genes, label, batch_size=128):\n    x = np.empty((len(indices), len(genes)), dtype=np.float32)\n    y = np.empty_like(x)\n    cursor = 0\n    loader = DataLoader(Subset(dataset, indices), batch_size=batch_size, shuffle=False, drop_last=False, num_workers=0)\n    for batch in loader:\n        features, target = batch_features(batch)\n        n = len(target)\n        x[cursor:cursor + n] = features\n        y[cursor:cursor + n] = target\n        cursor += n\n        if cursor == n or cursor % (batch_size * 80) == 0 or cursor == len(indices):\n            print(f\'{label} features: {cursor}/{len(indices)}\', flush=True)\n    if cursor != len(indices):\n        raise ValueError(\'Incomplete feature construction\')\n    return (x, y)\n\ndef evaluate_arrays(prediction, target):\n    """Match the established SpaGAT PCC/MSE conventions; use paper EV_zero."""\n    if prediction.shape != target.shape or not np.isfinite(prediction).all() or (not np.isfinite(target).all()):\n        raise ValueError(\'Invalid prediction/target arrays\')\n    prediction = torch.as_tensor(prediction).float()\n    target = torch.as_tensor(target).float()\n    pc = prediction - prediction.mean(dim=0, keepdim=True)\n    tc = target - target.mean(dim=0, keepdim=True)\n    denominator = torch.sqrt((pc.square().sum(dim=0) * tc.square().sum(dim=0)).clamp_min(1e-12))\n    pcc = ((pc * tc).sum(dim=0) / denominator).clamp(-1, 1)\n    mse = torch.mean((prediction - target).square()).item()\n    mse0 = torch.mean(target.square()).item()\n    metrics = dict(median_gene_pcc=torch.median(pcc).item(), mean_gene_pcc=torch.mean(pcc).item(), mse=mse, mse_zero=mse0, ev_zero_percent=None if mse0 == 0 else 100 * (1 - mse / mse0), n_cells=int(target.shape[0]), n_genes=int(target.shape[1]))\n    return (metrics, pcc.numpy())\n\ndef run_experiment(dataset, split, genes, out_dir, lgb, batch_size=128):\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=False)\n    (out_dir / \'models\').mkdir()\n    write_json(out_dir / \'shared_split.json\', split)\n    write_json(out_dir / \'run_config.json\', {\'dataset\': \'SEA_AD\', \'model\': \'LightGBM\', \'seed\': SEED, \'parameters\': PARAMETERS, \'features\': \'49-neighbor residual mean; homotypic residuals zeroed; center excluded; denominator 49\', \'feature_count\': len(genes), \'new_features_added\': False, \'selection\': \'Fixed 100 trees per gene; no early stopping or hyperparameter search\', \'pcc\': \'SpaGAT float32 centered PCC, clamp denominator product at 1e-12, torch.median lower-middle convention; constants yield zero\', \'ev\': \'100*(1-MSE/MSE0), zero-residual reference\', \'target_space\': \'residual relative to training-derived cell-type baseline\', \'versions\': {\'python\': sys.version, \'numpy\': np.__version__, \'torch\': torch.__version__, \'pandas\': pd.__version__, \'lightgbm\': getattr(lgb, \'__version__\', \'test double\')}})\n    started = time.perf_counter()\n    x_train, y_train = build_features(dataset, split[\'train_indices\'], genes, \'train\', batch_size)\n    x_val, y_val = build_features(dataset, split[\'val_indices\'], genes, \'validation\', batch_size)\n    val_pred = np.empty_like(y_val)\n    records = []\n    for g, gene in enumerate(genes):\n        t = time.perf_counter()\n        model = lgb.LGBMRegressor(**PARAMETERS)\n        model.fit(x_train, y_train[:, g])\n        val_pred[:, g] = model.predict(x_val)\n        model.booster_.save_model(str(out_dir / \'models\' / f\'gene_{g:04d}.txt\'))\n        records.append(dict(gene_index=g, gene=gene, runtime_seconds=time.perf_counter() - t))\n        pd.DataFrame(records).to_csv(out_dir / \'training_progress.csv\', index=False)\n        if g == 0 or (g + 1) % 10 == 0 or g + 1 == len(genes):\n            print(f\'Trained {g + 1}/{len(genes)} genes; elapsed {(time.perf_counter() - started) / 60:.1f} min\', flush=True)\n        del model\n    pd.DataFrame({\'gene_index\': range(len(genes)), \'gene\': genes}).to_csv(out_dir / \'gene_order.csv\', index=False)\n    val_metrics, _ = evaluate_arrays(val_pred, y_val)\n    write_json(out_dir / \'validation_metrics.json\', val_metrics)\n    del x_train, y_train, x_val, y_val, val_pred\n    print(\'All models fixed and saved. Starting the single final test evaluation.\', flush=True)\n    x_test, y_test = build_features(dataset, split[\'test_indices\'], genes, \'test\', batch_size)\n    prediction = np.empty_like(y_test)\n    for g in range(len(genes)):\n        model = lgb.Booster(model_file=str(out_dir / \'models\' / f\'gene_{g:04d}.txt\'))\n        prediction[:, g] = model.predict(x_test)\n    metrics, pcc = evaluate_arrays(prediction, y_test)\n    metrics.update(dataset=\'SEA_AD\', model=\'LightGBM\', seed=SEED, runtime_seconds=time.perf_counter() - started)\n    np.savez_compressed(out_dir / \'test_predictions.npz\', prediction=prediction, target=y_test, test_indices=np.asarray(split[\'test_indices\'], dtype=np.int64), genes=np.asarray(genes, dtype=str))\n    pd.DataFrame({\'gene\': genes, \'PCC\': pcc}).to_csv(out_dir / \'test_per_gene_PCC.csv\', index=False)\n    write_json(out_dir / \'test_metrics.json\', metrics)\n    pd.DataFrame([metrics]).to_csv(out_dir / \'test_metrics.csv\', index=False)\n    print(json.dumps(metrics, indent=2), flush=True)\n    print(\'Completed; saved to:\', out_dir, flush=True)\n    return metrics\n\ndef main():\n    global SEED\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--revision-dir\', default=DEFAULT_REVISION)\n    parser.add_argument(\'--output-dir\', default=None)\n    parser.add_argument(\'--seed\', type=int, default=123)\n    args = parser.parse_args()\n    SEED = args.seed\n    PARAMETERS["random_state"] = SEED\n    revision = Path(args.revision_dir)\n    out = Path(args.output_dir) if args.output_dir else revision / \'baseline_results\' / f\'SEA_AD_LightGBM_seed{SEED}\'\n    if out.exists():\n        raise FileExistsError(f\'Refusing to overwrite existing results: {out}\')\n    split_path = revision / \'splits\' / \'SEA_AD_inductive_split_indices.json\'\n    metadata = json.loads(split_path.read_text())\n    data_dir = Path(metadata[\'processed_dir\'])\n    if tuple((len(metadata[key]) for key in (\'train_indices\', \'val_indices\', \'test_indices\'))) != (257152, 54026, 54762):\n        raise ValueError(\'SEA-AD split counts differ from the verified experiment\')\n    loader_path = revision / \'spagat\' / \'dataloader.py\'\n    manifest = json.loads((revision / \'revision_manifest.json\').read_text())\n    digest = hashlib.sha256(loader_path.read_bytes()).hexdigest()\n    if digest != manifest[\'files_sha256\'][\'spagat/dataloader.py\']:\n        raise ValueError(\'Frozen SpaGAT dataloader changed since revision installation\')\n    spec = importlib.util.spec_from_file_location(\'frozen_spagat_data\', loader_path)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    import lightgbm as lgb\n    print(\'Loading the verified SEA-AD inductive processed data...\', flush=True)\n    dataset = module.SPAGAT_dataset(processed_dir=str(data_dir), num_neighbors=50)\n    split = load_split(split_path, dataset, data_dir)\n    genes = list(dataset.genes)\n    if len(genes) != 140 or len(set(genes)) != len(genes):\n        raise ValueError(\'Expected 140 unique SEA-AD genes\')\n    print(\'Fixed split and sample audit passed. Training fixed 100-tree models.\', flush=True)\n    run_experiment(dataset, split, genes, out, lgb)\n    write_json(out / \'source_provenance.json\', {\'revision\': str(revision), \'split_sha256\': hashlib.sha256(split_path.read_bytes()).hexdigest(), \'dataloader_sha256\': digest, \'runner_sha256\': hashlib.sha256(Path(__file__).read_bytes()).hexdigest()})\nif __name__ == \'__main__\':\n    main()\n'}
SOURCE_HASHES = {'spagat/__init__.py': 'af200e1784421e427d17548ed9449a9257cd96ab1a0be46cc81e3c4fbf4b82ea', 'spagat/attention.py': '5a8c0fcb97e7973a5f842e4515b17e8211eb779d529800f0a0c4e40b2ef792af', 'spagat/dataloader.py': '9f7e97c8fd12a5f9d8fc968987549fbdcac52deff784ac9bfac3bd8e14fb12a2', 'spagat/embedding.py': '8e3d12e28410b3b40642564e5c34be9da87d1ee6a3de939e6a78f29f90634987', 'spagat/gene_program_model.py': 'e9f7cdf2280f14659cfa8dd477f1ec3087a1c09bed981dcf31b890ceb50aa5a6', 'spagat/preprocess.py': '9bb77b4e200f8b1456bf2eb8901982816c0706be5878cea7f5e54e19676b7490', 'runners/GAT.py': '7f54709176e2b855323955bfc5957c8d0907a80ef60fb43cd31a9e5516b83fd5', 'runners/GITIII_official.py': '3a7e96a783ffd4544e5e09429f7f86d034eee564089a6f57f8cc46f889ed458c', 'runners/SPICE_adapted.py': '25e0daec779ba2913f0b117788663b4de9537318b491c78cf2199af2fdf0ff4b', 'runners/LightGBM.py': 'e4e64b10e445e9f2e90a15bbacd4c641995e2d2a7d9a99d819daa742c7a38c02'}
SPECS = {'SEA_AD': {'expected_counts': [257152, 54026, 54762], 'n_genes': 140, 'split_sha256': 'b3536c1b8ca6b198a588ae1fa2b846b06214106c9f687200c79511f3d5b8250c', 'audit_sha256': '56f6a99b6b71149b7b17116198c34149d5f34e19e71ae811fe28a9b807c2be54', 'expected_parameters': {'SpaGAT': 2520652, 'GAT': 90508, 'SPICE_adapted': 71564, 'GITIII_official': 1295992}}, 'Mouse': {'expected_counts': [122886, 36193, 117257], 'n_genes': 254, 'split_sha256': 'd7ec406fc3be3cf2aba12c53d0e9a9d901ed830f9827554e901fed494f8bb113', 'audit_sha256': 'f3d3022be6bd09f519834e2b7ea0cc614542e59ea3a0fe98de54f4332f3c5e4e', 'expected_parameters': {'SpaGAT': 2768288, 'GAT': 119806, 'SPICE_adapted': 100862, 'GITIII_official': 2363858}}}
WORKER_SOURCE = '"""Isolated, bounded resource profiling of the frozen SpaGAT benchmark.\n\nAll writes go to a new local output directory. Existing models are read only.\nThis script is not used to replace the published prediction/accuracy results.\n"""\nfrom pathlib import Path\nimport argparse\nimport contextlib\nimport gc\nimport hashlib\nimport importlib.util\nfrom importlib import metadata as package_metadata\nimport json\nimport os\nimport platform\nimport random\nimport resource\nimport sys\nimport threading\nimport time\nimport traceback\n\nimport numpy as np\nimport pandas as pd\nimport psutil\nimport torch\nfrom torch.utils.data import DataLoader, Subset\n\nGIB = 1024 ** 3\n\ndef write_json(path, obj):\n    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False))\n\nclass Measure:\n    """20 ms process-tree RSS samples and PyTorch allocator high-water marks."""\n    def __init__(self, gpu=False):\n        self.gpu = gpu\n        self.proc = psutil.Process()\n        self.stop = threading.Event()\n        self.peak_rss = 0\n\n    def sample(self):\n        try:\n            rss = self.proc.memory_info().rss\n            for child in self.proc.children(recursive=True):\n                with contextlib.suppress(psutil.Error):\n                    rss += child.memory_info().rss\n            self.peak_rss = max(self.peak_rss, rss)\n        except psutil.Error:\n            pass\n\n    def poll(self):\n        while not self.stop.wait(0.02):\n            self.sample()\n\n    def __enter__(self):\n        if self.gpu:\n            torch.cuda.synchronize()\n            torch.cuda.reset_peak_memory_stats()\n        self.sample()\n        self.initial_rss = self.peak_rss\n        self.started = time.perf_counter()\n        self.thread = threading.Thread(target=self.poll, daemon=True)\n        self.thread.start()\n        return self\n\n    def __exit__(self, *exc):\n        try:\n            if self.gpu:\n                torch.cuda.synchronize()\n        finally:\n            self.seconds = time.perf_counter() - self.started\n            self.sample()\n            self.stop.set()\n            self.thread.join()\n            self.result = {\n                \'seconds\': self.seconds,\n                \'peak_process_tree_rss_gib_sampled\': self.peak_rss / GIB,\n                \'initial_process_tree_rss_gib\': self.initial_rss / GIB,\n                \'rss_sampling_interval_seconds\': 0.02,\n                \'peak_cuda_allocated_gib\': torch.cuda.max_memory_allocated() / GIB if self.gpu else None,\n                \'peak_cuda_reserved_gib\': torch.cuda.max_memory_reserved() / GIB if self.gpu else None,\n            }\n\ndef module_from_path(name, path):\n    spec = importlib.util.spec_from_file_location(name, path)\n    module = importlib.util.module_from_spec(spec)\n    sys.modules[name] = module\n    spec.loader.exec_module(module)\n    return module\n\ndef initialize_dataset(config):\n    from spagat.dataloader import SPAGAT_dataset\n    with Measure() as measure:\n        dataset = SPAGAT_dataset(config[\'processed_local\'], num_neighbors=50)\n    split = json.loads(Path(config[\'split_local\']).read_text())\n    counts = tuple(len(split[key]) for key in (\'train_indices\', \'val_indices\', \'test_indices\'))\n    if counts != tuple(config[\'expected_counts\']) or len(dataset) != sum(counts):\n        raise ValueError(f\'Dataset or partition count mismatch: {counts}, dataset={len(dataset)}\')\n    if len(dataset.genes) != config[\'n_genes\']:\n        raise ValueError(\'Gene count mismatch\')\n    # Validate every eligible row is assigned once without regenerating the split.\n    joined = np.concatenate([np.asarray(split[k], dtype=np.int64) for k in (\'train_indices\',\'val_indices\',\'test_indices\')])\n    if not np.array_equal(np.sort(joined), np.arange(len(dataset))):\n        raise ValueError(\'Split indices are not a partition of the dataset\')\n    audit = pd.read_csv(config[\'sample_audit_local\'])\n    if \'sample\' not in audit or list(audit[\'sample\'].astype(str)) != list(map(str,dataset.samples)):\n        raise ValueError(\'Saved sample order differs from the dataset order\')\n    eligible=np.asarray(dataset.meta_counts,dtype=np.int64)\n    ends=np.cumsum(eligible);starts=ends-eligible\n    for col,expected in [(\'eligible_cells\',eligible),(\'global_start\',starts),(\'global_end_exclusive\',ends)]:\n        if col not in audit or not np.array_equal(audit[col].to_numpy(),expected):\n            raise ValueError(\'Sample audit count/offset mismatch: \'+col)\n    membership=np.empty(len(dataset),dtype=np.int8)\n    for label,key in enumerate((\'train_indices\',\'val_indices\',\'test_indices\')):membership[split[key]]=label\n    labels=(\'train\',\'validation\',\'test\')\n    for row,start,end in zip(audit.itertuples(index=False),starts,ends):\n        if row.split not in labels or not np.all(membership[start:end]==labels.index(row.split)):\n            raise ValueError(\'Sample partition mismatch: \'+row.sample)\n    return dataset, split, measure.result\n\ndef neural_model(name, dataset, config, device):\n    runner = None\n    criterion = None\n    checkpoint_path = config[\'models_local\'][name]\n    if name == \'SpaGAT\':\n        from spagat.gene_program_model import SpaGP, SpaGP_Loss\n        saved = torch.load(checkpoint_path, map_location=\'cpu\', weights_only=False)\n        old = saved.get(\'config\', {})\n        if old.get(\'num_neighbors\',50)!=50 or old.get(\'batch_size\',32)!=32:\n            raise ValueError(\'Unexpected SpaGAT neighborhood/batch configuration\')\n        model = SpaGP(genes=dataset.genes, ligands_info=dataset.interactions,\n                      node_dim=256, edge_dim=48, num_heads=2, n_layers=1,\n                      att_dim=8, n_programs=4, program_aware=True,\n                      program_token_dim=32, message_decoder=\'free\', routing_mode=\'program\')\n        model.load_state_dict(saved[\'model\'], strict=True)\n        criterion = SpaGP_Loss(dataset.genes,dataset.interactions,lambda_orth=0.1,lambda_sparse=0.01).to(device)\n        del saved\n    else:\n        runner = module_from_path(\'profile_runner_\'+name, Path(config[\'source_local\'])/\'runners\'/f\'{name}.py\')\n        if name == \'GAT\':\n            model = runner.LocalGATMouseNoLog(gene_dim=len(dataset.genes),num_celltypes=len(dataset.cell_types_dict))\n        elif name == \'SPICE_adapted\':\n            model = runner.SpiceAdaptedMouseNoLog(gene_dim=len(dataset.genes),num_celltypes=len(dataset.cell_types_dict))\n        elif name == \'GITIII_official\':\n            cls = runner.load_official_model_class()\n            model = cls(dataset.genes,dataset.interactions,node_dim=256,edge_dim=48,\n                        num_heads=2,n_layers=1,node_dim_small=16,att_dim=8,use_cell_type_embedding=True)\n        else:\n            raise ValueError(name)\n        saved = torch.load(checkpoint_path,map_location=\'cpu\',weights_only=True)\n        model.load_state_dict(saved[\'model_state_dict\'],strict=True)\n        del saved\n    parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    if parameter_count != config[\'expected_parameters\'][name]:\n        raise ValueError(f\'{name} parameter count mismatch: {parameter_count}\')\n    model = model.to(device)\n    return model, runner, criterion, parameter_count\n\ndef forward(name, model, runner, criterion, batch, device, training=False):\n    if name == \'SpaGAT\':\n        values={k:v.to(device) for k,v in batch.items()}\n        output=model(values)\n        pred=output[0]\n        loss=criterion(output,values[\'y\'])[0] if training else None\n    elif name == \'GITIII_official\':\n        values,target=runner.prepare_batch(batch,device)\n        pred=model(values)[0]\n        loss=torch.nn.functional.mse_loss(pred,target) if training else None\n    else:\n        x,ct,target=runner.prepare_batch(batch,device)\n        pred=model(x,ct)\n        loss=torch.nn.functional.mse_loss(pred,target) if training else None\n    return pred,loss\n\ndef neural_train(name,dataset,split,config,device):\n    with Measure(gpu=True) as setup:\n        model,runner,criterion,count=neural_model(name,dataset,config,device)\n    betas=(0.99,0.999) if name in (\'SpaGAT\',\'GITIII_official\') else (0.9,0.999)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=0.0001,betas=betas)\n    # Same fixed shuffled receiver sequence across all neural methods.\n    rng=np.random.default_rng(config[\'seed\'])\n    indices=rng.permutation(np.asarray(split[\'train_indices\']))\n    total=(config[\'warmup_batches\']+config[\'train_batches\'])*config[\'batch_size\']\n    if len(indices)<total:\n        raise ValueError(\'Insufficient training receivers for the requested profile\')\n    loader=iter(DataLoader(Subset(dataset,indices[:total].tolist()),batch_size=config[\'batch_size\'],num_workers=0))\n    model.train()\n    def step(batch):\n        optimizer.zero_grad(set_to_none=True)\n        _,loss=forward(name,model,runner,criterion,batch,device,training=True)\n        if not bool(torch.isfinite(loss)):\n            raise ValueError(\'Nonfinite training loss\')\n        loss.backward()\n        optimizer.step()\n    with Measure(gpu=True) as warmup:\n        for _ in range(config[\'warmup_batches\']):\n            step(next(loader))\n    with Measure(gpu=True) as measure:\n        for i in range(config[\'train_batches\']):\n            step(next(loader))\n            if (i+1)%25==0:\n                print(f\'{name}: measured training batch {i+1}/{config["train_batches"]}\',flush=True)\n    return {\'parameter_count\':count,\'warmup_batches\':config[\'warmup_batches\'],\n            \'measured_batches\':config[\'train_batches\'],\'batch_size\':config[\'batch_size\'],\n            \'model_setup\':setup.result,\'warmup_profile\':warmup.result,\'training_profile\':measure.result,\n            \'scope\':\'Fixed-batch training profile from a saved checkpoint with a fresh optimizer; excludes validation and checkpoint serialization. No trained weights saved.\'}\n\ndef validate_predictions(pred,config,name):\n    path=Path(config[\'references_original\'][name])\n    with np.load(path,allow_pickle=False) as z:\n        key=\'prediction\' if \'prediction\' in z.files else (\'predictions\' if \'predictions\' in z.files else None)\n        if key is None:\n            raise ValueError(\'Saved reference prediction key not recognized\')\n        reference=np.asarray(z[key],dtype=np.float32)\n        if reference.shape!=pred.shape:\n            raise ValueError(\'Reference prediction shape mismatch\')\n        if \'genes\' not in z.files or list(z[\'genes\'].astype(str))!=list(config[\'genes\']):\n            raise ValueError(\'Reference gene order mismatch\')\n        if \'test_indices\' not in z.files or not np.array_equal(z[\'test_indices\'],config[\'test_indices\']):\n            raise ValueError(\'Reference test receiver order mismatch\')\n        error=np.abs(pred-reference)\n        valid=bool(np.allclose(pred,reference,rtol=1e-4,atol=1e-4))\n        result={\'matches_saved_predictions_within_tolerance\':valid,\'max_abs_error\':float(error.max()),\n                \'mean_abs_error\':float(error.mean()),\'rtol\':1e-4,\'atol\':1e-4}\n    return result\n\ndef inference(name,dataset,split,config,device):\n    # Start from cached processed data and local checkpoint files. Include model\n    # loading, test input preparation, forward passes, and CPU output assembly.\n    with Measure(gpu=name!=\'LightGBM\') as measure:\n        predictions=np.empty((len(split[\'test_indices\']),len(dataset.genes)),dtype=np.float32)\n        if name==\'LightGBM\':\n            import lightgbm as lgb\n            runner=module_from_path(\'profile_lgb\',Path(config[\'source_local\'])/\'runners/LightGBM.py\')\n            with Measure() as features:\n                x_test,y_test=runner.build_features(dataset,split[\'test_indices\'],dataset.genes,\'test\',128)\n            for g in range(len(dataset.genes)):\n                booster=lgb.Booster(model_file=str(Path(config[\'models_local\'][name])/f\'gene_{g:04d}.txt\'))\n                predictions[:,g]=booster.predict(x_test)\n                del booster\n                if (g+1)%20==0: print(f\'LightGBM inference: {g+1}/{len(dataset.genes)} genes\',flush=True)\n            extra={\'test_feature_construction\':features.result,\'parameter_count\':None}\n        else:\n            model,runner,criterion,count=neural_model(name,dataset,config,device)\n            model.eval()\n            cursor=0\n            loader=DataLoader(Subset(dataset,split[\'test_indices\']),batch_size=config[\'batch_size\'],num_workers=0)\n            with torch.no_grad():\n                for i,batch in enumerate(loader):\n                    pred,_=forward(name,model,runner,criterion,batch,device)\n                    n=len(pred)\n                    predictions[cursor:cursor+n]=pred.detach().cpu().numpy()\n                    cursor+=n\n                    if (i+1)%400==0:print(f\'{name} inference: {cursor}/{len(predictions)} receivers\',flush=True)\n            if cursor!=len(predictions):raise ValueError(\'Incomplete inference\')\n            extra={\'parameter_count\':count}\n    if not np.isfinite(predictions).all():raise ValueError(\'Nonfinite predictions\')\n    comparison_config={**config,\'genes\':list(map(str,dataset.genes)),\'test_indices\':split[\'test_indices\']}\n    agreement=validate_predictions(predictions,comparison_config,name)\n    return {**extra,\'inference\':measure.result,\'prediction_agreement\':agreement,\n            \'n_test_receivers\':len(predictions),\'n_genes\':len(dataset.genes),\n            \'scope\':\'Checkpoint-to-prediction wall time; includes local model loading, test input preparation, and CPU prediction collection; excludes processed dataset loading, initial Drive copy, accuracy scoring and reference comparison.\'}\n\ndef lightgbm_train(dataset,split,config,job_dir):\n    import lightgbm as lgb\n    runner=module_from_path(\'profile_lgb_train\',Path(config[\'source_local\'])/\'runners/LightGBM.py\')\n    params=dict(runner.PARAMETERS);params[\'random_state\']=config[\'seed\']\n    with Measure() as measure:\n        with Measure() as feature_measure:\n            x_train,y_train=runner.build_features(dataset,split[\'train_indices\'],dataset.genes,\'train\',128)\n            x_val,y_val=runner.build_features(dataset,split[\'val_indices\'],dataset.genes,\'validation\',128)\n        val_pred=np.empty_like(y_val)\n        progress=[]\n        for g,gene in enumerate(dataset.genes):\n            started=time.perf_counter()\n            model=lgb.LGBMRegressor(**params)\n            model.fit(x_train,y_train[:,g])\n            val_pred[:,g]=model.predict(x_val)\n            progress.append({\'gene_index\':g,\'gene\':str(gene),\'fit_and_validation_seconds\':time.perf_counter()-started})\n            pd.DataFrame(progress).to_csv(job_dir/\'per_gene_profile.csv\',index=False)\n            del model\n            if (g+1)%10==0:print(f\'LightGBM fitting: {g+1}/{len(dataset.genes)} genes\',flush=True)\n    return {\'training_profile\':measure.result,\'feature_construction\':feature_measure.result,\n            \'n_genes_fitted\':len(progress),\'trees_per_gene\':params[\'n_estimators\'],\n            \'cpu_model\':platform.processor(),\'logical_cpu_count\':os.cpu_count(),\n            \'scope\':\'All genes, full training and validation partitions, original 100-tree settings; includes feature construction, fitting and validation prediction; excludes checkpoint serialization. Fresh fitted models discarded.\'}\n\ndef nearest_row(x,y,i,k=50):\n    # Same clipped-distance argpartition/sort operations as the supplied\n    # process_dataset.py get_index/argsort_topk implementation.\n    dx=np.abs(x-x[i]);dy=np.abs(y-y[i])\n    dx[dx>1e4]=1e4;dy[dy>1e4]=1e4\n    distance=np.sqrt(dx*dx+dy*dy)\n    indices=np.argpartition(distance,k)[:k]\n    return indices[np.argsort(distance[indices])]\n\ndef distance_equivalent(x,y,i,a,b):\n    def distances(indices):\n        dx=np.minimum(np.abs(x[indices]-x[i]),1e4)\n        dy=np.minimum(np.abs(y[indices]-y[i]),1e4)\n        return np.sort(np.sqrt(dx*dx+dy*dy))\n    return bool(np.allclose(distances(a),distances(b),rtol=1e-7,atol=1e-7))\n\ndef graph_profile(config,job_dir):\n    folder=Path(config[\'processed_local\']);reports=[]\n    with Measure() as measure:\n        for metadata in sorted(folder.glob(\'*_TypeExp.npz\')):\n            name=metadata.name[:-len(\'_TypeExp.npz\')]\n            cols=[\'centerx\',\'centery\']+[f\'index_{j}\' for j in range(50)]\n            frame=pd.read_csv(folder/(name+\'.csv\'),usecols=cols)\n            x=frame.centerx.to_numpy(dtype=np.float64);y=frame.centery.to_numpy(dtype=np.float64)\n            saved=frame[[f\'index_{j}\' for j in range(50)]].to_numpy(dtype=np.int64)\n            n=len(x)\n            if n<=50 or saved.min()<0 or saved.max()>=n or not np.isfinite(x).all() or not np.isfinite(y).all():\n                raise ValueError(\'Invalid section geometry/index table: \'+name)\n            checks=np.unique(np.linspace(0,n-1,min(n,32),dtype=int))\n            compatible=all(distance_equivalent(x,y,int(i),nearest_row(x,y,int(i)),saved[i]) for i in checks)\n            if not compatible:\n                reports.append({\'section\':name,\'n_cells\':n,\'status\':\'incompatible_neighbor_rule\',\'search_seconds\':None})\n                print(f\'{name}: neighbor rule differs; no timing extrapolated\',flush=True)\n                continue\n            rebuilt=np.empty_like(saved)\n            started=time.perf_counter()\n            for i in range(n):rebuilt[i]=nearest_row(x,y,i)\n            seconds=time.perf_counter()-started\n            mismatch=np.flatnonzero(np.any(rebuilt!=saved,axis=1))\n            unresolved=sum(not distance_equivalent(x,y,int(i),rebuilt[i],saved[i]) for i in mismatch)\n            reports.append({\'section\':name,\'n_cells\':n,\'status\':\'verified\' if not unresolved else \'unresolved_difference\',\n                            \'search_seconds\':seconds,\'index_order_mismatch_rows\':len(mismatch),\'unresolved_rows\':unresolved})\n            pd.DataFrame(reports).to_csv(job_dir/\'graph_sections.csv\',index=False)\n            print(f\'Neighborhood reconstruction {len(reports)}: {name}, {n} cells, {seconds:.1f}s\',flush=True)\n            del frame,x,y,saved,rebuilt\n    complete=bool(reports) and all(r[\'status\']==\'verified\' for r in reports)\n    return {\'shared_neighbor_construction\':measure.result,\'all_sections_verified\':complete,\n            \'sections\':reports,\'total_neighbor_search_seconds\':sum(r[\'search_seconds\'] or 0 for r in reports) if complete else None,\n            \'scope\':\'Shared 50-index neighborhood reconstruction from saved float64 coordinates, including all cells per section. Search timer excludes CSV loading, index allocation, eligibility filtering, expression preprocessing and verification. Whole-stage RSS includes CSV loading and verification. Existing input graph files are not changed. Model-specific edge tensor creation remains part of model runtime.\'}\n\ndef main():\n    parser=argparse.ArgumentParser();parser.add_argument(\'--config\',required=True);parser.add_argument(\'--model\',required=True)\n    parser.add_argument(\'--phase\',required=True,choices=[\'graph\',\'train\',\'inference\']);parser.add_argument(\'--output\',required=True)\n    args=parser.parse_args();config=json.loads(Path(args.config).read_text());job_dir=Path(args.output);job_dir.mkdir(parents=True,exist_ok=False)\n    sys.path.insert(0,config[\'source_local\'])\n    random.seed(config[\'seed\']);np.random.seed(config[\'seed\']);torch.manual_seed(config[\'seed\'])\n    gpu=args.model not in (\'LightGBM\',\'shared_graph\')\n    if gpu and not torch.cuda.is_available():raise RuntimeError(\'Select a Colab GPU runtime for neural profiling.\')\n    if gpu:\n        torch.cuda.manual_seed_all(config[\'seed\']);torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True\n    device=torch.device(\'cuda\' if gpu else \'cpu\')\n    result={\'dataset\':config[\'dataset\'],\'model\':args.model,\'phase\':args.phase,\'seed\':config[\'seed\'],\'status\':\'started\',\n            \'versions\':{\'python\':sys.version,\'torch\':torch.__version__,\'numpy\':np.__version__,\'pandas\':pd.__version__},\n            \'hardware\':{\'gpu\':torch.cuda.get_device_name() if gpu else None,\'logical_cpus\':os.cpu_count(),\'torch_cpu_threads\':torch.get_num_threads(),\n                        \'cpu_affinity\':psutil.Process().cpu_affinity(),\n                        \'host_total_ram_gib\':psutil.virtual_memory().total/GIB},\n            \'original_files_modified\':False,\'source_hashes\':config[\'source_hashes\']}\n    for package in [\'torch-geometric\',\'lightgbm\',\'psutil\']:\n        with contextlib.suppress(package_metadata.PackageNotFoundError):result[\'versions\'][package]=package_metadata.version(package)\n    with contextlib.suppress(OSError):\n        result[\'hardware\'][\'cpu_model\']=next((line.split(\':\',1)[1].strip() for line in Path(\'/proc/cpuinfo\').read_text().splitlines() if line.startswith(\'model name\')),platform.processor())\n    started=time.perf_counter()\n    try:\n        if args.phase==\'graph\':result.update(graph_profile(config,job_dir))\n        else:\n            dataset,split,loading=initialize_dataset(config);result[\'dataset_loading\']=loading\n            if args.phase==\'inference\':result.update(inference(args.model,dataset,split,config,device))\n            elif args.model==\'LightGBM\':result.update(lightgbm_train(dataset,split,config,job_dir))\n            else:result.update(neural_train(args.model,dataset,split,config,device))\n        result[\'status\']=\'complete\'\n    except Exception as exc:\n        result.update(status=\'error\',error=str(exc),traceback=traceback.format_exc());traceback.print_exc()\n    finally:\n        result[\'job_wall_seconds\']=time.perf_counter()-started\n        result[\'whole_process_peak_rss_gib_including_postcheck\']=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss*1024/GIB\n        write_json(job_dir/\'profile.json\',result)\n    if result[\'status\']!=\'complete\':raise SystemExit(1)\n    print(\'Completed:\',args.model,args.phase,flush=True)\n\nif __name__==\'__main__\':main()\n'
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import shutil
import subprocess
import sys
import uuid
import zipfile

def sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda:handle.read(1024*1024),b''):digest.update(chunk)
    return digest.hexdigest()

def model_directory(dataset,model):
    if model=='SpaGAT':return ROOT/'results'/dataset
    if dataset=='SEA_AD':return ROOT/'seaad_main_benchmark_v1/results'/f'{model}_seed123'
    return ROOT/'baseline_results'/f'Mouse_{model}_seed123'

def prepare_profiles():
    import torch
    if not torch.cuda.is_available():raise RuntimeError('请选择 GPU runtime；这里将开始真实资源测量。')
    expected_dependencies=['numpy','pandas','psutil','lightgbm','torch_geometric']
    for name in expected_dependencies:__import__(name)
    stamp=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')
    run_dir=Path('/content')/f'R2_resource_profile_{stamp}_{uuid.uuid4().hex[:6]}'
    run_dir.mkdir(exist_ok=False);source_dir=run_dir/'sources';source_dir.mkdir()
    for relative,content in SOURCES.items():
        p=source_dir/relative;p.parent.mkdir(parents=True,exist_ok=True);p.write_text(content)
        if sha256(p)!=SOURCE_HASHES[relative]:raise ValueError('Embedded source hash mismatch: '+relative)
    (run_dir/'profile_worker.py').write_text(WORKER_SOURCE)
    result_dir=run_dir/'results';result_dir.mkdir()
    revision_manifest=json.loads((ROOT/'revision_manifest.json').read_text())
    for relative,digest in SOURCE_HASHES.items():
        if relative.startswith('spagat/'):
            if revision_manifest['files_sha256'].get(relative)!=digest:raise ValueError('Frozen source manifest mismatch: '+relative)
    # Check all inputs and checkpoints before starting any expensive job.
    prepared=[]
    for dataset in DATASETS:
        if dataset not in SPECS:raise ValueError('Unsupported dataset: '+dataset)
        split=ROOT/'splits'/f'{dataset}_inductive_split_indices.json'
        audit=split.with_name(split.stem+'_samples.csv')
        if sha256(split)!=SPECS[dataset]['split_sha256'] or sha256(audit)!=SPECS[dataset]['audit_sha256']:
            raise ValueError('Split/sample audit hash mismatch: '+dataset)
        metadata=json.loads(split.read_text());processed=Path(metadata['processed_dir'])
        if not processed.is_dir():raise FileNotFoundError(processed)
        models={};references={}
        for model in MODELS:
            folder=model_directory(dataset,model)
            if model=='LightGBM':
                checkpoint=folder/'models'
                for g in range(SPECS[dataset]['n_genes']):
                    if not (checkpoint/f'gene_{g:04d}.txt').is_file():raise FileNotFoundError(checkpoint/f'gene_{g:04d}.txt')
            else:
                filename={'SpaGAT':'spagat','GAT':'gat','GITIII_official':'gitiii_official','SPICE_adapted':'spice_adapted'}[model]+'_seed123_best.pth'
                checkpoint=folder/filename
                if not checkpoint.is_file():raise FileNotFoundError(checkpoint)
            prediction=folder/('seed123_test/test_predictions.npz' if model=='SpaGAT' else 'test_predictions.npz')
            if not prediction.is_file():raise FileNotFoundError(prediction)
            models[model]=checkpoint;references[model]=prediction
        prepared.append((dataset,split,audit,processed,models,references))
    configs=[];copied=[]
    for dataset,split,audit,processed,models,references in prepared:
        print(f'{dataset}: 将已有数据复制到本次临时目录，避免不同模型反复从 Drive 读取。',flush=True)
        data_root=run_dir/'cache'/dataset/'data';destination=data_root/'processed';destination.mkdir(parents=True)
        for name in ['genes.pth','ligands.pth']:
            shutil.copy2(processed.parent/name,data_root/name)
        metadata_files=sorted(processed.glob('*_TypeExp.npz'))
        if not metadata_files:raise FileNotFoundError('No processed section files')
        for i,p in enumerate(metadata_files):
            csv_path=processed/(p.name[:-len('_TypeExp.npz')]+'.csv')
            for src in [p,csv_path]:
                shutil.copy2(src,destination/src.name)
                copied.append({'source':str(src),'bytes':src.stat().st_size})
            if (i+1)%10==0:print(f'{dataset}: copied {i+1}/{len(metadata_files)} sections',flush=True)
        local_split=run_dir/'cache'/dataset/split.name;local_audit=local_split.with_name(audit.name)
        shutil.copy2(split,local_split);shutil.copy2(audit,local_audit)
        local_models={};model_hashes={}
        for model,src in models.items():
            dst=run_dir/'cache'/dataset/'models'/model;dst.mkdir(parents=True)
            if model=='LightGBM':
                for g in range(SPECS[dataset]['n_genes']):
                    p=src/f'gene_{g:04d}.txt';shutil.copy2(p,dst/p.name)
                local_models[model]=str(dst)
                model_hashes[model]={p.name:sha256(p) for p in sorted(dst.glob('*.txt'))}
            else:
                shutil.copy2(src,dst/src.name);local_models[model]=str(dst/src.name);model_hashes[model]=sha256(dst/src.name)
        config={**SPECS[dataset],'dataset':dataset,'seed':123,'batch_size':32,'train_batches':100,'warmup_batches':10,
                'processed_local':str(destination),'split_local':str(local_split),'sample_audit_local':str(local_audit),
                'models_local':local_models,'references_original':{k:str(v) for k,v in references.items()},
                'source_local':str(source_dir),'source_hashes':SOURCE_HASHES,'model_hashes':model_hashes}
        config_path=run_dir/f'{dataset}_config.json';config_path.write_text(json.dumps(config,indent=2));configs.append(config_path)
    manifest={'original_root':str(ROOT),'datasets':DATASETS,'seed':123,'created_utc':stamp,
              'profile_scope':'Four neural models: 10 warm-up plus 100 measured training batches, and full test inference. LightGBM: all-gene 100-tree fitting/validation and saved-model full test inference. Shared 50-neighbor reconstruction once per dataset.',
              'source_hashes':SOURCE_HASHES,'worker_sha256':sha256(run_dir/'profile_worker.py'),
              'cuda_gpu':torch.cuda.get_device_name(),'copied_inputs':copied,'original_inputs_modified':False}
    (result_dir/'manifest.json').write_text(json.dumps(manifest,indent=2))
    for path in configs:shutil.copy2(path,result_dir/path.name)
    print('准备完成。下面一格开始实际测量；每个模型/阶段都使用独立进程。',flush=True)
    return run_dir,configs

def run_profiles(run_dir,configs):
    result_dir=run_dir/'results';job_status=[]
    for config_path in configs:
        config=json.loads(config_path.read_text());dataset=config['dataset']
        jobs=[('shared_graph','graph')]+[(model,phase) for model in MODELS for phase in ('inference','train')]
        for model,phase in jobs:
            output=result_dir/f'{dataset}_{model}_{phase}'
            if output.exists():
                print('跳过已有测量目录：',output.name,flush=True)
                continue
            print(f'开始 {dataset} / {model} / {phase}',flush=True)
            env=os.environ.copy();env['PYTHONUNBUFFERED']='1'
            if model in ('LightGBM','shared_graph'):env['CUDA_VISIBLE_DEVICES']=''
            log_path=result_dir/f'{dataset}_{model}_{phase}.log'
            with log_path.open('x') as logfile:
                process=subprocess.Popen([sys.executable,str(run_dir/'profile_worker.py'),'--config',str(config_path),
                    '--model',model,'--phase',phase,'--output',str(output)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
                try:
                    for line in process.stdout:
                        logfile.write(line);logfile.flush()
                        # Keep progress concise while retaining complete logs in the ZIP.
                        if any(s in line for s in ['Completed:','fitting:','inference:','measured training batch','Neighborhood reconstruction','differs;','Traceback','Error:']):print(line.rstrip(),flush=True)
                    code=process.wait()
                except BaseException:
                    process.terminate()
                    try:process.wait(timeout=10)
                    except subprocess.TimeoutExpired:process.kill();process.wait()
                    raise
            job_status.append({'dataset':dataset,'model':model,'phase':phase,'return_code':code})
            (result_dir/'job_status.json').write_text(json.dumps(job_status,indent=2))
            if code:
                print(f'该阶段未完成；日志已保留：{log_path.name}。继续独立阶段。',flush=True)
    return package_results(run_dir)

def package_results(run_dir):
    result_dir=run_dir/'results'
    status=[]
    for path in sorted(result_dir.glob('*/profile.json')):
        obj=json.loads(path.read_text())
        status.append({'dataset':obj['dataset'],'model':obj['model'],'phase':obj['phase'],'status':obj['status'],
                       'prediction_agreement':obj.get('prediction_agreement'),
                       'graph_verified':obj.get('all_sections_verified')})
    (result_dir/'completion_summary.json').write_text(json.dumps(status,indent=2))
    report_text='真实资源测量结果。训练/推理采用独立进程。原有文件未修改。\n'
    report_text+='神经训练为固定批次资源测量，不是完整训练峰值；LightGBM 为完整基因面板拟合。\n'
    report_text+='共享邻域重建须 all_sections_verified=true 才可报告总构建时间。\n'
    report_text+='各模型的重新推理需与原预测匹配，才能将资源结果关联到原 benchmark。\n'
    report_text+='原始权重、表达矩阵、缓存及完整预测不打包。\n'
    (result_dir/'README.txt').write_text(report_text)
    bundle=run_dir.parent/(run_dir.name+'_results.zip')
    with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(result_dir.rglob('*')):
            if p.is_file():z.write(p,p.relative_to(run_dir))
    print('已打包结果：',bundle,flush=True)
    print('请上传此 ZIP；测量值和复现检查通过核对后再写入补充材料。',flush=True)
    return bundle


In [ ]:
run_dir, configs = prepare_profiles()


## 4. 开始实际测量

每个模型/阶段在独立进程中运行，避免前一个模型的内存残留影响后一个模型。RAM 采用 20 ms 间隔采样的进程 RSS，GPU 记录 PyTorch allocated 和 reserved 两种峰值。CPU LightGBM 的 GPU 指标记为不适用。

神经训练阶段不写新权重；完整推理始终加载原 checkpoint。共享邻域核对不通过时不外推总构图时间，其他独立测量仍保留。

可以观察进度。若某阶段失败，将保留日志并继续其余独立阶段。请把最终 ZIP 发回核对，不要先把未经核对的数字写进回复信。

In [ ]:
bundle = run_profiles(run_dir, configs)


## 5. 下载结果

只有下面生成的结果 ZIP 需要上传。若第 4 步中断，可在停止运行后单独执行本格打包已完成的记录。

In [ ]:
bundle = package_results(run_dir)
from google.colab import files
files.download(str(bundle))
